In [1]:
import os
import warnings
import random
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
import optuna
import joblib
from sklearn.inspection import permutation_importance
from catboost import CatBoostRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import json
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from xgboost import XGBRegressor

/Users/dhanujiamanda/Documents/Projects/Agentic AI /Pipeline/Agentic-AI-for-Pharma-Stockout-Problem/ENV/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ALLOW_FUTURE_VALIDATION = True  

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=FutureWarning)


# 01_PREPROCESSING

In [3]:
# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
DATA_PATH = "/Users/dhanujiamanda/Documents/Projects/Agentic AI /Pipeline/Agentic-AI-for-Pharma-Stockout-Problem/data/fact_monthly_closed.xlsx"
FOCUS_SKU_PATH = "/Users/dhanujiamanda/Documents/Projects/Agentic AI /Pipeline/Agentic-AI-for-Pharma-Stockout-Problem/data/Master Data/FocusItemCodes.xlsx"

LATEST_COMPLETE_YEAR = 2026
LATEST_COMPLETE_MONTH = 1

DEMAND_COL = "Secondary_Sales_Qty"

In [4]:
# ------------------------------------------------------------
# BASIC HELPERS
# ------------------------------------------------------------
def force_itemcode_str(df):
    df = df.copy()
    if "ItemCode" in df.columns:
        df["ItemCode"] = pd.to_numeric(df["ItemCode"], errors="coerce")
        df = df.dropna(subset=["ItemCode"])
        df["ItemCode"] = df["ItemCode"].astype(int).astype(str)

    if "ItemCode_Original" in df.columns:
        df["ItemCode_Original"] = df["ItemCode_Original"].astype(str)

    return df


def load_focus_skus(path):
    focus_df = pd.read_excel(path)

    focus_df["Code"] = pd.to_numeric(focus_df["Code"], errors="coerce")
    focus_df = focus_df.dropna(subset=["Code"])
    focus_df["Code"] = focus_df["Code"].astype(int).astype(str)

    return sorted(focus_df["Code"].unique().tolist())


def remove_incomplete_periods(df):
    Data = df.copy()
    Data = Data[
        (Data["Year"] < LATEST_COMPLETE_YEAR) |
        (
            (Data["Year"] == LATEST_COMPLETE_YEAR) &
            (Data["Month_Number"] <= LATEST_COMPLETE_MONTH)
        )
    ].copy()
    return Data


def clean_negative_values(df):
    df = df.copy()

    non_negative_cols = [
        "Secondary_Sales_Qty",
        "Primary_Sales_Qty",
        "Free_Qty",
        "Available_Primary_Inventory_Qty",
        "Distributor_Inventory_Qty",
        "Blocked_Stock_Qty",
        "Inspection_Stock_Qty",
        "Total_Primary_Inventory_Qty"
    ]
    for c in non_negative_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).clip(lower=0)

    flag_cols = [
        "Bonus_Flag",
        "Supply_Constraint_Flag",
        "Distributor_Buffer_Flag"
    ]
    for c in flag_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

    return df


# ------------------------------------------------------------
# OBSERVED DEMAND FEATURES
# used only for cleaning demand signal
# ------------------------------------------------------------
def add_observed_demand_features(df):
    df = df.copy()
    df = force_itemcode_str(df)
    df = df.sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)

    df["Observed_Demand"] = df[DEMAND_COL].clip(lower=0)

    grp = df.groupby("ItemCode")

    df["Lag1_Obs"] = grp["Observed_Demand"].shift(1)
    df["Lag2_Obs"] = grp["Observed_Demand"].shift(2)
    df["Lag3_Obs"] = grp["Observed_Demand"].shift(3)
    df["Lag6_Obs"] = grp["Observed_Demand"].shift(6)
    df["Lag12_Obs"] = grp["Observed_Demand"].shift(12)

    df["Rolling3M_Obs_Mean"] = grp["Observed_Demand"].transform(
        lambda x: x.rolling(3, min_periods=1).mean().shift(1)
    )

    df["Rolling6M_Obs_Mean"] = grp["Observed_Demand"].transform(
        lambda x: x.rolling(6, min_periods=1).mean().shift(1)
    )

    df["Rolling3M_Obs_Std"] = grp["Observed_Demand"].transform(
        lambda x: x.rolling(3, min_periods=1).std().shift(1)
    ).fillna(0)

    df["Baseline_Demand"] = (
        df["Rolling3M_Obs_Mean"]
        .fillna(df["Lag1_Obs"])
        .fillna(0)
    )

    safe_baseline = np.maximum(df["Baseline_Demand"], 1)

    df["Uplift_vs_Baseline"] = (
        df["Observed_Demand"] / safe_baseline
    ).clip(0, 5)

    df["Z_Score_Obs"] = (
        (df["Observed_Demand"] - df["Rolling3M_Obs_Mean"]) /
        (df["Rolling3M_Obs_Std"] + 1)
    ).fillna(0)

    return df


# ------------------------------------------------------------
# BONUS PATTERN FEATURES
# ------------------------------------------------------------
def detect_recurring_bonus_skus(
    df,
    min_bonus_months=3,
    gap_tolerance=1,
    uplift_threshold=1.4
):
    df = force_itemcode_str(df)
    out = []

    for item, g in df.groupby("ItemCode"):
        g = g.sort_values(["Year", "Month_Number"]).copy()
        g["Time_Index"] = g["Year"].astype(int) * 12 + g["Month_Number"].astype(int)

        bonus_rows = g[g["Bonus_Flag"] == 1].copy()

        recurring_flag = 0
        cycle_len = 0
        avg_gap = 0
        bonus_frequency = 0.0
        avg_bonus_uplift = 1.0

        if len(g) > 0:
            bonus_frequency = len(bonus_rows) / len(g)

        if len(bonus_rows) >= min_bonus_months:
            gaps = bonus_rows["Time_Index"].diff().dropna()

            if len(gaps) > 0:
                avg_gap = gaps.mean()
                rounded_gap = int(round(avg_gap))
                stable_gap_rate = ((gaps - rounded_gap).abs() <= gap_tolerance).mean()

                avg_bonus_uplift = (
                    bonus_rows["Uplift_vs_Baseline"]
                    .replace([np.inf, -np.inf], np.nan)
                    .clip(0, 5)
                    .fillna(1.0)
                    .median()
                )

                if stable_gap_rate >= 0.6 and avg_bonus_uplift >= uplift_threshold:
                    recurring_flag = 1
                    cycle_len = rounded_gap

        out.append({
            "ItemCode": item,
            "Recurring_Bonus_SKU": recurring_flag,
            "Bonus_Cycle_Length": cycle_len,
            "Avg_Bonus_Gap": avg_gap,
            "Bonus_Frequency_All": bonus_frequency,
            "Avg_Bonus_Uplift": avg_bonus_uplift
        })

    return pd.DataFrame(out)


def add_bonus_cycle_features(df):
    df = force_itemcode_str(df)
    df = df.sort_values(["ItemCode", "Year", "Month_Number"]).copy()

    pieces = []

    for item_code, g in df.groupby("ItemCode", sort=False):
        g = g.sort_values(["Year", "Month_Number"]).copy()
        g["Time_Index"] = g["Year"].astype(int) * 12 + g["Month_Number"].astype(int)

        bonus_time_idx = g.loc[g["Bonus_Flag"] == 1, "Time_Index"].tolist()

        months_since_last_bonus = []
        expected_bonus_month = []

        for _, r in g.iterrows():
            current_t = int(r["Time_Index"])
            past_bonus = [t for t in bonus_time_idx if t < current_t]

            if len(past_bonus) == 0:
                months_since_last_bonus.append(999)
            else:
                months_since_last_bonus.append(current_t - past_bonus[-1])

            cyc = float(r.get("Bonus_Cycle_Length", 0) or 0)
            recurring = int(r.get("Recurring_Bonus_SKU", 0) or 0)

            if recurring == 1 and cyc > 0 and len(past_bonus) > 0:
                expected_bonus_month.append(
                    1 if abs((current_t - past_bonus[-1]) - cyc) <= 1 else 0
                )
            else:
                expected_bonus_month.append(0)

        g["Months_Since_Last_Bonus"] = months_since_last_bonus
        g["Expected_Bonus_Month"] = expected_bonus_month

        g["Bonus_Flag_Lag1"] = g["Bonus_Flag"].shift(1).fillna(0)
        g["Bonus_Flag_Lag2"] = g["Bonus_Flag"].shift(2).fillna(0)
        g["Bonus_Flag_Lag3"] = g["Bonus_Flag"].shift(3).fillna(0)

        g["Bonus_Frequency_12M"] = (
            g["Bonus_Flag"]
            .rolling(12, min_periods=1)
            .mean()
            .shift(1)
            .fillna(0)
        )

        g = g.drop(columns=["Time_Index"], errors="ignore")
        pieces.append(g)

    return pd.concat(pieces, ignore_index=True)


def add_bonus_profile_features(df):
    df = df.copy()
    df = force_itemcode_str(df)

    bonus_pattern_df = detect_recurring_bonus_skus(df)

    df = df.drop(columns=[
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All",
        "Avg_Bonus_Uplift",
        "Months_Since_Last_Bonus",
        "Expected_Bonus_Month",
        "Bonus_Flag_Lag1",
        "Bonus_Flag_Lag2",
        "Bonus_Flag_Lag3",
        "Bonus_Frequency_12M"
    ], errors="ignore")

    df = df.merge(bonus_pattern_df, on="ItemCode", how="left")

    for c in [
        "Recurring_Bonus_SKU",
        "Bonus_Cycle_Length",
        "Avg_Bonus_Gap",
        "Bonus_Frequency_All"
    ]:
        df[c] = df[c].fillna(0)

    df["Avg_Bonus_Uplift"] = df["Avg_Bonus_Uplift"].fillna(1.0)

    df = add_bonus_cycle_features(df)

    return df


def add_last_bonus_demand_feature(df):
    df = force_itemcode_str(df)
    df = df.sort_values(["ItemCode", "Year", "Month_Number"]).copy()

    pieces = []

    for item, g in df.groupby("ItemCode", sort=False):
        g = g.copy()

        last_bonus_demand = []
        last_val = 0.0

        for _, r in g.iterrows():
            last_bonus_demand.append(last_val)

            if int(r.get("Bonus_Flag", 0)) == 1:
                last_val = float(r.get("Clean_Demand", r.get("Observed_Demand", 0)) or 0)

        g["Last_Bonus_Demand"] = last_bonus_demand
        pieces.append(g)

    return pd.concat(pieces, ignore_index=True)


# ------------------------------------------------------------
# STOCK FLOW FEATURES
# ------------------------------------------------------------
def add_stock_flow_features(df):
    df = df.copy()

    df["Net_Available_Stock"] = (
        df["Total_Primary_Inventory_Qty"]
        - df["Blocked_Stock_Qty"]
        - df["Inspection_Stock_Qty"]
    ).clip(lower=0)

    df["Primary_Stock_Cover"] = np.where(
        df["Baseline_Demand"] <= 0,
        0,
        df["Net_Available_Stock"] / (df["Baseline_Demand"] + 1)
    )

    df["Distributor_Stock_Cover"] = np.where(
        df["Baseline_Demand"] <= 0,
        0,
        df["Distributor_Inventory_Qty"] / (df["Baseline_Demand"] + 1)
    )

    df["Primary_to_Distributor_Ratio"] = np.where(
        df["Distributor_Inventory_Qty"] <= 0,
        0,
        df["Net_Available_Stock"] / (df["Distributor_Inventory_Qty"] + 1)
    )

    df["Blocked_Stock_Ratio"] = np.where(
        df["Total_Primary_Inventory_Qty"] <= 0,
        0,
        df["Blocked_Stock_Qty"] / (df["Total_Primary_Inventory_Qty"] + 1)
    )

    df["Inspection_Stock_Ratio"] = np.where(
        df["Total_Primary_Inventory_Qty"] <= 0,
        0,
        df["Inspection_Stock_Qty"] / (df["Total_Primary_Inventory_Qty"] + 1)
    )

    grp = df.groupby("ItemCode")

    df["Primary_Inv_Change"] = grp["Net_Available_Stock"].diff().fillna(0)
    df["Distributor_Inv_Change"] = grp["Distributor_Inventory_Qty"].diff().fillna(0)

    for c in [
        "Primary_to_Distributor_Ratio",
        "Distributor_Inv_Change",
        "Primary_Inv_Change"
    ]:
        if c in df.columns:
            upper = df[c].replace([np.inf, -np.inf], np.nan).quantile(0.99)
            df[c] = df[c].clip(upper=upper)

    return df


# ------------------------------------------------------------
# DEMAND CLEANING RULES
# ------------------------------------------------------------
def clean_demand_signal(df):
    df = df.copy()

    df["Effective_Demand"] = df["Observed_Demand"].copy()

    df["Supply_Baseline"] = (
        df["Rolling3M_Obs_Mean"]
        .fillna(df["Lag1_Obs"])
        .fillna(df["Observed_Demand"])
    )

    supply_constrained = df["Supply_Constraint_Flag"] == 1

    df["Effective_Demand"] = np.where(
        supply_constrained,
        np.maximum(df["Observed_Demand"], 0.85 * df["Supply_Baseline"]),
        df["Effective_Demand"]
    )

    irregular_bonus_spike = (
        (df["Bonus_Flag"] == 1) &
        (df["Recurring_Bonus_SKU"] == 0) &
        (df["Z_Score_Obs"] > 2.0) &
        (df["Uplift_vs_Baseline"] > 1.6)
    )

    grp = df.groupby("ItemCode")

    prev_obs = grp["Observed_Demand"].shift(1)
    prev_primary_cover = grp["Primary_Stock_Cover"].shift(1)
    prev_dist_cover = grp["Distributor_Stock_Cover"].shift(1)

    stockout_drop_condition = (
        (df["Observed_Demand"] < 0.65 * prev_obs.fillna(df["Observed_Demand"])) &
        (df["Supply_Constraint_Flag"] == 1) &
        (
            (prev_primary_cover.fillna(99) < 1.0) |
            (prev_dist_cover.fillna(99) < 1.0)
        )
    )

    df["Clean_Demand"] = df["Effective_Demand"].copy()

    df.loc[irregular_bonus_spike, "Clean_Demand"] = (
        0.60 * df.loc[irregular_bonus_spike, "Observed_Demand"] +
        0.40 * df.loc[irregular_bonus_spike, "Baseline_Demand"]
    )

    df.loc[stockout_drop_condition, "Clean_Demand"] = np.maximum(
        df.loc[stockout_drop_condition, "Observed_Demand"],
        0.90 * df.loc[stockout_drop_condition, "Baseline_Demand"]
    )

    df["Clean_Demand"] = df["Clean_Demand"].clip(lower=0)

    df["Bonus_Shock"] = irregular_bonus_spike.astype(int)
    df["Supply_Shock"] = stockout_drop_condition.astype(int)

    df["Free_Ratio"] = np.where(
        df["Primary_Sales_Qty"] <= 0,
        0,
        df["Free_Qty"] / (df["Primary_Sales_Qty"] + 1)
    )

    return df

def add_structural_demand_state(df):
    df = force_itemcode_str(df)
    df = df.sort_values(["ItemCode", "Year", "Month_Number"]).copy()

    states = []

    for item, g in df.groupby("ItemCode", sort=False):
        g = g.copy()

        roll3 = (
            g["Clean_Demand"]
            .rolling(3, min_periods=1)
            .mean()
            .shift(1)
        )

        roll6 = (
            g["Clean_Demand"]
            .rolling(6, min_periods=1)
            .mean()
            .shift(1)
        )

        growth_ratio = roll3 / (roll6 + 1)

        zero_rate_6m = (
            g["Clean_Demand"]
            .rolling(6, min_periods=1)
            .apply(lambda x: (x == 0).mean(), raw=False)
            .shift(1)
        )

        state = np.where(
            zero_rate_6m >= 0.50,
            "DYING_OR_INTERMITTENT",
            np.where(
                growth_ratio >= 1.30,
                "GROWING",
                np.where(growth_ratio <= 0.70, "DECLINING", "MATURE")
            )
        )

        g["Demand_State"] = pd.Series(state, index=g.index).fillna("MATURE")
        states.append(g)

    out = pd.concat(states, ignore_index=True)

    state_map = {
        "MATURE": 0,
        "GROWING": 1,
        "DECLINING": 2,
        "DYING_OR_INTERMITTENT": 3
    }

    out["Demand_State_Encoded"] = out["Demand_State"].map(state_map).fillna(0).astype(int)

    return out


# ------------------------------------------------------------
# SEGMENTATION
# ------------------------------------------------------------
def add_history_length(df):
    df = force_itemcode_str(df)

    hist_len = (
        df[["ItemCode", "Year", "Month_Number"]]
        .drop_duplicates()
        .groupby("ItemCode")
        .size()
        .reset_index(name="History_Length")
    )

    df = df.drop(columns=["History_Length", "History_Segment"], errors="ignore")
    df = df.merge(hist_len, on="ItemCode", how="left")

    df["History_Segment"] = np.select(
        [
            df["History_Length"] >= 18,
            (df["History_Length"] >= 10) & (df["History_Length"] < 18),
            df["History_Length"] < 10
        ],
        ["LONG", "MEDIUM", "SHORT"],
        default="SHORT"
    )

    return df


# ------------------------------------------------------------
# MAIN PREPROCESSING PIPELINE
# ------------------------------------------------------------
def run_preprocessing():
    Data = pd.read_excel(DATA_PATH)

    # normalize raw column names
    Data = Data.rename(columns={"MonthNo": "Month_Number"})
    # optional: ensure Month column is datetime
    if "Month" in Data.columns:
        Data["Month"] = pd.to_datetime(Data["Month"], errors="coerce")

    # Column check
    required_cols = [
        "Year", "Month_Number", "ItemCode",
        "Secondary_Sales_Qty", "Primary_Sales_Qty", "Free_Qty",
        "Available_Primary_Inventory_Qty",
        "Distributor_Inventory_Qty",
        "Blocked_Stock_Qty",
        "Inspection_Stock_Qty",
        "Total_Primary_Inventory_Qty",
        "Bonus_Flag",
        "Supply_Constraint_Flag",
        "Distributor_Buffer_Flag"
    ]
    missing_cols = [c for c in required_cols if c not in Data.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    Data = force_itemcode_str(Data)

    PHARMA_SKUS = load_focus_skus(FOCUS_SKU_PATH)

    Data = remove_incomplete_periods(Data)

    Data = Data[Data["ItemCode"].isin(PHARMA_SKUS)].copy()
    Data = Data.sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)

    Data = clean_negative_values(Data)
    Data = add_observed_demand_features(Data)
    Data = add_bonus_profile_features(Data)
    Data = add_stock_flow_features(Data)
    Data = clean_demand_signal(Data)
    Data = add_last_bonus_demand_feature(Data)
    Data = add_structural_demand_state(Data)
    Data = add_history_length(Data)

    Data = Data.sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)

    print("Rows:", len(Data))
    print("Unique SKUs:", Data["ItemCode"].nunique())
    print("Segment split:")
    print(Data[["ItemCode", "History_Segment"]].drop_duplicates()["History_Segment"].value_counts())

    missing_skus = sorted(set(PHARMA_SKUS) - set(Data["ItemCode"].unique()))
    print("Missing focus SKUs:", len(missing_skus))

    return Data, PHARMA_SKUS


In [5]:
Cleaned_Base_Data, PHARMA_SKUS = run_preprocessing()

Cleaned_Base_Data.to_csv("base_cleaned_data.csv", index=False)
Data = Cleaned_Base_Data.copy()

Rows: 35708
Unique SKUs: 727
Segment split:
History_Segment
LONG      602
MEDIUM    113
SHORT      12
Name: count, dtype: int64
Missing focus SKUs: 1


# 02_MODEL_FEATURE_ENGINEERING

In [6]:
# ============================================================
# MODEL_CONFIG
# ============================================================

ACTUAL_TARGET_COL = "Target"
MODEL_TARGET_COL = "Residual_Target"
BASELINE_COL = "Residual_Baseline"

In [7]:
# ============================================================
# HELPER_FUNCS
# ============================================================

def sanitize(df):
    df = df.replace([np.inf, -np.inf], np.nan)
    return df.fillna(0)

def recency_weights(df, yearly_boost=0.25, base=1.0):
    y0 = df["Year"].min()
    return base + (df["Year"] - y0) * yearly_boost

def wmape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    denominator = np.sum(np.abs(y_true))
    if denominator == 0:
        return 0
    return np.sum(np.abs(y_true - y_pred)) / denominator * 100

def forecast_bias(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    denominator = np.sum(np.abs(y_true))
    if denominator == 0:
        return 0
    return np.sum(y_pred - y_true) / denominator * 100

def underforecast_rate(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    diff = y_true - y_pred
    under = np.where(diff > 0, diff, 0)
    denom = np.sum(np.abs(y_true))
    if denom == 0:
        return 0
    return np.sum(under) / denom * 100

def evaluate_all_metrics(y_true, y_pred):
    return {
        "WMAPE": wmape(y_true, y_pred),
        "Bias": forecast_bias(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "Underforecast_Rate": underforecast_rate(y_true, y_pred)
    }

def compute_clip_caps(train_df, cols, q=0.99):
    """Compute clipping caps using TRAIN only."""
    caps = {}
    for c in cols:
        if c in train_df.columns:
            caps[c] = float(train_df[c].replace([np.inf, -np.inf], np.nan).dropna().quantile(q))
    return caps

def apply_clip_caps(df, caps):
    """Apply previously computed caps to any df."""
    df = df.copy()
    for c, cap in caps.items():
        if c in df.columns:
            df[c] = df[c].clip(upper=cap)
    return df

def assert_features_exist(df, feature_cols, where=""):
    missing = [c for c in feature_cols if c not in df.columns]
    if missing:
        raise KeyError(f"[{where}] Missing required features: {missing}")

def recompute_target(df):
    df = df.copy()
    df = force_itemcode_str(df)
    df[ACTUAL_TARGET_COL] = df.groupby("ItemCode")["Clean_Demand"].shift(-1)
    return df

def add_residual_target(df):
    df = df.copy()

    required = ["Rolling3M_Mean", "Lag1", ACTUAL_TARGET_COL]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"add_residual_target missing columns: {missing}")

    df[BASELINE_COL] = df["Rolling3M_Mean"].fillna(df["Lag1"]).fillna(0)
    df[MODEL_TARGET_COL] = df[ACTUAL_TARGET_COL] - df[BASELINE_COL]
    return df


# ItemCode Encoding
def encode_itemcode(train_df, valid_df):
    train_df = force_itemcode_str(train_df)
    valid_df = force_itemcode_str(valid_df)

    train_df = train_df.copy()
    valid_df = valid_df.copy()

    categories = pd.Index(train_df["ItemCode"].astype(str).unique())
    cat_to_code = {k: i for i, k in enumerate(categories)}
    unk_code = len(cat_to_code)

    train_df["ItemCode"] = train_df["ItemCode"].astype(str).map(cat_to_code).fillna(unk_code).astype(int)
    valid_df["ItemCode"] = valid_df["ItemCode"].astype(str).map(cat_to_code).fillna(unk_code).astype(int)

    return train_df, valid_df, categories

# SKU Cap Function
def apply_sku_cap(train_df, valid_df, quantile=0.999):
    train_df = force_itemcode_str(train_df)
    valid_df = force_itemcode_str(valid_df)

    train_df = train_df.copy()
    valid_df = valid_df.copy()

    sku_cap = train_df.groupby("ItemCode")["Clean_Demand"].quantile(quantile)

    # clip TRAIN only
    train_df["Clean_Demand"] = np.minimum(
        train_df["Clean_Demand"],
        train_df["ItemCode"].map(sku_cap)
    )

    # do NOT modify valid/test target
    return train_df, valid_df

# ABC Classification Function
def apply_abc_classification(train_df, valid_df):
    train_df = force_itemcode_str(train_df)
    valid_df = force_itemcode_str(valid_df)

    train_df = train_df.copy()
    valid_df = valid_df.copy()

    sku_total = (
        train_df.groupby("ItemCode")["Clean_Demand"]
        .sum()
        .sort_values(ascending=False)
    )

    total_sum = sku_total.sum()
    if total_sum <= 0:
        abc_map = {}
        train_df["ABC_Class"] = 2
        valid_df["ABC_Class"] = 2
        return train_df, valid_df, abc_map

    cum_pct = sku_total.cumsum() / total_sum

    abc_series = pd.cut(
        cum_pct,
        bins=[0, 0.7, 0.9, 1.0],
        labels=[0, 1, 2]
    )

    abc_map = abc_series.to_dict()

    train_df["ABC_Class"] = train_df["ItemCode"].map(abc_map).fillna(2)
    valid_df["ABC_Class"] = valid_df["ItemCode"].map(abc_map).fillna(2)

    return train_df, valid_df, abc_map

# Combined Wrapper
def apply_fold_adjustments(train_df, valid_df):
    train_df, valid_df = apply_sku_cap(train_df, valid_df)
    train_df, valid_df, abc_map = apply_abc_classification(train_df, valid_df)

    return train_df, valid_df, abc_map

# Segmentation Leakage 
def add_history_length_from_subset(df_subset, df_target=None):
    """
    Compute history length from df_subset only.
    Apply the segment labels onto df_target if provided.
    Otherwise return labels on df_subset itself.
    """
    df_subset = force_itemcode_str(df_subset)

    hist_len = (
        df_subset[["ItemCode", "Year", "Month_Number"]]
        .drop_duplicates()
        .groupby("ItemCode")
        .size()
        .reset_index(name="History_Length")
    )

    hist_len["History_Segment"] = np.select(
        [
            hist_len["History_Length"] >= 18,
            (hist_len["History_Length"] >= 10) & (hist_len["History_Length"] < 18),
            hist_len["History_Length"] < 10
        ],
        ["LONG", "MEDIUM", "SHORT"],
        default="SHORT"
    )

    if df_target is None:
        out = df_subset.copy()
        out = out.drop(columns=["History_Length", "History_Segment"], errors="ignore")
        out = out.merge(hist_len, on="ItemCode", how="left")
        out["History_Length"] = out["History_Length"].fillna(0)
        out["History_Segment"] = out["History_Segment"].fillna("SHORT")
        out = force_itemcode_str(out)
        return out

    out = force_itemcode_str(df_target.copy())
    out = out.drop(columns=["History_Length", "History_Segment"], errors="ignore")
    out = out.merge(hist_len, on="ItemCode", how="left")
    out["History_Length"] = out["History_Length"].fillna(0)
    out["History_Segment"] = out["History_Segment"].fillna("SHORT")
    out = force_itemcode_str(out)
    return out

# Demand Regime
def classify_demand_regime(row):
    lag1 = row.get("Lag1", 0)
    roll3 = row.get("Rolling3M_Mean", 0)
    roll6 = row.get("Rolling6M_Mean", roll3)
    bonus = row.get("Bonus_Flag", 0)
    supply = row.get("Supply_Constraint_Flag", 0)
    expected_bonus = row.get("Expected_Bonus_Month", 0)
    post_bonus = row.get("Post_Bonus_Month", 0)
    demand_state = row.get("Demand_State", "MATURE")

    anchor = max(roll3, roll6, 1)

    if supply == 1:
        return "SUPPLY_SHOCK"
    if bonus == 1 or expected_bonus == 1:
        return "PROMO"
    if post_bonus == 1:
        return "POST_PROMO_DROP"
    if demand_state == "GROWING":
        return "GROWING"
    if demand_state == "DECLINING":
        return "DECLINING"
    if lag1 > 1.8 * anchor:
        return "RECENT_SPIKE"
    if lag1 < 0.5 * anchor:
        return "RECENT_DROP"

    return "NORMAL"
def safe_num(x, default=0):

    return default if pd.isna(x) else float(x)

# DYNAMIC TIME WINDOWS FUNCS
def get_latest_complete_period(df):
    x = df[["Year", "Month_Number"]].dropna().copy()
    x["Year"] = x["Year"].astype(int)
    x["Month_Number"] = x["Month_Number"].astype(int)
    x = x.sort_values(["Year", "Month_Number"])
    last_row = x.iloc[-1]
    return int(last_row["Year"]), int(last_row["Month_Number"])
def add_period_index(df):
    out = df.copy()
    out["Period_Index"] = out["Year"].astype(int) * 12 + out["Month_Number"].astype(int)
    return out
def shift_year_month(year, month, offset_months):
    idx = year * 12 + month
    new_idx = idx + offset_months
    new_year = (new_idx - 1) // 12
    new_month = (new_idx - 1) % 12 + 1
    return int(new_year), int(new_month)
def build_time_windows(df, holdout_months=12, recent_months=4):
    temp = add_period_index(df)
    latest_year, latest_month = get_latest_complete_period(temp)
    latest_idx = latest_year * 12 + latest_month

    test_start_idx = latest_idx - holdout_months + 1
    recent_start_idx = latest_idx - recent_months + 1

    all_test_periods = (
        temp.loc[temp["Period_Index"].between(test_start_idx, latest_idx),
                 ["Year", "Month_Number", "Period_Index"]]
        .drop_duplicates()
        .sort_values("Period_Index")
        .reset_index(drop=True)
    )

    recent_periods = (
        temp.loc[temp["Period_Index"].between(recent_start_idx, latest_idx),
                 ["Year", "Month_Number", "Period_Index"]]
        .drop_duplicates()
        .sort_values("Period_Index")
        .reset_index(drop=True)
    )

    return {
        "latest_year": latest_year,
        "latest_month": latest_month,
        "latest_idx": latest_idx,
        "test_start_idx": test_start_idx,
        "recent_start_idx": recent_start_idx,
        "holdout_months": holdout_months,
        "recent_months": recent_months,
        "test_periods": all_test_periods,
        "recent_periods": recent_periods
    }


def build_model_features(df):
    df = df.copy()
    df = force_itemcode_str(df)
    df = df.sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)

    grp = df.groupby("ItemCode")

    # demand lags from cleaned demand
    for lag in [1, 2, 3, 6, 12]:
        df[f"Lag{lag}"] = grp["Clean_Demand"].shift(lag)

    # rolling demand features - shifted to avoid leakage
    df["Rolling3M_Mean"] = grp["Clean_Demand"].transform(
        lambda x: x.rolling(3, min_periods=1).mean().shift(1)
    )
    df["Rolling6M_Mean"] = grp["Clean_Demand"].transform(
        lambda x: x.rolling(6, min_periods=1).mean().shift(1)
    )
    df["Rolling3M_Std"] = grp["Clean_Demand"].transform(
        lambda x: x.rolling(3, min_periods=1).std().shift(1)
    ).fillna(0)

    df["Momentum"] = df["Lag1"] - df["Lag3"]

    # Promo instability
    df["Bonus_Last_Month"] = df["Bonus_Flag_Lag1"].fillna(0)
    df["Bonus_2M_Ago"] = df["Bonus_Flag_Lag2"].fillna(0)

    df["Post_Bonus_Month"] = np.where(
        (df["Bonus_Last_Month"] == 1) & (df["Bonus_Flag"] == 0),
        1, 0
    )

    df["Promo_Decay_Ratio"] = np.where(
        df["Rolling6M_Mean"] > 0,
        df["Lag1"] / (df["Rolling6M_Mean"] + 1),
        0
    )

    df["Current_Total_Usable_Stock"] = (
        df["Available_Primary_Inventory_Qty"].fillna(0) +
        df["Distributor_Inventory_Qty"].fillna(0)
    )

    df["Current_Stock_Cover"] = (
        df["Current_Total_Usable_Stock"] /
        (df["Rolling3M_Mean"].fillna(0) + 1)
    )

    df["Current_Stockout_Risk"] = np.where(
        df["Current_Stock_Cover"] < 0.5,
        1, 0
    )

    df = add_last_bonus_demand_feature(df)

    # seasonality
    df["Month_Sin"] = np.sin(2 * np.pi * df["Month_Number"] / 12)
    df["Month_Cos"] = np.cos(2 * np.pi * df["Month_Number"] / 12)

    # stock cover using cleaned rolling demand
    df["Stock_Cover_Months"] = np.where(
        df["Rolling3M_Mean"] > 0,
        df["Net_Available_Stock"] / (df["Rolling3M_Mean"] + 1),
        0
    )

    # SKU profile
    sku_profile = (
        df.groupby("ItemCode")
        .agg(
            SKU_Mean_Demand=("Clean_Demand", "mean"),
            SKU_Std_Demand=("Clean_Demand", "std"),
            SKU_ZeroRate=("Clean_Demand", lambda x: (x == 0).mean())
        )
        .reset_index()
    )

    sku_profile["SKU_Std_Demand"] = sku_profile["SKU_Std_Demand"].fillna(0)
    sku_profile["SKU_CV"] = np.where(
        sku_profile["SKU_Mean_Demand"] > 0,
        sku_profile["SKU_Std_Demand"] / (sku_profile["SKU_Mean_Demand"] + 1),
        0
    )

    # IMPORTANT: prevent _x / _y columns during repeated inference/backtest calls
    df = df.drop(columns=["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV", "SKU_Std_Demand"],errors="ignore")

    df = df.merge(
        sku_profile[["ItemCode", "SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]],
        on="ItemCode",
        how="left"
    )

    # Recompute demand state after model rolling features
    df = add_structural_demand_state(df)


    # Demand Regime
    df["Demand_Regime"] = df.apply(classify_demand_regime, axis=1)
    regime_map = {
        "NORMAL": 0,
        "PROMO": 1,
        "POST_PROMO_DROP": 2,
        "SUPPLY_SHOCK": 3,
        "RECENT_SPIKE": 4,
        "RECENT_DROP": 5,
        "GROWING": 6,
        "DECLINING": 7
    }
    df["Demand_Regime_Encoded"] = df["Demand_Regime"].map(regime_map).fillna(0).astype(int)

    grp = df.groupby("ItemCode")

    # target = next month demand
    df[ACTUAL_TARGET_COL] = grp["Clean_Demand"].shift(-1)

    # residual learning
    df[BASELINE_COL] = df["Rolling3M_Mean"].fillna(df["Lag1"]).fillna(0)
    df[MODEL_TARGET_COL] = df[ACTUAL_TARGET_COL] - df[BASELINE_COL]

    # behavior type
    df["Behavior_Type"] = np.select(
        [
            df["SKU_ZeroRate"] >= 0.40,
            df["Bonus_Frequency_All"] >= 0.25,
            df["SKU_CV"] >= 1.0,
        ],
        [
            "INTERMITTENT",
            "PROMO_DRIVEN",
            "VOLATILE",
        ],
        default="STABLE"
    )

    df = df.replace([np.inf, -np.inf], np.nan)

    return df


In [8]:
# ============================================================
# MODEL_FEATURE_ENGINEERING
# ============================================================
Model_Data_All = build_model_features(Cleaned_Base_Data)
Model_Data = Model_Data_All.dropna(subset=[ACTUAL_TARGET_COL]).copy()

print("Model rows:", len(Model_Data))
print("Unique SKUs:", Model_Data["ItemCode"].nunique())
print("Segments:")
print(Model_Data[["ItemCode", "History_Segment"]].drop_duplicates()["History_Segment"].value_counts())
print("Behavior:")
print(Model_Data[["ItemCode", "Behavior_Type"]].drop_duplicates()["Behavior_Type"].value_counts())

Model rows: 34981
Unique SKUs: 726
Segments:
History_Segment
LONG      602
MEDIUM    113
SHORT      11
Name: count, dtype: int64
Behavior:
Behavior_Type
PROMO_DRIVEN    482
STABLE          215
VOLATILE         16
INTERMITTENT     13
Name: count, dtype: int64


In [9]:
# ============================================================
# TIME_WINDOWS
# ============================================================

TIME_WINDOWS = build_time_windows(Model_Data, holdout_months=12, recent_months=4)

print("Latest period:", TIME_WINDOWS["latest_year"], TIME_WINDOWS["latest_month"])
print("Test periods:")
print(TIME_WINDOWS["test_periods"])
print("Recent periods:")
print(TIME_WINDOWS["recent_periods"])

Latest period: 2025 12
Test periods:
    Year  Month_Number  Period_Index
0   2025             1         24301
1   2025             2         24302
2   2025             3         24303
3   2025             4         24304
4   2025             5         24305
5   2025             6         24306
6   2025             7         24307
7   2025             8         24308
8   2025             9         24309
9   2025            10         24310
10  2025            11         24311
11  2025            12         24312
Recent periods:
   Year  Month_Number  Period_Index
0  2025             9         24309
1  2025            10         24310
2  2025            11         24311
3  2025            12         24312


In [10]:
# ============================================================
# FEATURE_LIST
# ============================================================

MODEL_FEATURES = [
    "ItemCode",
    "Month_Number",
    "Month_Sin", "Month_Cos",

    "Lag1", "Lag2", "Lag3", "Lag6", "Lag12",
    "Rolling3M_Mean", "Rolling6M_Mean", "Rolling3M_Std",
    "Momentum",

    "Bonus_Flag",
    "Bonus_Flag_Lag1", "Bonus_Flag_Lag2", "Bonus_Flag_Lag3",
    "Bonus_Frequency_12M",
    "Bonus_Frequency_All",
    "Recurring_Bonus_SKU",
    "Bonus_Cycle_Length",
    "Months_Since_Last_Bonus",
    "Expected_Bonus_Month",
    "Avg_Bonus_Uplift",
    "Bonus_Shock",

    "Last_Bonus_Demand",
    "Demand_State_Encoded", 

    "Bonus_Last_Month",
    "Bonus_2M_Ago",
    "Post_Bonus_Month",
    "Promo_Decay_Ratio",
    "Demand_Regime_Encoded",
    "Current_Stock_Cover",
    "Current_Stockout_Risk",

    "Supply_Constraint_Flag",
    "Distributor_Buffer_Flag",
    "Supply_Shock",

    "Net_Available_Stock",
    "Distributor_Inventory_Qty",
    "Primary_Stock_Cover",
    "Distributor_Stock_Cover",
    "Stock_Cover_Months",

    "Free_Ratio",

    "SKU_Mean_Demand",
    "SKU_CV",
    "SKU_ZeroRate",
    "History_Length"
]

LONG_FEATURES = MODEL_FEATURES.copy()
MEDIUM_FEATURES = MODEL_FEATURES.copy()

print("Feature count:", len(MODEL_FEATURES))

Feature count: 47


# 03_MODEL_PIPELINE

In [11]:
# ============================================================
# DATA_SPLIT_BY_SEGMENT
# ============================================================

Data = Model_Data.copy()

Data_long = Data[Data["History_Segment"] == "LONG"].copy()
Data_medium = Data[Data["History_Segment"] == "MEDIUM"].copy()
Data_short = Data[Data["History_Segment"] == "SHORT"].copy()

print("LONG rows:", len(Data_long), "| SKUs:", Data_long["ItemCode"].nunique())
print("MEDIUM rows:", len(Data_medium), "| SKUs:", Data_medium["ItemCode"].nunique())
print("SHORT rows:", len(Data_short), "| SKUs:", Data_short["ItemCode"].nunique())

print("\nBehavior split:")
print(Data[["ItemCode", "Behavior_Type"]].drop_duplicates()["Behavior_Type"].value_counts())

LONG rows: 33653 | SKUs: 602
MEDIUM rows: 1254 | SKUs: 113
SHORT rows: 74 | SKUs: 11

Behavior split:
Behavior_Type
PROMO_DRIVEN    482
STABLE          215
VOLATILE         16
INTERMITTENT     13
Name: count, dtype: int64


### HELPER FUNCS

In [12]:
# SHARED EVALUATION REPORT HELPERS
def print_model_eval_report(eval_df, title="EVALUATION", group_cols=None):
    df = eval_df.copy()

    if group_cols is None:
        group_cols = []

    required_cols = ["Actual", "Pred", "Abs_Error"]
    for c in required_cols:
        if c not in df.columns:
            raise ValueError(f"Missing required column: {c}")

    print(f"\n========== {title} ==========")

    overall_metrics = evaluate_all_metrics(df["Actual"].values, df["Pred"].values)

    if group_cols:
        grouped = (
            df.groupby(group_cols, as_index=False)
            .agg(
                Actual_Sum=("Actual", "sum"),
                Pred_Sum=("Pred", "sum"),
                Total_Abs_Error=("Abs_Error", "sum"),
                MAE=("Abs_Error", "mean")
            )
        )

        grouped["WMAPE"] = np.where(
            grouped["Actual_Sum"] > 0,
            grouped["Total_Abs_Error"] / grouped["Actual_Sum"] * 100,
            np.nan
        )

        grouped["Bias"] = np.where(
            grouped["Actual_Sum"] > 0,
            (grouped["Pred_Sum"] - grouped["Actual_Sum"]) / grouped["Actual_Sum"] * 100,
            np.nan
        )

        print(grouped.sort_values("WMAPE"))

    print("Overall WMAPE:", overall_metrics["WMAPE"])
    print(overall_metrics)

    return overall_metrics

def build_model_summary_table(eval_df, model_col="Model_Name", extra_group_cols=None):
    df = eval_df.copy()

    group_cols = [model_col]
    if extra_group_cols:
        group_cols += extra_group_cols

    out = (
        df.groupby(group_cols, as_index=False)
        .agg(
            Actual_Sum=("Actual", "sum"),
            Pred_Sum=("Pred", "sum"),
            Total_Abs_Error=("Abs_Error", "sum"),
            MAE=("Abs_Error", "mean")
        )
    )

    out["WMAPE"] = np.where(
        out["Actual_Sum"] > 0,
        out["Total_Abs_Error"] / out["Actual_Sum"] * 100,
        np.nan
    )

    out["Bias"] = np.where(
        out["Actual_Sum"] > 0,
        (out["Pred_Sum"] - out["Actual_Sum"]) / out["Actual_Sum"] * 100,
        np.nan
    )

    return out.sort_values("WMAPE").reset_index(drop=True)

# SHARED
def permutation_rank(model, eval_df, feature_cols, target_col, n_repeats=5):
    """
    Permutation importance on an evaluation dataframe.
    Use validation windows for pruning, not the final holdout test window.
    """
    X = sanitize(eval_df[feature_cols])
    y = eval_df[target_col].values

    r = permutation_importance(
        model,
        X,
        y,
        scoring="neg_mean_absolute_error",
        n_repeats=n_repeats,
        random_state=42
    )

    imp = pd.DataFrame({
        "feature": feature_cols,
        "perm_importance": r.importances_mean
    }).sort_values("perm_importance", ascending=False)

    return imp


### LONG SEGMENT

In [13]:
# COMMON LONG PREP HELPERS
def build_sku_history_profile(train_df):
    train_df = force_itemcode_str(train_df)

    profile = (
        train_df.groupby("ItemCode")
        .agg(
            SKU_Mean_Demand=("Clean_Demand", "mean"),
            SKU_Std_Demand=("Clean_Demand", "std"),
            SKU_ZeroRate=("Clean_Demand", lambda x: (x == 0).mean())
        )
        .reset_index()
    )

    profile["SKU_Std_Demand"] = profile["SKU_Std_Demand"].fillna(0)
    profile["SKU_CV"] = np.where(
        profile["SKU_Mean_Demand"] > 0,
        profile["SKU_Std_Demand"] / (profile["SKU_Mean_Demand"] + 1),
        0
    )

    return force_itemcode_str(profile[[
        "ItemCode", "SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"
    ]])

def merge_sku_history_profile(df, profile_df):
    df = force_itemcode_str(df.copy())
    profile_df = force_itemcode_str(profile_df.copy())

    df = df.drop(
        columns=["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"],
        errors="ignore"
    )

    df = df.merge(profile_df, on="ItemCode", how="left")

    for c in ["SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV"]:
        df[c] = df[c].fillna(0)

    return df

def get_long_tune_windows(df, time_windows, n_folds=2, fold_size_months=12):
    temp = add_period_index(df)
    periods = (
        temp[["Year", "Month_Number", "Period_Index"]]
        .drop_duplicates()
        .sort_values("Period_Index")
        .reset_index(drop=True)
    )

    usable = periods[periods["Period_Index"] < time_windows["test_start_idx"]].copy()

    if len(usable) < fold_size_months * (n_folds + 1):
        raise ValueError("Not enough history to build long tune windows.")

    windows = []
    end_idx = usable["Period_Index"].max()

    for _ in range(n_folds):
        valid_end_idx = end_idx
        valid_start_idx = valid_end_idx - fold_size_months + 1
        windows.append({
            "valid_start_idx": int(valid_start_idx),
            "valid_end_idx": int(valid_end_idx)
        })
        end_idx = valid_start_idx - 1

    windows = list(reversed(windows))
    return windows

def get_long_validation_window(time_windows, valid_months=12):
    valid_end_idx = int(time_windows["test_start_idx"] - 1)
    valid_start_idx = int(valid_end_idx - valid_months + 1)
    return {
        "valid_start_idx": valid_start_idx,
        "valid_end_idx": valid_end_idx,
        "valid_months": int(valid_months)
    }

def get_long_recent_window(time_windows, recent_months=4):
    # latest_idx itself cannot be scored after shift(-1),
    # because the next-month actual target does not exist there.
    recent_end_idx = int(time_windows["latest_idx"] - 1)
    recent_start_idx = int(recent_end_idx - recent_months + 1)

    return {
        "valid_start_idx": recent_start_idx,
        "valid_end_idx": recent_end_idx,
        "recent_months": int(recent_months)
    }

def get_long_pre_recent_holdout_window(time_windows, holdout_months=12, recent_months=4):
    recent_window = get_long_recent_window(time_windows, recent_months=recent_months)

    holdout12_end_idx = int(recent_window["valid_start_idx"] - 1)
    holdout12_start_idx = int(holdout12_end_idx - holdout_months + 1)

    if holdout12_start_idx >= holdout12_end_idx:
        raise ValueError("Invalid LONG pre-recent holdout window.")

    return {
        "valid_start_idx": holdout12_start_idx,
        "valid_end_idx": holdout12_end_idx,
        "holdout_months": int(holdout_months)
    }

def combine_long_model_a_window(time_windows, holdout_months=12, recent_months=4):
    holdout12_window = get_long_pre_recent_holdout_window(
        time_windows=time_windows,
        holdout_months=holdout_months,
        recent_months=recent_months
    )
    recent4_window = get_long_recent_window(
        time_windows=time_windows,
        recent_months=recent_months
    )

    return {
        "valid_start_idx": int(holdout12_window["valid_start_idx"]),
        "valid_end_idx": int(recent4_window["valid_end_idx"]),
        "holdout_months": int(holdout_months),
        "recent_months": int(recent_months)
    }

def standardize_long_model_output(
    df,
    actual_col,
    pred_col,
    item_col="ItemCode",
    year_col="Year",
    month_col="Month_Number",
    model_name="UNKNOWN",
    segment="LONG"
):
    out = df.copy()
    out = force_itemcode_str(out)

    if item_col not in out.columns:
        raise KeyError(f"Missing item column: {item_col}")
    if actual_col not in out.columns:
        raise KeyError(f"Missing actual column: {actual_col}")
    if pred_col not in out.columns:
        raise KeyError(f"Missing pred column: {pred_col}")

    out["ItemCode_Original"] = out[item_col].astype(str)
    out["ItemCode"] = out["ItemCode_Original"]
    out["Actual"] = pd.to_numeric(out[actual_col], errors="coerce")
    out["Pred"] = pd.to_numeric(out[pred_col], errors="coerce").clip(lower=0)

    if year_col in out.columns:
        out["Year"] = out[year_col]
    else:
        out["Year"] = np.nan

    if month_col in out.columns:
        out["Month_Number"] = out[month_col]
    else:
        out["Month_Number"] = np.nan

    out["Error"] = out["Actual"] - out["Pred"]
    out["Abs_Error"] = np.abs(out["Error"])
    out["Segment"] = segment
    out["Model_Name"] = model_name

    keep_cols = [
        "ItemCode",
        "ItemCode_Original",
        "Year",
        "Month_Number",
        "Actual",
        "Pred",
        "Error",
        "Abs_Error",
        "Segment",
        "Model_Name"
    ]

    extra_cols = [c for c in out.columns if c not in keep_cols]
    return out[keep_cols + extra_cols].copy()

def build_train_weights(df, yearly_boost=0.25):
    w = recency_weights(df, yearly_boost=yearly_boost).astype(float)

    abc_weight = np.where(
        df["ABC_Class"] == 0, 2.5,
        np.where(df["ABC_Class"] == 1, 1.2, 1.0)
    )

    underforecast_risk = np.where(
        df["Clean_Demand"] > df["Rolling3M_Mean"].fillna(0),
        1.3,
        1.0
    )

    promo_weight = np.where(
        (df["Bonus_Flag"] == 1) | (df.get("Expected_Bonus_Month", 0) == 1),
        1.25,
        1.0
    )

    post_promo_weight = np.where(
        df.get("Post_Bonus_Month", 0) == 1,
        1.15,
        1.0
    )

    return w * abc_weight * underforecast_risk * promo_weight * post_promo_weight


In [14]:
# LONG_WINDOWS
LONG_TUNE_WINDOWS = get_long_tune_windows(
    Data,
    TIME_WINDOWS,
    n_folds=2,
    fold_size_months=12
)

LONG_VALID_WINDOW = get_long_validation_window(
    TIME_WINDOWS,
    valid_months=12
)

RECENT_MONTHS = TIME_WINDOWS.get("recent_months", 4)

LONG_RECENT4_WINDOW = get_long_recent_window(
    TIME_WINDOWS,
    recent_months=RECENT_MONTHS
)

LONG_HOLDOUT12_WINDOW = get_long_pre_recent_holdout_window(
    TIME_WINDOWS,
    holdout_months=12,
    recent_months=RECENT_MONTHS
)

LONG_MODEL_A_WINDOW = combine_long_model_a_window(
    TIME_WINDOWS,
    holdout_months=12,
    recent_months=RECENT_MONTHS
)

print("LONG_TUNE_WINDOWS:", LONG_TUNE_WINDOWS)
print("LONG_VALID_WINDOW:", LONG_VALID_WINDOW)
print("LONG_HOLDOUT12_WINDOW:", LONG_HOLDOUT12_WINDOW)
print("LONG_RECENT4_WINDOW:", LONG_RECENT4_WINDOW)
print("LONG_MODEL_A_WINDOW:", LONG_MODEL_A_WINDOW)

LONG_TUNE_WINDOWS: [{'valid_start_idx': 24277, 'valid_end_idx': 24288}, {'valid_start_idx': 24289, 'valid_end_idx': 24300}]
LONG_VALID_WINDOW: {'valid_start_idx': 24289, 'valid_end_idx': 24300, 'valid_months': 12}
LONG_HOLDOUT12_WINDOW: {'valid_start_idx': 24296, 'valid_end_idx': 24307, 'holdout_months': 12}
LONG_RECENT4_WINDOW: {'valid_start_idx': 24308, 'valid_end_idx': 24311, 'recent_months': 4}
LONG_MODEL_A_WINDOW: {'valid_start_idx': 24296, 'valid_end_idx': 24311, 'holdout_months': 12, 'recent_months': 4}


#### XGB & CAT

In [15]:
def prepare_long_frame_foldsafe(train_df, valid_df):
    train_df = force_itemcode_str(train_df.copy())
    valid_df = force_itemcode_str(valid_df.copy())

    train_df = train_df[train_df["History_Segment"] == "LONG"].copy()
    valid_df = valid_df[valid_df["History_Segment"] == "LONG"].copy()

    if train_df.empty or valid_df.empty:
        return None, None, None

    train_df, valid_df, abc_map = apply_fold_adjustments(train_df, valid_df)

    sku_profile_df = build_sku_history_profile(train_df)

    train_df = merge_sku_history_profile(train_df, sku_profile_df)
    valid_df = merge_sku_history_profile(valid_df, sku_profile_df)

    clip_cols = [
        "Rolling3M_Std",
        "Momentum",
        "Stock_Cover_Months",
        "Primary_Stock_Cover",
        "Distributor_Stock_Cover",
        "Free_Ratio",
        "SKU_CV"
    ]

    caps = compute_clip_caps(train_df, clip_cols, q=0.99)
    train_df = apply_clip_caps(train_df, caps)
    valid_df = apply_clip_caps(valid_df, caps)

    train_df = train_df.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()
    valid_df = valid_df.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()

    if train_df.empty or valid_df.empty:
        return None, None, None

    return train_df, valid_df, {
        "abc_map": abc_map,
        "clip_caps": caps,
        "sku_profile_df": sku_profile_df
    }

def get_long_model(model_name, params):
    model_name = model_name.upper()

    if model_name == "XGBOOST":
        return xgb.XGBRegressor(
            objective="reg:squarederror",
            eval_metric="rmse",
            random_state=42,
            tree_method="hist",
            n_jobs=-1,
            **params
        )

    if model_name == "CATBOOST":
        return CatBoostRegressor(
            loss_function="RMSE",
            eval_metric="RMSE",
            random_seed=42,
            verbose=0,
            **params
        )

    raise ValueError(f"Unknown model: {model_name}")

def suggest_long_params(trial, model_name):
    model_name = model_name.upper()

    if model_name == "XGBOOST":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 500, 1000),
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.06),
            "max_depth": trial.suggest_int("max_depth", 4, 7),
            "max_leaves": 64,
            "grow_policy": "lossguide",
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 6),
            "subsample": trial.suggest_float("subsample", 0.75, 0.9),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.75, 0.9),
            "gamma": trial.suggest_float("gamma", 0, 0.3),
            "reg_lambda": trial.suggest_float("reg_lambda", 2, 12),
            "reg_alpha": trial.suggest_float("reg_alpha", 0, 3),
        }

    if model_name == "CATBOOST":
        return {
            "iterations": trial.suggest_int("iterations", 500, 1000),
            "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.06),
            "depth": trial.suggest_int("depth", 4, 8),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 2.0, 15.0),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 5, 40),
            "subsample": trial.suggest_float("subsample", 0.70, 0.95),
            "rsm": trial.suggest_float("rsm", 0.70, 1.00),
            "random_strength": trial.suggest_float("random_strength", 0.0, 3.0),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 3.0)
        }

    raise ValueError(f"Unknown model: {model_name}")

def tune_residual_long_model(full_data, feature_cols, model_name, n_trials=15, study_name=None):
    full_data = full_data.copy()
    full_data_pi = add_period_index(full_data)

    def objective(trial):
        params = suggest_long_params(trial, model_name)
        scores = []

        for window in LONG_TUNE_WINDOWS:
            train_df = full_data_pi[full_data_pi["Period_Index"] < window["valid_start_idx"]].copy()
            valid_df = full_data_pi[
                full_data_pi["Period_Index"].between(window["valid_start_idx"], window["valid_end_idx"])
            ].copy()

            train_df, valid_df, _ = prepare_long_frame_foldsafe(train_df, valid_df)

            if train_df is None or valid_df is None:
                continue

            train_df = recompute_target(train_df)
            train_df = add_residual_target(train_df)
            train_df = train_df.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()

            train_df["ItemCode_Original"] = train_df["ItemCode"]
            valid_df["ItemCode_Original"] = valid_df["ItemCode"]

            train_df, valid_df, _ = encode_itemcode(train_df, valid_df)

            assert_features_exist(train_df, feature_cols, where=f"{model_name}_LONG_TUNE_TRAIN")
            assert_features_exist(valid_df, feature_cols, where=f"{model_name}_LONG_TUNE_VALID")

            model = get_long_model(model_name, params)

            model.fit(
                sanitize(train_df[feature_cols]),
                train_df[MODEL_TARGET_COL],
                sample_weight=build_train_weights(train_df),
                verbose=False
            )

            pred_residual = model.predict(sanitize(valid_df[feature_cols]))
            pred_final = np.clip(valid_df[BASELINE_COL].values + pred_residual, 0, None)

            scores.append(wmape(valid_df[ACTUAL_TARGET_COL].values, pred_final))

        return 999999.0 if len(scores) == 0 else np.mean(scores)

    study = optuna.create_study(direction="minimize", study_name=study_name)
    study.optimize(objective, n_trials=n_trials)

    return study.best_params, study

def train_eval_validation_long_model(
    full_data,
    feature_cols,
    best_params,
    valid_window,
    model_name,
    yearly_boost=0.25
):
    full_data = add_period_index(full_data)

    train_df = full_data[full_data["Period_Index"] < valid_window["valid_start_idx"]].copy()
    valid_df = full_data[
        full_data["Period_Index"].between(valid_window["valid_start_idx"], valid_window["valid_end_idx"])
    ].copy()

    train_df, valid_df, prep_artifacts = prepare_long_frame_foldsafe(train_df, valid_df)

    if train_df is None or valid_df is None:
        raise ValueError(f"No usable LONG rows for {model_name}")

    train_df = recompute_target(train_df)
    train_df = add_residual_target(train_df)
    train_df = train_df.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()

    train_df["ItemCode_Original"] = train_df["ItemCode"]
    valid_df["ItemCode_Original"] = valid_df["ItemCode"]

    train_df, valid_df, itemcode_categories = encode_itemcode(train_df, valid_df)
    prep_artifacts["itemcode_categories"] = itemcode_categories

    assert_features_exist(train_df, feature_cols, where=f"{model_name}_LONG_TRAIN")
    assert_features_exist(valid_df, feature_cols, where=f"{model_name}_LONG_VALID")

    model = get_long_model(model_name, best_params)

    model.fit(
        sanitize(train_df[feature_cols]),
        train_df[MODEL_TARGET_COL],
        sample_weight=build_train_weights(train_df, yearly_boost=yearly_boost),
        verbose=False
    )

    valid_df["Pred_Residual"] = model.predict(sanitize(valid_df[feature_cols]))
    valid_df["Pred"] = np.clip(valid_df[BASELINE_COL] + valid_df["Pred_Residual"], 0, None)

    metrics = evaluate_all_metrics(
        valid_df[ACTUAL_TARGET_COL].values,
        valid_df["Pred"].values
    )

    return model, valid_df, metrics, prep_artifacts

def iterative_feature_prune_long_model(
    full_data,
    start_features,
    best_params,
    valid_window,
    model_name,
    drop_k=1,
    min_features=20,
    max_rounds=6,
    tolerance=0.10,
    n_repeats=3
):
    history = []
    features = start_features.copy()

    model, eval_df, m, _ = train_eval_validation_long_model(
        full_data=full_data,
        feature_cols=features,
        best_params=best_params,
        valid_window=valid_window,
        model_name=model_name
    )

    best_wmape = m["WMAPE"]
    last_accepted_state = (features.copy(), model, eval_df.copy(), m.copy())

    protected = {
        "ItemCode",
        "ABC_Class",
        "Lag1",
        "Rolling3M_Mean",
        "Rolling6M_Mean",
        "Month_Sin",
        "Month_Cos",
        "Recurring_Bonus_SKU",
        "Expected_Bonus_Month",
        "Avg_Bonus_Uplift"
    }

    for r in range(1, max_rounds + 1):
        if len(features) <= min_features:
            break

        imp = permutation_rank(
            model=model,
            eval_df=eval_df,
            feature_cols=features,
            target_col=MODEL_TARGET_COL,
            n_repeats=n_repeats
        )

        drop_candidates = [
            f for f in imp.sort_values("perm_importance").feature.tolist()
            if f not in protected
        ]

        to_drop = drop_candidates[:drop_k]

        if not to_drop:
            break

        new_features = [f for f in features if f not in to_drop]

        new_model, new_eval_df, new_m, _ = train_eval_validation_long_model(
            full_data=full_data,
            feature_cols=new_features,
            best_params=best_params,
            valid_window=valid_window,
            model_name=model_name
        )

        history.append({
            "round": r,
            "model": model_name,
            "segment": "LONG",
            "dropped": to_drop,
            "n_features": len(new_features),
            **new_m
        })

        if new_m["WMAPE"] <= best_wmape + tolerance:
            features = new_features
            model, eval_df = new_model, new_eval_df
            best_wmape = min(best_wmape, new_m["WMAPE"])
            last_accepted_state = (features.copy(), model, eval_df.copy(), new_m.copy())
        else:
            break

    return (
        last_accepted_state[0],
        last_accepted_state[1],
        last_accepted_state[2],
        last_accepted_state[3],
        pd.DataFrame(history)
    )

def prepare_long_deploy_frame(full_data):
    deploy_df = force_itemcode_str(full_data.copy())
    deploy_df = deploy_df[deploy_df["History_Segment"] == "LONG"].copy()

    train_df, deploy_df, abc_map = apply_fold_adjustments(
        deploy_df.copy(),
        deploy_df.copy()
    )

    sku_profile_df = build_sku_history_profile(train_df)
    deploy_df = merge_sku_history_profile(deploy_df, sku_profile_df)

    clip_cols = [
        "Rolling3M_Std",
        "Momentum",
        "Stock_Cover_Months",
        "Primary_Stock_Cover",
        "Distributor_Stock_Cover",
        "Free_Ratio",
        "SKU_CV"
    ]

    caps = compute_clip_caps(deploy_df, clip_cols, q=0.99)
    deploy_df = apply_clip_caps(deploy_df, caps)

    deploy_df = deploy_df.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()

    deploy_df["ItemCode_Original"] = deploy_df["ItemCode"]
    deploy_df, _, itemcode_categories = encode_itemcode(
        deploy_df.copy(),
        deploy_df.copy()
    )

    artifacts = {
        "abc_map": abc_map,
        "clip_caps": caps,
        "itemcode_categories": itemcode_categories,
        "sku_profile_df": sku_profile_df,
    }

    return deploy_df, artifacts

In [16]:
# TUNNING
xgb_long_best_params, xgb_long_study = tune_residual_long_model(
    full_data=Data,
    feature_cols=LONG_FEATURES,
    model_name="XGBOOST",
    n_trials=30,
    study_name="long_xgboost"
)

catboost_long_best_params, catboost_long_study = tune_residual_long_model(
    full_data=Data,
    feature_cols=LONG_FEATURES,
    model_name="CATBOOST",
    n_trials=30,
    study_name="long_catboost"
)

[I 2026-07-26 11:04:14,855] A new study created in memory with name: long_xgboost
[I 2026-07-26 11:04:29,035] Trial 0 finished with value: 26.510375690959084 and parameters: {'n_estimators': 638, 'learning_rate': 0.024484632673662604, 'max_depth': 6, 'min_child_weight': 5, 'subsample': 0.7676373874677408, 'colsample_bytree': 0.8227026693675435, 'gamma': 0.1967291767393459, 'reg_lambda': 5.814906220555309, 'reg_alpha': 1.6058136637475577}. Best is trial 0 with value: 26.510375690959084.
[I 2026-07-26 11:04:45,795] Trial 1 finished with value: 26.445137442119357 and parameters: {'n_estimators': 910, 'learning_rate': 0.020301129530592807, 'max_depth': 6, 'min_child_weight': 2, 'subsample': 0.7738762213759586, 'colsample_bytree': 0.8960590287073382, 'gamma': 0.2946286473311682, 'reg_lambda': 7.560678063702546, 'reg_alpha': 1.689647977726205}. Best is trial 1 with value: 26.445137442119357.
[I 2026-07-26 11:05:02,540] Trial 2 finished with value: 26.50378574980998 and parameters: {'n_estima

In [17]:
# PRUNING
xgb_long_best_feats, _, _, xgb_long_best_metrics, xgb_long_prune_log = iterative_feature_prune_long_model(
    full_data=Data,
    start_features=LONG_FEATURES,
    best_params=xgb_long_best_params,
    valid_window=LONG_VALID_WINDOW,
    model_name="XGBOOST"
)

catboost_long_best_feats, _, _, catboost_long_best_metrics, catboost_long_prune_log = iterative_feature_prune_long_model(
    full_data=Data,
    start_features=LONG_FEATURES,
    best_params=catboost_long_best_params,
    valid_window=LONG_VALID_WINDOW,
    model_name="CATBOOST"
)

#### DL_LONG

In [18]:
# 1) CONFIG
GRU_SEED = 42
GRU_SEQ_LEN = 18
GRU_LR = 8e-4
GRU_WEIGHT_DECAY = 1e-5
GRU_DROPOUT = 0.25
GRU_EMBED_DIM = 32
GRU_DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
GRU_BATCH_SIZE = 256
GRU_EPOCHS = 10
GRU_HIDDEN_SIZE = 32
GRU_NUM_LAYERS = 1
GRU_HOLDOUT_MONTHS = 12
GRU_RECENT_MONTHS = 4
GRU_ABC_WEIGHT_MAP = {0: 2.5, 1: 1.2, 2: 1.0}
GRU_PROMO_WEIGHT = 1.35
GRU_SUPPLY_WEIGHT = 1.20
GRU_RECURRING_PROMO_WEIGHT = 1.25
GRU_EXPECTED_PROMO_WEIGHT = 1.20
GRU_UNDER_PENALTY = 1.10
GRU_EVAL_DIR = "gru_long_eval_artifacts"
GRU_DEPLOY_DIR = "gru_long_deploy_artifacts"


# FEATURE SETS FOR GRU
GRU_LONG_SEQ_FEATURES = [
    "Clean_Demand",
    "Secondary_Sales_Qty",
    "Primary_Sales_Qty",
    "Free_Qty",
    "Free_Ratio",

    "Bonus_Flag",
    "Bonus_Flag_Lag1",
    "Bonus_Flag_Lag2",
    "Bonus_Frequency_12M",
    "Recurring_Bonus_SKU",
    "Bonus_Cycle_Length",
    "Months_Since_Last_Bonus",
    "Expected_Bonus_Month",
    "Avg_Bonus_Uplift",

    "Last_Bonus_Demand",
    "Post_Bonus_Month",
    "Promo_Decay_Ratio",
    "Demand_Regime_Encoded",
    "Demand_State_Encoded",
    "Current_Stock_Cover",
    "Current_Stockout_Risk",

    "Supply_Constraint_Flag",
    "Supply_Shock",

    "Available_Primary_Inventory_Qty",
    "Distributor_Inventory_Qty",
    "Net_Available_Stock",
    "Stock_Cover_Months",
    "Primary_Stock_Cover",
    "Distributor_Stock_Cover",

    "Lag1", "Lag2", "Lag3", "Lag6", "Lag12",
    "Rolling3M_Mean",
    "Rolling6M_Mean",
    "Rolling3M_Std",
    "Momentum",

    "Month_Sin",
    "Month_Cos"
]
GRU_LONG_STATIC_FEATURES = [
    "ABC_Class",
    "SKU_Mean_Demand",
    "SKU_ZeroRate",
    "SKU_CV",
    "Recurring_Bonus_SKU",
    "Bonus_Cycle_Length",
    "Avg_Bonus_Uplift",
    "Demand_State_Encoded",
    "Last_Bonus_Demand",
    "History_Length"
]
GRU_LONG_SEQ_FEATURES = list(dict.fromkeys(GRU_LONG_SEQ_FEATURES))
GRU_LONG_STATIC_FEATURES = list(dict.fromkeys(GRU_LONG_STATIC_FEATURES))

GRU_LONG_TARGET_COL = ACTUAL_TARGET_COL
GRU_LONG_BASELINE_COL = BASELINE_COL
GRU_LONG_RESIDUAL_COL = MODEL_TARGET_COL
GRU_LONG_RESIDUAL_LOG_COL = "Residual_Target_Log"


# =========================
# GRU DEBUG CONFIG
# =========================
GRU_DEBUG = False
GRU_DEBUG_SKUS = {"600308", "600311", "600315", "600319"}   # add/remove as needed
GRU_DEBUG_MAX_SEQ_ROWS = 8
GRU_MIN_SEQUENCES = 2

In [19]:
# 2) REPRODUCIBILITY
def gru_seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# 3) GRU HELPERS
def gru_signed_log_transform(x):
    x = np.asarray(x, dtype=float)
    return np.sign(x) * np.log1p(np.abs(x))

def gru_signed_log_inverse(x):
    x = np.asarray(x, dtype=float)
    return np.sign(x) * np.expm1(np.abs(x))

def make_item_mapping_from_train(train_df):
    train_df = force_itemcode_str(train_df)

    key_col = "ItemCode_Original" if "ItemCode_Original" in train_df.columns else "ItemCode"
    item_codes = sorted(train_df[key_col].astype(str).unique().tolist())

    return {item: i for i, item in enumerate(item_codes)}

# 4.1) LONG DATA ADAPTER FOR GRU
def get_gru_log_clip_value(train_df, q=0.995):
    residual_abs = train_df[MODEL_TARGET_COL].abs().replace([np.inf, -np.inf], np.nan).dropna()

    if residual_abs.empty:
        return 7.0

    residual_cap = residual_abs.quantile(q)
    return float(np.log1p(max(residual_cap, 1)))

def prepare_long_gru_from_prepared_df(prepared_df, log_clip_value=None):
    df = force_itemcode_str(prepared_df)
    df = df.copy().sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)

    if df.empty:
        return df

    if log_clip_value is None:
        log_clip_value = get_gru_log_clip_value(df)

    df["Residual_Target_Log"] = gru_signed_log_transform(df[MODEL_TARGET_COL].fillna(0))
    df["Residual_Target_Log"] = df["Residual_Target_Log"].clip(
        -log_clip_value,
        log_clip_value
    )

    return df

def prepare_long_gru_eval_data_for_window(full_data, valid_window):
    full_data = add_period_index(full_data)

    train_df = full_data[full_data["Period_Index"] < valid_window["valid_start_idx"]].copy()
    valid_df = full_data[
        full_data["Period_Index"].between(valid_window["valid_start_idx"], valid_window["valid_end_idx"])
    ].copy()

    if train_df.empty or valid_df.empty:
        raise ValueError("No usable LONG rows for GRU window evaluation.")

    train_prep, valid_prep, prep_artifacts = prepare_long_frame_foldsafe(train_df, valid_df)

    if train_prep is None or valid_prep is None:
        raise ValueError("No usable LONG rows after GRU fold-safe preparation.")

    train_prep = recompute_target(train_prep)
    train_prep = add_residual_target(train_prep)
    train_prep = train_prep.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()

    if train_prep is None or valid_prep is None:
        raise ValueError("No usable LONG rows after GRU fold-safe preparation.")

    train_prep["ItemCode_Original"] = train_prep["ItemCode"].astype(str)
    valid_prep["ItemCode_Original"] = valid_prep["ItemCode"].astype(str)

    train_prep, valid_prep, itemcode_categories = encode_itemcode(train_prep, valid_prep)
    prep_artifacts["itemcode_categories"] = itemcode_categories

    gru_log_clip_value = get_gru_log_clip_value(train_prep)

    train_prep = prepare_long_gru_from_prepared_df(train_prep, gru_log_clip_value)
    valid_prep = prepare_long_gru_from_prepared_df(valid_prep, gru_log_clip_value)

    artifacts = {
        "abc_map": prep_artifacts["abc_map"],
        "clip_caps": prep_artifacts["clip_caps"],
        "itemcode_categories": prep_artifacts["itemcode_categories"],
        "gru_log_clip_value": gru_log_clip_value
    }

    return train_prep, valid_prep, artifacts

def prepare_long_gru_deploy_data(full_data):
    deploy_prep, deploy_artifacts = prepare_long_deploy_frame(full_data)
    deploy_prep = recompute_target(deploy_prep)
    deploy_prep = add_residual_target(deploy_prep)
    deploy_prep = deploy_prep.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()
    gru_log_clip_value = get_gru_log_clip_value(deploy_prep)
    deploy_prep = prepare_long_gru_from_prepared_df(
        deploy_prep,
        gru_log_clip_value
    )
    deploy_artifacts["gru_log_clip_value"] = gru_log_clip_value

    return deploy_prep, deploy_artifacts

# 5) Min Debug
def gru_should_debug(sku_code):

    return GRU_DEBUG and str(sku_code) in GRU_DEBUG_SKUS


In [20]:
# 6) DATASET
class LongGRUSequenceDataset(Dataset):
    def __init__(self, X_seq, X_static, X_item, y_res_log, sample_w):
        self.X_seq = torch.tensor(X_seq, dtype=torch.float32)
        self.X_static = torch.tensor(X_static, dtype=torch.float32)
        self.X_item = torch.tensor(X_item, dtype=torch.long)
        self.y_res_log = torch.tensor(y_res_log, dtype=torch.float32)
        self.sample_w = torch.tensor(sample_w, dtype=torch.float32)

    def __len__(self):
        return len(self.y_res_log)

    def __getitem__(self, idx):
        return (
            self.X_seq[idx],
            self.X_static[idx],
            self.X_item[idx],
            self.y_res_log[idx],
            self.sample_w[idx],
        )

# 7) MODEL
class LongGRUResidualForecaster(nn.Module):
    def __init__(
        self,
        num_items,
        seq_input_dim,
        static_input_dim,
        embed_dim=32,
        hidden_size=64,
        num_layers=2,
        dropout=0.25,
    ):
        super().__init__()

        self.item_embedding = nn.Embedding(num_items, embed_dim)

        self.gru = nn.GRU(
            input_size=seq_input_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.seq_fc = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.static_fc = nn.Sequential(
            nn.Linear(static_input_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.head = nn.Sequential(
            nn.Linear(64 + 32 + embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x_seq, x_static, x_item):
        out, _ = self.gru(x_seq)
        last_hidden = out[:, -1, :]

        seq_repr = self.seq_fc(last_hidden)
        static_repr = self.static_fc(x_static)
        item_repr = self.item_embedding(x_item)

        x = torch.cat([seq_repr, static_repr, item_repr], dim=1)
        pred_res_log = self.head(x).squeeze(1)
        return pred_res_log

# 8) LOSS
class WeightedAsymmetricMAELoss(nn.Module):
    def __init__(self, under_penalty=1.75):
        super().__init__()
        self.under_penalty = under_penalty

    def forward(self, preds, targets, sample_weights):
        err = preds - targets
        abs_err = torch.abs(err)
        penalty = torch.where(err < 0, self.under_penalty, 1.0)
        loss = abs_err * penalty * sample_weights
        return loss.mean()

# 9) SCALERS
@dataclass
class LongGRUScalerBundle:
    seq_scaler: StandardScaler
    static_scaler: StandardScaler

def fit_long_gru_scalers(train_df):
    seq_scaler = StandardScaler()
    static_scaler = StandardScaler()

    seq_scaler.fit(train_df[GRU_LONG_SEQ_FEATURES].fillna(0))
    static_scaler.fit(train_df[GRU_LONG_STATIC_FEATURES].fillna(0))

    return LongGRUScalerBundle(
        seq_scaler=seq_scaler,
        static_scaler=static_scaler
    )

# ============================================================
# 10) BUILD SEQUENCES
# ============================================================
def build_long_gru_sequences(df, item_to_idx, scalers, seq_len=18):
    X_seq, X_static, X_item = [], [], []
    y_res_log, sample_w = [], []
    meta = []
    debug_summary = []

    work = force_itemcode_str(df.copy())

    key_col = "ItemCode_Original" if "ItemCode_Original" in work.columns else "ItemCode"
    work[key_col] = work[key_col].astype(str)

    for item, g in work.groupby(key_col):
        item = str(item)

        if item not in item_to_idx:
            if gru_should_debug(item):
                print(f"[GRU BUILD DEBUG] SKU={item} skipped: not in item_to_idx")
            continue

        g = g.sort_values(["Year", "Month_Number"]).reset_index(drop=True)

        usable_idx = g.index[g[GRU_LONG_RESIDUAL_LOG_COL].notna()].tolist()
        seq_count_for_sku = 0

        if gru_should_debug(item):
            debug_summary.append({
                "ItemCode": item,
                "Rows": len(g),
                "Usable_Target_Rows": len(usable_idx),
                "Min_YearMonth": f"{int(g['Year'].iloc[0])}-{int(g['Month_Number'].iloc[0]):02d}" if len(g) else None,
                "Max_YearMonth": f"{int(g['Year'].iloc[-1])}-{int(g['Month_Number'].iloc[-1]):02d}" if len(g) else None,
            })

        valid_sequence_indices = []

        for idx in usable_idx:
            start = idx - seq_len + 1
            if start < 0:
                continue
            valid_sequence_indices.append(idx)

        if len(valid_sequence_indices) < GRU_MIN_SEQUENCES:
            if gru_should_debug(item):
                print(
                    f"[GRU BUILD DEBUG] SKU={item} skipped: "
                    f"only {len(valid_sequence_indices)} valid sequences"
                )
            continue

        for idx in valid_sequence_indices:
            start = idx - seq_len + 1
            if start < 0:
                continue

            seq_slice = g.iloc[start:idx + 1]

            seq_vals = seq_slice[GRU_LONG_SEQ_FEATURES].fillna(0).values
            seq_vals = scalers.seq_scaler.transform(seq_vals)

            static_vals = (
                seq_slice.iloc[-1][GRU_LONG_STATIC_FEATURES]
                .fillna(0)
                .values
                .reshape(1, -1)
            )
            static_vals = scalers.static_scaler.transform(static_vals)[0]

            last_row = seq_slice.iloc[-1]

            abc_class = int(last_row["ABC_Class"])
            bonus_flag = int(last_row["Bonus_Flag"])
            supply_flag = int(last_row["Supply_Constraint_Flag"])
            recurring_flag = int(last_row["Recurring_Bonus_SKU"])
            expected_bonus_flag = int(last_row["Expected_Bonus_Month"])

            w = GRU_ABC_WEIGHT_MAP.get(abc_class, 1.0)
            w *= GRU_PROMO_WEIGHT if bonus_flag == 1 else 1.0
            w *= GRU_SUPPLY_WEIGHT if supply_flag == 1 else 1.0
            w *= GRU_RECURRING_PROMO_WEIGHT if recurring_flag == 1 else 1.0
            w *= GRU_EXPECTED_PROMO_WEIGHT if expected_bonus_flag == 1 else 1.0

            X_seq.append(seq_vals.astype(np.float32))
            X_static.append(static_vals.astype(np.float32))
            X_item.append(item_to_idx[item])
            y_res_log.append(np.float32(g.iloc[idx][GRU_LONG_RESIDUAL_LOG_COL]))
            sample_w.append(np.float32(w))

            seq_count_for_sku += 1

            meta.append({
                "ItemCode": item,
                "ItemCode_Original": item,
                "Year": int(g.iloc[idx]["Year"]),
                "Month_Number": int(g.iloc[idx]["Month_Number"]),
                "ABC_Class": abc_class,
                "Bonus_Flag": bonus_flag,
                "Supply_Constraint_Flag": supply_flag,
                "Recurring_Bonus_SKU": recurring_flag,
                "Expected_Bonus_Month": expected_bonus_flag,
                "Actual": float(g.iloc[idx][GRU_LONG_TARGET_COL]),
                "Residual_Baseline": float(g.iloc[idx][GRU_LONG_BASELINE_COL]),
                "Residual_Target": float(g.iloc[idx][GRU_LONG_RESIDUAL_COL]),
            })

        if gru_should_debug(item):
            print(
                f"[GRU BUILD DEBUG] SKU={item} | rows={len(g)} | "
                f"usable_idx={len(usable_idx)} | built_sequences={seq_count_for_sku}"
            )

    if GRU_DEBUG and debug_summary:
        print(pd.DataFrame(debug_summary).head(GRU_DEBUG_MAX_SEQ_ROWS))

    meta_columns = [
        "ItemCode",
        "ItemCode_Original",
        "Year",
        "Month_Number",
        "ABC_Class",
        "Bonus_Flag",
        "Supply_Constraint_Flag",
        "Recurring_Bonus_SKU",
        "Expected_Bonus_Month",
        "Actual",
        "Residual_Baseline",
        "Residual_Target",
    ]

    meta_df = pd.DataFrame(meta, columns=meta_columns)

    return (
        np.array(X_seq, dtype=np.float32),
        np.array(X_static, dtype=np.float32),
        np.array(X_item, dtype=np.int64),
        np.array(y_res_log, dtype=np.float32),
        np.array(sample_w, dtype=np.float32),
        meta_df,
    )

def filter_sequence_pack_by_period_index(X_seq, X_static, X_item, y, w, meta_df, start_idx, end_idx):
    if meta_df.empty:
        return X_seq[:0], X_static[:0], X_item[:0], y[:0], w[:0], meta_df.copy()

    meta = meta_df.copy()
    meta["Period_Index"] = meta["Year"].astype(int) * 12 + meta["Month_Number"].astype(int)

    mask = meta["Period_Index"].between(start_idx, end_idx)
    idx = np.where(mask.values)[0]

    return (
        X_seq[idx],
        X_static[idx],
        X_item[idx],
        y[idx],
        w[idx],
        meta.iloc[idx].reset_index(drop=True)
    )

# ============================================================
# 12) TRAIN / PREDICT
# ============================================================
def train_long_gru_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    for x_seq, x_static, x_item, y_res_log, sample_w in loader:
        x_seq = x_seq.to(GRU_DEVICE)
        x_static = x_static.to(GRU_DEVICE)
        x_item = x_item.to(GRU_DEVICE)
        y_res_log = y_res_log.to(GRU_DEVICE)
        sample_w = sample_w.to(GRU_DEVICE)

        optimizer.zero_grad()
        preds = model(x_seq, x_static, x_item)
        loss = criterion(preds, y_res_log, sample_w)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * len(y_res_log)

    return total_loss / len(loader.dataset)

def split_gru_train_valid_before_window(X_seq, X_static, X_item, y, w, meta_df, valid_window):
    if meta_df.empty:
        return None, None

    meta = meta_df.copy()
    meta["Period_Index"] = meta["Year"].astype(int) * 12 + meta["Month_Number"].astype(int)

    trainable_meta = meta[meta["Period_Index"] < valid_window["valid_start_idx"]].copy()

    if trainable_meta.empty:
        return None, None

    unique_periods = (
        trainable_meta[["Period_Index"]]
        .drop_duplicates()
        .sort_values("Period_Index")
        .reset_index(drop=True)
    )

    if len(unique_periods) >= 4:
        valid_periods = unique_periods.tail(4)["Period_Index"].tolist()
        train_mask = meta["Period_Index"].isin(unique_periods.iloc[:-4]["Period_Index"])
        valid_mask = meta["Period_Index"].isin(valid_periods)
    else:
        n = len(meta_df)
        split_idx = int(n * 0.85)
        train_mask = np.zeros(n, dtype=bool)
        valid_mask = np.zeros(n, dtype=bool)
        train_mask[:split_idx] = True
        valid_mask[split_idx:] = True

    def take(mask):
        idx = np.where(mask)[0]
        return (
            X_seq[idx],
            X_static[idx],
            X_item[idx],
            y[idx],
            w[idx],
            meta_df.iloc[idx].reset_index(drop=True)
        )

    return take(train_mask), take(valid_mask)

@torch.no_grad()
def predict_long_gru_residual_log(model, loader):
    model.eval()

    preds_all, y_all = [], []

    for x_seq, x_static, x_item, y_res_log, sample_w in loader:
        x_seq = x_seq.to(GRU_DEVICE)
        x_static = x_static.to(GRU_DEVICE)
        x_item = x_item.to(GRU_DEVICE)

        preds = model(x_seq, x_static, x_item).cpu().numpy()
        preds_all.append(preds)
        y_all.append(y_res_log.numpy())

    preds_all = np.concatenate(preds_all)
    y_all = np.concatenate(y_all)
    return preds_all, y_all

def evaluate_long_gru_on_loader(model, loader, meta_df, log_clip_value=7.0):
    pred_res_log, true_res_log = predict_long_gru_residual_log(model, loader)

    pred_res_log = np.clip(pred_res_log, -log_clip_value, log_clip_value)
    true_res_log = np.clip(true_res_log, -log_clip_value, log_clip_value)

    pred_residual = gru_signed_log_inverse(pred_res_log)
    true_residual = gru_signed_log_inverse(true_res_log)

    out = meta_df.copy()
    out["Pred_Residual_Log"] = pred_res_log
    out["Pred_Residual"] = pred_residual
    out["True_Residual"] = true_residual

    out["Pred"] = out["Residual_Baseline"] + out["Pred_Residual"]
    out["Pred"] = out["Pred"].clip(lower=0)

    out["Error"] = out["Actual"] - out["Pred"]
    out["Abs_Error"] = np.abs(out["Error"])

    metrics = evaluate_all_metrics(out["Actual"].values, out["Pred"].values)
    return metrics, out

def fit_long_gru_model(train_dataset, valid_dataset, valid_meta, num_items, seq_input_dim, static_input_dim, log_clip_value=7.0):
    train_loader = DataLoader(train_dataset, batch_size=GRU_BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=(GRU_DEVICE.type == "cuda"))
    valid_loader = DataLoader(valid_dataset, batch_size=GRU_BATCH_SIZE, shuffle=False)

    print("GRU_DEVICE:", GRU_DEVICE)
    print("Train batches:", len(train_loader))
    print("Valid batches:", len(valid_loader))

    model = LongGRUResidualForecaster(
        num_items=num_items,
        seq_input_dim=seq_input_dim,
        static_input_dim=static_input_dim,
        embed_dim=GRU_EMBED_DIM,
        hidden_size=GRU_HIDDEN_SIZE,
        num_layers=GRU_NUM_LAYERS,
        dropout=GRU_DROPOUT,
    ).to(GRU_DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=GRU_LR, weight_decay=GRU_WEIGHT_DECAY)
    criterion = WeightedAsymmetricMAELoss(under_penalty=GRU_UNDER_PENALTY)

    best_valid_wmape = float("inf")
    best_state = None
    patience = 8
    wait = 0

    for epoch in range(1, GRU_EPOCHS + 1):
        train_loss = train_long_gru_one_epoch(model, train_loader, optimizer, criterion)
        valid_metrics, _ = evaluate_long_gru_on_loader(model, valid_loader, valid_meta, log_clip_value=log_clip_value)

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.5f} | "
            f"Valid WMAPE: {valid_metrics['WMAPE']:.4f} | "
            f"Valid Bias: {valid_metrics['Bias']:.4f}"
        )

        if valid_metrics["WMAPE"] < best_valid_wmape:
            best_valid_wmape = valid_metrics["WMAPE"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print("Early stopping triggered.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model

# 13) STANDARDIZE OUTPUT FOR MODEL COMPARISON
def convert_gru_long_output_for_comparison(test_result_df):
    out = force_itemcode_str(test_result_df.copy())

    if "ItemCode_Original" in out.columns:
        out["ItemCode_Original"] = out["ItemCode_Original"].astype(str)
        out["ItemCode"] = out["ItemCode_Original"].astype(str)
    else:
        out["ItemCode"] = out["ItemCode"].astype(str)
        out["ItemCode_Original"] = out["ItemCode"]

    out["Segment"] = "LONG"
    out["Model_Name"] = "GRU"
    out["Actual"] = pd.to_numeric(out["Actual"], errors="coerce")
    out["Pred"] = pd.to_numeric(out["Pred"], errors="coerce").clip(lower=0)
    out["Error"] = out["Actual"] - out["Pred"]
    out["Abs_Error"] = np.abs(out["Error"])

    keep_cols = [
        "ItemCode",
        "ItemCode_Original",
        "Year",
        "Month_Number",
        "Actual",
        "Pred",
        "Error",
        "Abs_Error",
        "Segment",
        "Model_Name"
    ]

    extra_cols = [c for c in out.columns if c not in keep_cols]
    return out[keep_cols + extra_cols].copy()

# 14) LONG GRU HOLDOUT EVALUATION PIPELINE
def run_long_gru_window_evaluation(full_data, valid_window, run_label="GRU_WINDOW"):
    print(f"\n========== LONG GRU EVALUATION PIPELINE → {run_label} ==========")

    train_prep, valid_prep, artifacts_meta = prepare_long_gru_eval_data_for_window(
        full_data=full_data,
        valid_window=valid_window
    )

    if train_prep.empty or valid_prep.empty:
        raise ValueError(f"Prepared LONG GRU data is empty for {run_label}.")

    item_to_idx = make_item_mapping_from_train(train_prep)

    key_col_train = "ItemCode_Original" if "ItemCode_Original" in train_prep.columns else "ItemCode"
    key_col_valid = "ItemCode_Original" if "ItemCode_Original" in valid_prep.columns else "ItemCode"

    train_prep[key_col_train] = train_prep[key_col_train].astype(str)
    valid_prep[key_col_valid] = valid_prep[key_col_valid].astype(str)

    train_prep = train_prep[train_prep[key_col_train].isin(item_to_idx.keys())].copy()
    valid_prep = valid_prep[valid_prep[key_col_valid].isin(item_to_idx.keys())].copy()

    if train_prep.empty:
        raise ValueError(f"GRU train_prep became empty after key filtering for {run_label}.")
    if valid_prep.empty:
        raise ValueError(f"GRU valid_prep became empty after key filtering for {run_label}.")

    scalers = fit_long_gru_scalers(train_prep)

    print("GRU key_col_train:", key_col_train)
    print("GRU key_col_valid:", key_col_valid)
    print("GRU train_prep rows after key filter:", len(train_prep))
    print("GRU valid_prep rows after key filter:", len(valid_prep))
    print("GRU unique train SKUs:", train_prep[key_col_train].nunique())
    print("GRU unique valid SKUs:", valid_prep[key_col_valid].nunique())

    train_seq_source = train_prep.copy()
    valid_seq_source = pd.concat([train_prep, valid_prep], ignore_index=True).sort_values(
        ["ItemCode", "Year", "Month_Number"]
    )

    train_all = build_long_gru_sequences(
        train_seq_source, item_to_idx, scalers, seq_len=GRU_SEQ_LEN
    )
    valid_all = build_long_gru_sequences(
        valid_seq_source, item_to_idx, scalers, seq_len=GRU_SEQ_LEN
    )

    X_seq_train_all, X_static_train_all, X_item_train_all, y_train_all, w_train_all, meta_train_all = train_all

    X_seq_valid_eval, X_static_valid_eval, X_item_valid_eval, y_valid_eval, w_valid_eval, meta_valid_eval = filter_sequence_pack_by_period_index(
        *valid_all,
        start_idx=valid_window["valid_start_idx"],
        end_idx=valid_window["valid_end_idx"]
    )

    print("GRU total train sequences:", len(y_train_all))
    print("GRU total valid eval sequences:", len(y_valid_eval))

    if len(y_train_all) == 0:
        raise ValueError(f"No LONG GRU train sequences were created for {run_label}.")
    if len(y_valid_eval) == 0:
        raise ValueError(f"No LONG GRU eval sequences were created for {run_label}.")

    train_pack, early_valid_pack = split_gru_train_valid_before_window(
        X_seq_train_all,
        X_static_train_all,
        X_item_train_all,
        y_train_all,
        w_train_all,
        meta_train_all,
        valid_window=valid_window
    )

    if train_pack is None or early_valid_pack is None:
        raise ValueError(f"GRU split_gru_train_valid_before_window returned None for {run_label}.")

    X_seq_train, X_static_train, X_item_train, y_train, w_train, meta_train = train_pack
    X_seq_early_valid, X_static_early_valid, X_item_early_valid, y_early_valid, w_early_valid, meta_early_valid = early_valid_pack

    print("GRU final train sequences:", len(y_train))
    print("GRU early valid sequences:", len(y_early_valid))

    if len(y_train) == 0 or len(y_early_valid) == 0:
        raise ValueError(f"No LONG GRU train/early-valid split could be created for {run_label}.")

    train_dataset = LongGRUSequenceDataset(
        X_seq_train, X_static_train, X_item_train, y_train, w_train
    )
    early_valid_dataset = LongGRUSequenceDataset(
        X_seq_early_valid, X_static_early_valid, X_item_early_valid, y_early_valid, w_early_valid
    )
    eval_dataset = LongGRUSequenceDataset(
        X_seq_valid_eval, X_static_valid_eval, X_item_valid_eval, y_valid_eval, w_valid_eval
    )

    model = fit_long_gru_model(
        train_dataset=train_dataset,
        valid_dataset=early_valid_dataset,
        valid_meta=meta_early_valid,
        num_items=len(item_to_idx),
        seq_input_dim=X_seq_train.shape[2],
        static_input_dim=X_static_train.shape[1],
        log_clip_value=artifacts_meta["gru_log_clip_value"]
    )

    eval_loader = DataLoader(eval_dataset, batch_size=GRU_BATCH_SIZE, shuffle=False)
    eval_metrics, eval_result_df = evaluate_long_gru_on_loader(model, eval_loader, meta_valid_eval, log_clip_value=artifacts_meta["gru_log_clip_value"])
    eval_std_df = convert_gru_long_output_for_comparison(eval_result_df)

    artifacts = {
        "model": model,
        "model_type": "GRU",
        "model_name": "GRU",
        "segment": "LONG",
        "seq_features": GRU_LONG_SEQ_FEATURES,
        "static_features": GRU_LONG_STATIC_FEATURES,
        "seq_len": GRU_SEQ_LEN,
        "embed_dim": GRU_EMBED_DIM,
        "hidden_size": GRU_HIDDEN_SIZE,
        "num_layers": GRU_NUM_LAYERS,
        "dropout": GRU_DROPOUT,
        "item_to_idx": item_to_idx,
        "abc_map": artifacts_meta["abc_map"],
        "clip_caps": artifacts_meta["clip_caps"],
        "itemcode_categories": artifacts_meta["itemcode_categories"],
        "gru_log_clip_value": artifacts_meta["gru_log_clip_value"],
        "eval_metrics": eval_metrics,
        "run_label": run_label
    }

    return artifacts, eval_result_df, eval_std_df, eval_metrics, scalers


#### COMPARISON

	•	per-SKU comparison table
	•	champion map: ItemCode -> Best_Model
	•	deployment artifacts per model
	•	inference routing:
	•	lookup SKU in champion map
	•	load chosen model
	•	forecast with that model
	•	save Used_Model

` ============================================================ `
  * LONG MODEL COMPARISON + CHAMPION MAP
  * DUAL-VIEW SELECTION
  * Model A = stable model trained before Holdout12 start evaluated on Holdout12 + Recent4
  * Model B = recent-aware model trained before Recent4 start evaluated on Recent4 only

 * FINAL SCORE: 0.30 Holdout12 + 0.25 Recent4A + 0.25 Recent4B + 0.20 Bias

NOTE:
This comparison includes XGBOOST, CATBOOST, and GRU.
` ============================================================ `


In [21]:
RECENT_MONTHS = TIME_WINDOWS.get("recent_months", 4)

LONG_RECENT4_WINDOW = get_long_recent_window(
    time_windows=TIME_WINDOWS,
    recent_months=RECENT_MONTHS
)

LONG_HOLDOUT12_WINDOW = get_long_pre_recent_holdout_window(
    time_windows=TIME_WINDOWS,
    holdout_months=12,
    recent_months=RECENT_MONTHS
)

LONG_MODEL_A_WINDOW = combine_long_model_a_window(
    time_windows=TIME_WINDOWS,
    holdout_months=12,
    recent_months=RECENT_MONTHS
)

LONG_MODEL_CONFIG = {
    "XGBOOST": {
        "features": xgb_long_best_feats,
        "params": xgb_long_best_params
    },
    "CATBOOST": {
        "features": catboost_long_best_feats,
        "params": catboost_long_best_params
    }
}

MODEL_PRIORITY = {"CATBOOST": 1, "XGBOOST": 2, "GRU": 3}

print("\nLONG WINDOWS")
print("LONG_HOLDOUT12_WINDOW:", LONG_HOLDOUT12_WINDOW)
print("LONG_RECENT4_WINDOW:", LONG_RECENT4_WINDOW)
print("LONG_MODEL_A_WINDOW:", LONG_MODEL_A_WINDOW)


LONG WINDOWS
LONG_HOLDOUT12_WINDOW: {'valid_start_idx': 24296, 'valid_end_idx': 24307, 'holdout_months': 12}
LONG_RECENT4_WINDOW: {'valid_start_idx': 24308, 'valid_end_idx': 24311, 'recent_months': 4}
LONG_MODEL_A_WINDOW: {'valid_start_idx': 24296, 'valid_end_idx': 24311, 'holdout_months': 12, 'recent_months': 4}


In [22]:
def build_sku_behavior_profile(df):
    df = force_itemcode_str(df)
    df = df.copy().sort_values(["ItemCode", "Year", "Month_Number"])

    def safe_autocorr(x, lag):
        x = x.dropna()
        if len(x) <= lag or x.std() == 0:
            return 0
        return x.autocorr(lag=lag)

    profile = (
        df.groupby("ItemCode")
        .agg(
            SKU_Mean_Demand=("Clean_Demand", "mean"),
            SKU_Std_Demand=("Clean_Demand", "std"),
            SKU_Zero_Rate=("Clean_Demand", lambda x: (x == 0).mean()),
            SKU_NonZero_Months=("Clean_Demand", lambda x: (x > 0).sum()),
            SKU_Total_Months=("Clean_Demand", "count"),
            SKU_Max_Demand=("Clean_Demand", "max"),
            Autocorr_Lag3=("Clean_Demand", lambda x: safe_autocorr(x, 3)),
            Autocorr_Lag6=("Clean_Demand", lambda x: safe_autocorr(x, 6)),
            Promo_Rate=("Bonus_Flag", "mean")
        )
        .reset_index()
    )

    profile["SKU_Std_Demand"] = profile["SKU_Std_Demand"].fillna(0)

    profile["SKU_CV"] = np.where(
        profile["SKU_Mean_Demand"] <= 0,
        0,
        profile["SKU_Std_Demand"] / (profile["SKU_Mean_Demand"] + 1)
    )

    profile["Peak_Ratio"] = np.where(
        profile["SKU_Mean_Demand"] <= 0,
        0,
        profile["SKU_Max_Demand"] / (profile["SKU_Mean_Demand"] + 1)
    )

    profile["Intermittent_Flag"] = np.where(
        (profile["SKU_Zero_Rate"] >= 0.40) |
        (profile["SKU_NonZero_Months"] < 8),
        1, 0
    )

    profile["Highly_Volatile_Flag"] = np.where(
        profile["SKU_CV"] >= 1.5,
        1, 0
    )

    profile["Spiky_Flag"] = np.where(
        profile["Peak_Ratio"] >= 2.5,
        1, 0
    )

    profile["Cyclic_Flag"] = np.where(
        (profile["Autocorr_Lag3"] >= 0.50) |
        (profile["Autocorr_Lag6"] >= 0.50),
        1, 0
    )

    return profile

def assign_final_routing(df):
    df = force_itemcode_str(df).copy()

    metric_cols = [
        "Best_Model_Score",
        "Best_Model_Holdout12_WMAPE",
        "Best_Model_Recent4A_WMAPE",
        "Best_Model_Recent4B_WMAPE",
        "SKU_CV",
        "SKU_Zero_Rate",
        "Intermittent_Flag",
        "Highly_Volatile_Flag"
    ]

    for c in metric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # stricter unreliability
    df["Unreliable_Flag"] = np.where(
        (
            (df["Best_Model_Holdout12_WMAPE"] > 90) &
            (df["Best_Model_Recent4B_WMAPE"] > 90)
        ) |
        (
            (df["Best_Model_Score"] > 100) &
            (df["Best_Model_Recent4B_WMAPE"] > 90)
        ),
        1, 0
    )

    # fallback only for extreme cases
    df["Use_Fallback"] = np.where(
        (
            (df["Intermittent_Flag"] == 1) &
            (df["Best_Model_Recent4B_WMAPE"] > 90)
        ) |
        (
            df["Unreliable_Flag"] == 1
        ) |
        (
            (df["Highly_Volatile_Flag"] == 1) &
            (df["Best_Model_Recent4B_WMAPE"] > 110)
        ),
        1, 0
    )

    df["Final_Model"] = np.where(
        df["Use_Fallback"] == 1,
        "FALLBACK",
        df["Best_Model"]
    )

    df["Fallback_Type"] = np.select(
        [
            (df["Intermittent_Flag"] == 1) & (df["Use_Fallback"] == 1),
            (df["Highly_Volatile_Flag"] == 1) & (df["Use_Fallback"] == 1),
            (df["Unreliable_Flag"] == 1) & (df["Use_Fallback"] == 1)
        ],
        [
            "ROLLING3",
            "MAX_LAG1_ROLL3",
            "ROLLING3"
        ],
        default="NONE"
    )

    return df

def eval_long_tree_model(model_name, config, valid_window, label):
    _, eval_df, metrics, _ = train_eval_validation_long_model(
        full_data=Data,
        feature_cols=config["features"],
        best_params=config["params"],
        valid_window=valid_window,
        model_name=model_name
    )

    std_df = standardize_long_model_output(
        df=eval_df,
        actual_col=ACTUAL_TARGET_COL,
        pred_col="Pred",
        item_col="ItemCode_Original",
        model_name=model_name,
        segment="LONG"
    )

    print(f"\n===== {model_name} {label} METRICS =====")
    print(metrics)

    return std_df

def clean_long_eval_df(df):
    df = df.copy()

    for c in ["ItemCode", "ItemCode_Original", "Model_Name"]:
        df[c] = df[c].astype(str)

    for c in ["Actual", "Pred", "Abs_Error", "Year", "Month_Number"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df.dropna(
        subset=["ItemCode", "Model_Name", "Actual", "Pred", "Abs_Error", "Year", "Month_Number"]
    ).copy()

    df["Period_Index"] = df["Year"].astype(int) * 12 + df["Month_Number"].astype(int)

    return df

def split_model_a_eval(df):
    holdout12_periods = set(range(
        LONG_HOLDOUT12_WINDOW["valid_start_idx"],
        LONG_HOLDOUT12_WINDOW["valid_end_idx"] + 1
    ))

    recent4_periods = set(range(
        LONG_RECENT4_WINDOW["valid_start_idx"],
        LONG_RECENT4_WINDOW["valid_end_idx"] + 1
    ))

    return (
        df[df["Period_Index"].isin(holdout12_periods)].copy(),
        df[df["Period_Index"].isin(recent4_periods)].copy()
    )

def add_behavior_type(df):
    behavior_lookup_df = (
        Data_long.copy()
        .sort_values(["ItemCode", "Year", "Month_Number"])
        .groupby("ItemCode")
        .tail(1)[["ItemCode", "Behavior_Type"]]
    )

    behavior_lookup_df["ItemCode"] = behavior_lookup_df["ItemCode"].astype(str)

    out = df.copy()
    out["ItemCode"] = out["ItemCode"].astype(str)
    out = out.drop(columns=["Behavior_Type"], errors="ignore")
    out = out.merge(behavior_lookup_df, on="ItemCode", how="left")
    out["Behavior_Type"] = out["Behavior_Type"].fillna("UNKNOWN")

    return out

def build_sku_metric_summary(df, prefix):
    out = (
        df.groupby(["ItemCode", "Model_Name"], as_index=False)
        .agg(
            Months=("Month_Number", "count"),
            Actual_Sum=("Actual", "sum"),
            Pred_Sum=("Pred", "sum"),
            MAE=("Abs_Error", "mean"),
            Total_Abs_Error=("Abs_Error", "sum")
        )
    )

    out[f"{prefix}_WMAPE"] = np.where(
        out["Actual_Sum"] > 0,
        out["Total_Abs_Error"] / out["Actual_Sum"] * 100,
        np.nan
    )

    out[f"{prefix}_Bias"] = np.where(
        out["Actual_Sum"] > 0,
        (out["Pred_Sum"] - out["Actual_Sum"]) / out["Actual_Sum"] * 100,
        np.nan
    )

    return out.rename(columns={
        "Months": f"{prefix}_Months",
        "Actual_Sum": f"{prefix}_Actual_Sum",
        "Pred_Sum": f"{prefix}_Pred_Sum",
        "MAE": f"{prefix}_MAE",
        "Total_Abs_Error": f"{prefix}_Total_Abs_Error"
    })


In [23]:
# ============================================================
# 1) EVALUATE MODEL A AND MODEL B
# ============================================================

model_a_outputs = []
model_b_outputs = []

for model_name, config in LONG_MODEL_CONFIG.items():
    model_a_outputs.append(
        eval_long_tree_model(model_name, config, LONG_MODEL_A_WINDOW, "MODEL_A")
    )

    model_b_outputs.append(
        eval_long_tree_model(model_name, config, LONG_RECENT4_WINDOW, "MODEL_B_RECENT4")
    )

# GRU MODEL A
gru_seed_everything(GRU_SEED)
gru_long_model_a_artifacts, gru_long_model_a_eval_df_raw, gru_long_model_a_eval_std, gru_long_model_a_eval_metrics, gru_long_model_a_scalers = run_long_gru_window_evaluation(
    full_data=Data,
    valid_window=LONG_MODEL_A_WINDOW,
    run_label="LONG_MODEL_A"
)
model_a_outputs.append(gru_long_model_a_eval_std)

# GRU MODEL B
gru_seed_everything(GRU_SEED)
gru_long_model_b_artifacts, gru_long_model_b_recent_df_raw, gru_long_model_b_recent_std, gru_long_model_b_recent_metrics, gru_long_model_b_scalers = run_long_gru_window_evaluation(
    full_data=Data,
    valid_window=LONG_RECENT4_WINDOW,
    run_label="LONG_MODEL_B_RECENT4"
)
model_b_outputs.append(gru_long_model_b_recent_std)


# ============================================================
# 2) COMBINE EVALUATION OUTPUTS
# ============================================================

long_model_a_eval_df = clean_long_eval_df(pd.concat(model_a_outputs, ignore_index=True))
long_model_b_recent_df = clean_long_eval_df(pd.concat(model_b_outputs, ignore_index=True))

long_model_a_holdout12_df, long_model_a_recent4_df = split_model_a_eval(long_model_a_eval_df)

long_model_a_holdout12_df = add_behavior_type(long_model_a_holdout12_df)
long_model_a_recent4_df = add_behavior_type(long_model_a_recent4_df)
long_model_b_recent_df = add_behavior_type(long_model_b_recent_df)

print("\nMODEL A Holdout12 rows:", len(long_model_a_holdout12_df))
print("MODEL A Recent4 rows:", len(long_model_a_recent4_df))
print("MODEL B Recent4 rows:", len(long_model_b_recent_df))


# ============================================================
# 3) MODEL PERFORMANCE TABLES
# ============================================================

for model_name in ["XGBOOST", "CATBOOST", "GRU"]:
    for label, df in [
        ("HOLDOUT12", long_model_a_holdout12_df),
        ("RECENT4A", long_model_a_recent4_df),
        ("RECENT4B", long_model_b_recent_df)
    ]:
        temp = df[df["Model_Name"] == model_name].copy()
        if not temp.empty:
            print_model_eval_report(temp, title=f"{model_name} LONG → {label}")

long_holdout12_model_table = build_model_summary_table(long_model_a_holdout12_df)
long_recent4a_model_table = build_model_summary_table(long_model_a_recent4_df)
long_recent4b_model_table = build_model_summary_table(long_model_b_recent_df)


# ============================================================
# 4) SKU MODEL SUMMARY
# ============================================================

h12_summary = build_sku_metric_summary(long_model_a_holdout12_df, "Holdout12")
r4a_summary = build_sku_metric_summary(long_model_a_recent4_df, "Recent4A")
r4b_summary = build_sku_metric_summary(long_model_b_recent_df, "Recent4B")

long_sku_model_summary = (
    h12_summary
    .merge(r4a_summary, on=["ItemCode", "Model_Name"], how="left")
    .merge(r4b_summary, on=["ItemCode", "Model_Name"], how="left")
)

for prefix in ["Recent4A", "Recent4B"]:
    for c in ["Months", "Actual_Sum", "Pred_Sum", "Total_Abs_Error"]:
        long_sku_model_summary[f"{prefix}_{c}"] = long_sku_model_summary[f"{prefix}_{c}"].fillna(0)

long_sku_model_summary["Recent4A_MAE"] = long_sku_model_summary["Recent4A_MAE"].fillna(long_sku_model_summary["Holdout12_MAE"])
long_sku_model_summary["Recent4A_WMAPE"] = long_sku_model_summary["Recent4A_WMAPE"].fillna(long_sku_model_summary["Holdout12_WMAPE"])
long_sku_model_summary["Recent4A_Bias"] = long_sku_model_summary["Recent4A_Bias"].fillna(long_sku_model_summary["Holdout12_Bias"])

long_sku_model_summary["Recent4B_MAE"] = long_sku_model_summary["Recent4B_MAE"].fillna(long_sku_model_summary["Recent4A_MAE"])
long_sku_model_summary["Recent4B_WMAPE"] = long_sku_model_summary["Recent4B_WMAPE"].fillna(long_sku_model_summary["Recent4A_WMAPE"])
long_sku_model_summary["Recent4B_Bias"] = long_sku_model_summary["Recent4B_Bias"].fillna(long_sku_model_summary["Recent4A_Bias"])

long_sku_model_summary["Champion_Score"] = (
    0.30 * long_sku_model_summary["Holdout12_WMAPE"] +
    0.25 * long_sku_model_summary["Recent4A_WMAPE"] +
    0.25 * long_sku_model_summary["Recent4B_WMAPE"] +
    0.20 * np.abs(long_sku_model_summary["Recent4B_Bias"])
)

long_sku_model_summary.loc[
    long_sku_model_summary["Recent4B_Bias"] > 20,
    "Champion_Score"
] += 5.0


===== XGBOOST MODEL_A METRICS =====
{'WMAPE': 25.144515427001068, 'Bias': -2.1710521547909094, 'MAE': 1258.2981384657835, 'RMSE': 3377.8442635261968, 'Underforecast_Rate': 13.657783790895989}

===== XGBOOST MODEL_B_RECENT4 METRICS =====
{'WMAPE': 24.39766890829248, 'Bias': -5.078135459304812, 'MAE': 1253.8691060667823, 'RMSE': 3530.6441256719286, 'Underforecast_Rate': 14.737902183798646}

===== CATBOOST MODEL_A METRICS =====
{'WMAPE': 24.699500601126953, 'Bias': 0.06022189302795673, 'MAE': 1236.0284181121447, 'RMSE': 3336.217199088751, 'Underforecast_Rate': 12.319639354049498}

===== CATBOOST MODEL_B_RECENT4 METRICS =====
{'WMAPE': 23.70192483256822, 'Bias': -4.4049504544947675, 'MAE': 1218.1127391139078, 'RMSE': 3360.5322978849103, 'Underforecast_Rate': 14.053437643531497}

========== LONG GRU EVALUATION PIPELINE → LONG_MODEL_A ==========
GRU key_col_train: ItemCode_Original
GRU key_col_valid: ItemCode_Original
GRU train_prep rows after key filter: 22818
GRU valid_prep rows after key

In [24]:
# ============================================================
# BEHAVIOR-LEVEL MODEL PERFORMANCE
# ============================================================

def build_behavior_model_table(df, label):
    out = (
        df.groupby(["Behavior_Type", "Model_Name"], as_index=False)
        .agg(
            Actual_Sum=("Actual", "sum"),
            Pred_Sum=("Pred", "sum"),
            Abs_Error_Sum=("Abs_Error", "sum"),
            MAE=("Abs_Error", "mean"),
            SKU_Count=("ItemCode", "nunique")
        )
    )

    out[f"{label}_WMAPE"] = np.where(
        out["Actual_Sum"] > 0,
        out["Abs_Error_Sum"] / out["Actual_Sum"] * 100,
        np.nan
    )

    out[f"{label}_Bias"] = np.where(
        out["Actual_Sum"] > 0,
        (out["Pred_Sum"] - out["Actual_Sum"]) / out["Actual_Sum"] * 100,
        np.nan
    )

    return out[[
        "Behavior_Type", "Model_Name", "SKU_Count",
        f"{label}_WMAPE", f"{label}_Bias"
    ]]

behavior_h12 = build_behavior_model_table(long_model_a_holdout12_df, "H12")
behavior_r4a = build_behavior_model_table(long_model_a_recent4_df, "R4A")
behavior_r4b = build_behavior_model_table(long_model_b_recent_df, "R4B")

behavior_model_summary = (
    behavior_h12
    .merge(behavior_r4a, on=["Behavior_Type", "Model_Name"], how="outer")
    .merge(behavior_r4b, on=["Behavior_Type", "Model_Name"], how="outer")
)

for c in ["H12_WMAPE", "R4A_WMAPE", "R4B_WMAPE"]:
    behavior_model_summary[c] = behavior_model_summary[c].fillna(behavior_model_summary[c].median())

for c in ["H12_Bias", "R4A_Bias", "R4B_Bias"]:
    behavior_model_summary[c] = behavior_model_summary[c].fillna(0)

behavior_model_summary["Behavior_Champion_Score"] = (
    0.30 * behavior_model_summary["H12_WMAPE"] +
    0.30 * behavior_model_summary["R4A_WMAPE"] +
    0.40 * behavior_model_summary["R4B_WMAPE"]
)

behavior_model_summary["Behavior_Model_Unreliable"] = np.where(
    (
        (behavior_model_summary["R4B_WMAPE"] > 100) |
        (np.abs(behavior_model_summary["R4B_Bias"]) > 45)
    ),
    1, 0
)

behavior_champion_df = (
    behavior_model_summary
    .sort_values([
        "Behavior_Type",
        "Behavior_Model_Unreliable",
        "Behavior_Champion_Score",
        "R4B_WMAPE"
    ])
    .groupby("Behavior_Type")
    .head(1)
    .rename(columns={
        "Model_Name": "Behavior_Best_Model",
        "Behavior_Champion_Score": "Behavior_Best_Model_Score",
        "R4B_WMAPE": "Behavior_Best_Model_Recent4B_WMAPE"
    })
    [["Behavior_Type", "Behavior_Best_Model", "Behavior_Best_Model_Score", "Behavior_Best_Model_Recent4B_WMAPE"]]
)

print("\n===== BEHAVIOR CHAMPION TABLE =====")
print(behavior_champion_df)

# ============================================================
# BEHAVIOR + RELIABILITY ADJUSTMENT
# ============================================================

latest_behavior_df = (
    Data_long.copy()
    .sort_values(["ItemCode", "Year", "Month_Number"])
    .groupby("ItemCode")
    .tail(1)[["ItemCode", "Behavior_Type"]]
)

latest_behavior_df["ItemCode"] = latest_behavior_df["ItemCode"].astype(str)
long_sku_model_summary["ItemCode"] = long_sku_model_summary["ItemCode"].astype(str)

long_sku_model_summary = (
    long_sku_model_summary
    .merge(latest_behavior_df, on="ItemCode", how="left")
    .merge(behavior_champion_df, on="Behavior_Type", how="left")
)

long_sku_model_summary["Behavior_Type"] = long_sku_model_summary["Behavior_Type"].fillna("UNKNOWN")
long_sku_model_summary["Behavior_Best_Model"] = long_sku_model_summary["Behavior_Best_Model"].fillna("UNKNOWN")

long_sku_model_summary["Champion_Score_Adjusted"] = long_sku_model_summary["Champion_Score"]

long_sku_model_summary["Model_Unreliable"] = np.where(
    (
        (long_sku_model_summary["Holdout12_WMAPE"] > 70) &
        (long_sku_model_summary["Recent4B_WMAPE"] > 80)
    ) |
    (long_sku_model_summary["Recent4A_WMAPE"] > 100) |
    (long_sku_model_summary["Recent4B_WMAPE"] > 100) |
    (np.abs(long_sku_model_summary["Recent4B_Bias"]) > 35),
    1,
    0
)

gru_mask = long_sku_model_summary["Model_Name"] == "GRU"

long_sku_model_summary.loc[gru_mask & (long_sku_model_summary["Recent4B_WMAPE"] > 35), "Champion_Score_Adjusted"] += 3
long_sku_model_summary.loc[gru_mask & (long_sku_model_summary["Recent4A_WMAPE"] > 35), "Champion_Score_Adjusted"] += 2
long_sku_model_summary.loc[gru_mask & (np.abs(long_sku_model_summary["Recent4B_Bias"]) > 20), "Champion_Score_Adjusted"] += 2
long_sku_model_summary.loc[gru_mask & (np.abs(long_sku_model_summary["Recent4A_Bias"]) > 20), "Champion_Score_Adjusted"] += 1
long_sku_model_summary.loc[gru_mask & (long_sku_model_summary["Model_Unreliable"] == 1), "Champion_Score_Adjusted"] += 5

long_sku_model_summary.loc[
    long_sku_model_summary["Model_Name"] == long_sku_model_summary["Behavior_Best_Model"],
    "Champion_Score_Adjusted"
] -= 1.5

long_sku_model_summary.loc[
    (long_sku_model_summary["Model_Name"] != long_sku_model_summary["Behavior_Best_Model"]) &
    (long_sku_model_summary["Recent4B_WMAPE"] > 45),
    "Champion_Score_Adjusted"
] += 2

long_sku_model_summary["Model_Priority"] = (
    long_sku_model_summary["Model_Name"].map(MODEL_PRIORITY).fillna(999)
)


# ============================================================
# CHAMPION MAP
# ============================================================

sort_cols = [
    "Champion_Score_Adjusted",
    "Recent4B_WMAPE",
    "Recent4A_WMAPE",
    "Holdout12_WMAPE",
    "Holdout12_MAE",
    "Model_Priority"
]

BEHAVIOR_CLOSE_SCORE_TOLERANCE = 5.0
champion_rows = []

for item_code, g in long_sku_model_summary.groupby("ItemCode", sort=False):
    candidate_pool = g[g["Model_Unreliable"] == 0].copy()
    if candidate_pool.empty:
        candidate_pool = g.copy()

    sku_best_row = candidate_pool.sort_values(sort_cols).iloc[0].copy()

    behavior_best_model = str(sku_best_row.get("Behavior_Best_Model", "UNKNOWN"))
    behavior_candidate = candidate_pool[
        candidate_pool["Model_Name"].astype(str) == behavior_best_model
    ].copy()

    final_row = sku_best_row.copy()
    final_row["Behavior_Routing_Applied"] = 0

    if not behavior_candidate.empty:
        behavior_row = behavior_candidate.sort_values(sort_cols).iloc[0].copy()
        score_gap = behavior_row["Champion_Score_Adjusted"] - sku_best_row["Champion_Score_Adjusted"]

        if score_gap <= BEHAVIOR_CLOSE_SCORE_TOLERANCE:
            final_row = behavior_row.copy()
            final_row["Behavior_Routing_Applied"] = 1

    champion_rows.append(final_row)

champion_long_map_df = pd.DataFrame(champion_rows).reset_index(drop=True)

champion_long_map_df = champion_long_map_df.rename(columns={
    "Model_Name": "Best_Model",
    "Champion_Score": "Best_Model_Score",
    "Champion_Score_Adjusted": "Best_Model_Score_Adjusted",
    "Holdout12_WMAPE": "Best_Model_Holdout12_WMAPE",
    "Recent4A_WMAPE": "Best_Model_Recent4A_WMAPE",
    "Recent4B_WMAPE": "Best_Model_Recent4B_WMAPE",
    "Holdout12_MAE": "Best_Model_Holdout12_MAE",
    "Recent4A_MAE": "Best_Model_Recent4A_MAE",
    "Recent4B_MAE": "Best_Model_Recent4B_MAE",
    "Holdout12_Bias": "Best_Model_Holdout12_Bias",
    "Recent4A_Bias": "Best_Model_Recent4A_Bias",
    "Recent4B_Bias": "Best_Model_Recent4B_Bias",
    "Holdout12_Months": "Evaluation_Months_Holdout12",
    "Recent4A_Months": "Evaluation_Months_Recent4A",
    "Recent4B_Months": "Evaluation_Months_Recent4B"
})

champion_long_map_df["Segment"] = "LONG"

sku_behavior_profile = build_sku_behavior_profile(Data_long)
champion_long_map_df = force_itemcode_str(champion_long_map_df)
sku_behavior_profile = force_itemcode_str(sku_behavior_profile)

champion_long_map_df = champion_long_map_df.merge(
    sku_behavior_profile[["ItemCode", "Intermittent_Flag", "Highly_Volatile_Flag", "SKU_Zero_Rate", "SKU_CV"]],
    on="ItemCode",
    how="left"
)

for c in ["Intermittent_Flag", "Highly_Volatile_Flag", "SKU_Zero_Rate", "SKU_CV"]:
    champion_long_map_df[c] = champion_long_map_df[c].fillna(0)

champion_long_map_df = assign_final_routing(champion_long_map_df)

print("\nLONG CHAMPION MAP")
print(champion_long_map_df.head())


===== BEHAVIOR CHAMPION TABLE =====
   Behavior_Type Behavior_Best_Model  Behavior_Best_Model_Score  \
1   INTERMITTENT                 GRU                  43.947154   
3   PROMO_DRIVEN            CATBOOST                  25.210781   
7         STABLE                 GRU                  18.841655   
10      VOLATILE                 GRU                  37.092907   

    Behavior_Best_Model_Recent4B_WMAPE  
1                            38.981962  
3                            24.273735  
7                            20.326047  
10                           29.130225  

LONG CHAMPION MAP
  ItemCode Best_Model  Evaluation_Months_Holdout12  Holdout12_Actual_Sum  \
0   600308        GRU                           12               81492.0   
1   600310        GRU                           12                  67.0   
2   600311        GRU                           12               86595.0   
3   600315        GRU                           12               12547.0   
4   600319   CATBOOST  

In [25]:
long_model_win_counts = (
    champion_long_map_df["Final_Model"]
    .value_counts(dropna=False)
    .reset_index()
)
long_model_win_counts.columns = ["Final_Model", "SKU_Count"]

long_eval_with_champion_df = force_itemcode_str(long_model_a_eval_df).merge(
    force_itemcode_str(champion_long_map_df)[
        [
            "ItemCode", "Best_Model", "Best_Model_Score",
            "Final_Model", "Fallback_Type",
            "Use_Fallback", "Unreliable_Flag"
        ]
    ],
    on="ItemCode",
    how="left"
)

long_eval_with_champion_df["Is_Champion_Model"] = (
    long_eval_with_champion_df["Model_Name"] == long_eval_with_champion_df["Best_Model"]
).astype(int)

joblib.dump(champion_long_map_df, "champion_long_map_df.pkl")
joblib.dump(long_eval_with_champion_df, "long_eval_with_champion_df.pkl")

print("Saved: champion_long_map_df.pkl")
print("Saved: long_eval_with_champion_df.pkl")

Saved: champion_long_map_df.pkl
Saved: long_eval_with_champion_df.pkl


#### DEPLOYMENT

##### XGB + CAT

In [26]:
# ============================================================
# TREE MODEL DEPLOYMENT
# ============================================================

def train_single_deployment_model_long(model_name, full_data, feature_cols, best_params):
    print(f"\n========== {model_name} LONG DEPLOYMENT MODEL ==========")

    deploy_df, prep_artifacts = prepare_long_deploy_frame(full_data)

    assert_features_exist(deploy_df, feature_cols, where=f"{model_name}_LONG_DEPLOY_TRAIN")

    model = get_long_model(model_name, best_params)

    model.fit(
        sanitize(deploy_df[feature_cols]),
        deploy_df[MODEL_TARGET_COL],
        sample_weight=build_train_weights(deploy_df),
        verbose=False
    )

    artifacts = {
        "model": model,
        "segment": "LONG",
        "feature_cols": feature_cols,
        "best_params": best_params,
        "itemcode_categories": prep_artifacts["itemcode_categories"],
        "abc_map": prep_artifacts["abc_map"],
        "clip_caps": prep_artifacts["clip_caps"],
        "promo_profile_df": prep_artifacts.get("promo_profile_df", None),    
        "sku_profile_df": prep_artifacts.get("sku_profile_df", None),
        "target_mode": "residual",
        "baseline_col": BASELINE_COL,
        "actual_target_col": ACTUAL_TARGET_COL,
        "model_target_col": MODEL_TARGET_COL,
        "model_name": model_name
    }

    return artifacts, deploy_df


TREE_DEPLOY_CONFIG = {
    "XGBOOST": {
        "features": xgb_long_best_feats,
        "params": xgb_long_best_params,
        "file": "xgb_long_deploy_artifacts_residual.pkl"
    },
    "CATBOOST": {
        "features": catboost_long_best_feats,
        "params": catboost_long_best_params,
        "file": "catboost_long_deploy_artifacts_residual.pkl"
    }
}

long_deploy_train_dfs = {}

xgb_long_deploy_artifacts = None
catboost_long_deploy_artifacts = None

for model_name, cfg in TREE_DEPLOY_CONFIG.items():
    artifacts, deploy_df = train_single_deployment_model_long(
        model_name=model_name,
        full_data=Data,
        feature_cols=cfg["features"],
        best_params=cfg["params"]
    )

    if model_name == "XGBOOST":
        xgb_long_deploy_artifacts = artifacts
        xgb_long_deploy_train_df = deploy_df

    elif model_name == "CATBOOST":
        catboost_long_deploy_artifacts = artifacts
        catboost_long_deploy_train_df = deploy_df

    joblib.dump(artifacts, cfg["file"])
    long_deploy_train_dfs[model_name] = deploy_df

    print(f"Saved: {cfg['file']}")


========== XGBOOST LONG DEPLOYMENT MODEL ==========
Saved: xgb_long_deploy_artifacts_residual.pkl

========== CATBOOST LONG DEPLOYMENT MODEL ==========
Saved: catboost_long_deploy_artifacts_residual.pkl


##### GRU

In [27]:
# 16) LONG GRU DEPLOYMENT TRAINING
def train_long_gru_deployment_model(full_data):
    print("\n========== LONG GRU DEPLOYMENT TRAINING ==========")

    deploy_df, deploy_meta = prepare_long_gru_deploy_data(full_data)

    if deploy_df.empty:
        raise ValueError("No LONG rows available for GRU deployment.")

    deploy_df = deploy_df.dropna(subset=[GRU_LONG_TARGET_COL, GRU_LONG_RESIDUAL_COL]).copy()

    item_to_idx = make_item_mapping_from_train(deploy_df)
    scalers = fit_long_gru_scalers(deploy_df)

    key_col = "ItemCode_Original" if "ItemCode_Original" in deploy_df.columns else "ItemCode"
    deploy_df[key_col] = deploy_df[key_col].astype(str)
    deploy_df = deploy_df[deploy_df[key_col].isin(item_to_idx.keys())].copy()

    X_seq, X_static, X_item, y, w, meta = build_long_gru_sequences(
        deploy_df,
        item_to_idx,
        scalers,
        seq_len=GRU_SEQ_LEN
    )

    if len(y) == 0:
        raise ValueError("No LONG GRU deployment sequences created.")

    dataset = LongGRUSequenceDataset(X_seq, X_static, X_item, y, w)
    loader = DataLoader(dataset, batch_size=GRU_BATCH_SIZE, shuffle=True)

    print("GRU_DEVICE:", GRU_DEVICE)
    print("Deployment sequences:", len(y))
    print("Deployment batches:", len(loader))

    model = LongGRUResidualForecaster(
        num_items=len(item_to_idx),
        seq_input_dim=len(GRU_LONG_SEQ_FEATURES),
        static_input_dim=len(GRU_LONG_STATIC_FEATURES),
        embed_dim=GRU_EMBED_DIM,
        hidden_size=GRU_HIDDEN_SIZE,
        num_layers=GRU_NUM_LAYERS,
        dropout=GRU_DROPOUT,
    ).to(GRU_DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=GRU_LR, weight_decay=GRU_WEIGHT_DECAY)
    criterion = WeightedAsymmetricMAELoss(under_penalty=GRU_UNDER_PENALTY)

    best_loss = float("inf")
    best_state = None

    for epoch in range(1, GRU_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        batch_count = 0

        print(f"Starting deploy epoch {epoch}/{GRU_EPOCHS}...")

        for x_seq, x_static, x_item, y_batch, w_batch in loader:
            batch_count += 1

            x_seq = x_seq.to(GRU_DEVICE)
            x_static = x_static.to(GRU_DEVICE)
            x_item = x_item.to(GRU_DEVICE)
            y_batch = y_batch.to(GRU_DEVICE)
            w_batch = w_batch.to(GRU_DEVICE)

            optimizer.zero_grad()
            preds = model(x_seq, x_static, x_item)
            loss = criterion(preds, y_batch, w_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item() * len(y_batch)

            if batch_count % 20 == 0:
                print(f"  deploy epoch {epoch} batch {batch_count}/{len(loader)} loss={loss.item():.5f}")

        epoch_loss = total_loss / len(loader.dataset)
        print(f"Epoch {epoch:02d} | Train Loss: {epoch_loss:.5f}")

        if epoch_loss < best_loss:
            best_loss = epoch_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    artifacts = {
        "model": model,
        "model_type": "GRU",
        "model_name": "GRU",
        "segment": "LONG",
        "seq_features": GRU_LONG_SEQ_FEATURES,
        "static_features": GRU_LONG_STATIC_FEATURES,
        "seq_len": GRU_SEQ_LEN,
        "embed_dim": GRU_EMBED_DIM,
        "hidden_size": GRU_HIDDEN_SIZE,
        "num_layers": GRU_NUM_LAYERS,
        "dropout": GRU_DROPOUT,
        "item_to_idx": item_to_idx,
        "abc_map": deploy_meta["abc_map"],
        "promo_profile_df": deploy_meta.get("promo_profile_df", None),
        "sku_profile_df": deploy_meta.get("sku_profile_df", None),
        "clip_caps": deploy_meta["clip_caps"],
        "gru_log_clip_value": deploy_meta["gru_log_clip_value"],
        "itemcode_categories": deploy_meta["itemcode_categories"]
    }

    return artifacts, scalers, deploy_df

# 17) SAVE LONG GRU DEPLOY ARTIFACTS
def save_long_gru_deploy_artifacts(artifacts, scalers):
    os.makedirs(GRU_DEPLOY_DIR, exist_ok=True)

    torch.save(
        {
            "model_state_dict": artifacts["model"].state_dict(),
            "model_type": artifacts["model_type"],
            "segment": artifacts["segment"],
            "seq_features": artifacts["seq_features"],
            "static_features": artifacts["static_features"],
            "seq_len": artifacts["seq_len"],
            "embed_dim": artifacts["embed_dim"],
            "hidden_size": artifacts["hidden_size"],
            "num_layers": artifacts["num_layers"],
            "dropout": artifacts["dropout"],
            "item_to_idx": artifacts["item_to_idx"],
            "abc_map": artifacts["abc_map"],
            "gru_log_clip_value": artifacts["gru_log_clip_value"],
            "clip_caps": artifacts["clip_caps"]
        },
        os.path.join(GRU_DEPLOY_DIR, "gru_long_deploy_model.pt")
    )

    joblib.dump(scalers.seq_scaler, os.path.join(GRU_DEPLOY_DIR, "gru_long_seq_scaler.pkl"))
    joblib.dump(scalers.static_scaler, os.path.join(GRU_DEPLOY_DIR, "gru_long_static_scaler.pkl"))
    if artifacts.get("promo_profile_df") is not None:
        joblib.dump(artifacts["promo_profile_df"], os.path.join(GRU_DEPLOY_DIR, "gru_long_promo_profile_df.pkl"))
    if artifacts.get("sku_profile_df") is not None:
        joblib.dump(artifacts["sku_profile_df"], os.path.join(GRU_DEPLOY_DIR, "gru_long_sku_profile_df.pkl"))
    joblib.dump(artifacts["item_to_idx"], os.path.join(GRU_DEPLOY_DIR, "gru_long_item_to_idx.pkl"))
    joblib.dump(artifacts["itemcode_categories"], os.path.join(GRU_DEPLOY_DIR, "gru_long_itemcode_categories.pkl"))

    deploy_meta = {
        "model_type": artifacts["model_type"],
        "model_name": artifacts["model_name"],
        "segment": artifacts["segment"],
        "seq_features": artifacts["seq_features"],
        "static_features": artifacts["static_features"],
        "seq_len": artifacts["seq_len"],
        "embed_dim": artifacts["embed_dim"],
        "hidden_size": artifacts["hidden_size"],
        "num_layers": artifacts["num_layers"],
        "dropout": artifacts["dropout"],
        "gru_log_clip_value": artifacts["gru_log_clip_value"],
    }

    with open(os.path.join(GRU_DEPLOY_DIR, "gru_long_deploy_meta.json"), "w") as f:
        json.dump(deploy_meta, f, indent=2)

    print("\nSaved LONG GRU deploy artifacts:")
    print(f" - {GRU_DEPLOY_DIR}/gru_long_deploy_model.pt")
    print(f" - {GRU_DEPLOY_DIR}/gru_long_seq_scaler.pkl")
    print(f" - {GRU_DEPLOY_DIR}/gru_long_static_scaler.pkl")
    print(f" - {GRU_DEPLOY_DIR}/gru_long_promo_profile_df.pkl")
    print(f" - {GRU_DEPLOY_DIR}/gru_long_deploy_meta.json")
    print(f" - {GRU_DEPLOY_DIR}/gru_long_item_to_idx.pkl")
    print(f" - {GRU_DEPLOY_DIR}/gru_long_itemcode_categories.pkl")


In [28]:
# ============================================================
# 22) RUN LONG GRU DEPLOY
# ============================================================
gru_seed_everything(GRU_SEED)

gru_long_deploy_artifacts, gru_long_deploy_scalers, gru_long_deploy_df = train_long_gru_deployment_model(
    full_data=Data
)

save_long_gru_deploy_artifacts(
    artifacts=gru_long_deploy_artifacts,
    scalers=gru_long_deploy_scalers
)

print("\nLONG GRU deployment rows:", len(gru_long_deploy_df))


========== LONG GRU DEPLOYMENT TRAINING ==========
GRU_DEVICE: mps
Deployment sequences: 22816
Deployment batches: 90
Starting deploy epoch 1/10...
  deploy epoch 1 batch 20/90 loss=11.23083
  deploy epoch 1 batch 40/90 loss=11.03921
  deploy epoch 1 batch 60/90 loss=10.41212
  deploy epoch 1 batch 80/90 loss=10.97281
Epoch 01 | Train Loss: 10.56745
Starting deploy epoch 2/10...
  deploy epoch 2 batch 20/90 loss=9.12588
  deploy epoch 2 batch 40/90 loss=9.20952
  deploy epoch 2 batch 60/90 loss=10.29004
  deploy epoch 2 batch 80/90 loss=9.89445
Epoch 02 | Train Loss: 9.89815
Starting deploy epoch 3/10...
  deploy epoch 3 batch 20/90 loss=8.82516
  deploy epoch 3 batch 40/90 loss=9.98407
  deploy epoch 3 batch 60/90 loss=9.34249
  deploy epoch 3 batch 80/90 loss=10.40801
Epoch 03 | Train Loss: 9.22438
Starting deploy epoch 4/10...
  deploy epoch 4 batch 20/90 loss=8.91000
  deploy epoch 4 batch 40/90 loss=8.06728
  deploy epoch 4 batch 60/90 loss=8.64990
  deploy epoch 4 batch 80/90 lo

##### Registry

In [29]:
print("\n===== WINDOW SANITY CHECK =====")
print("LONG_TUNE_WINDOWS:", LONG_TUNE_WINDOWS)
print("LONG_VALID_WINDOW:", LONG_VALID_WINDOW)
print("LONG_HOLDOUT12_WINDOW:", LONG_HOLDOUT12_WINDOW)
print("LONG_RECENT4_WINDOW:", LONG_RECENT4_WINDOW)
print("LONG_MODEL_A_WINDOW:", LONG_MODEL_A_WINDOW)

assert LONG_HOLDOUT12_WINDOW["valid_end_idx"] < LONG_RECENT4_WINDOW["valid_start_idx"], "Holdout12 overlaps Recent4"
assert LONG_MODEL_A_WINDOW["valid_start_idx"] == LONG_HOLDOUT12_WINDOW["valid_start_idx"]
assert LONG_MODEL_A_WINDOW["valid_end_idx"] == LONG_RECENT4_WINDOW["valid_end_idx"]

print("Window logic OK")

print("\n===== COVERAGE CHECK =====")
print(long_sku_model_summary.groupby("Model_Name").agg(
    SKU_Count=("ItemCode", "nunique"),
    Avg_H12_Months=("Holdout12_Months", "mean"),
    Avg_R4A_Months=("Recent4A_Months", "mean"),
    Avg_R4B_Months=("Recent4B_Months", "mean")
))


===== WINDOW SANITY CHECK =====
LONG_TUNE_WINDOWS: [{'valid_start_idx': 24277, 'valid_end_idx': 24288}, {'valid_start_idx': 24289, 'valid_end_idx': 24300}]
LONG_VALID_WINDOW: {'valid_start_idx': 24289, 'valid_end_idx': 24300, 'valid_months': 12}
LONG_HOLDOUT12_WINDOW: {'valid_start_idx': 24296, 'valid_end_idx': 24307, 'holdout_months': 12}
LONG_RECENT4_WINDOW: {'valid_start_idx': 24308, 'valid_end_idx': 24311, 'recent_months': 4}
LONG_MODEL_A_WINDOW: {'valid_start_idx': 24296, 'valid_end_idx': 24311, 'holdout_months': 12, 'recent_months': 4}
Window logic OK

===== COVERAGE CHECK =====
            SKU_Count  Avg_H12_Months  Avg_R4A_Months  Avg_R4B_Months
Model_Name                                                           
CATBOOST          602       12.000000             4.0             4.0
GRU               586       11.854949             4.0             4.0
XGBOOST           602       12.000000             4.0             4.0


In [30]:
long_deployment_artifact_registry = pd.DataFrame([
    {"Model_Name": "XGBOOST",  "Deploy_File": "xgb_long_deploy_artifacts_residual.pkl"},
    {"Model_Name": "CATBOOST", "Deploy_File": "catboost_long_deploy_artifacts_residual.pkl"},
    {"Model_Name": "GRU",      "Deploy_File": f"{GRU_DEPLOY_DIR}/gru_long_deploy_meta.json"}
])

print("\nLONG DEPLOYMENT ARTIFACT REGISTRY")
print(long_deployment_artifact_registry)

joblib.dump(long_deployment_artifact_registry, "long_deployment_artifact_registry.pkl")
print("Saved: long_deployment_artifact_registry.pkl")


LONG DEPLOYMENT ARTIFACT REGISTRY
  Model_Name                                        Deploy_File
0    XGBOOST             xgb_long_deploy_artifacts_residual.pkl
1   CATBOOST        catboost_long_deploy_artifacts_residual.pkl
2        GRU  gru_long_deploy_artifacts/gru_long_deploy_meta...
Saved: long_deployment_artifact_registry.pkl


### MEDIUM SEGMENT

In [31]:
# ============================================================
# 1) MEDIUM WINDOWS
# ============================================================
def get_medium_recent4_window(time_windows, recent_months=4):
    valid_end_idx = int(time_windows["latest_idx"] - 1)
    valid_start_idx = int(valid_end_idx - recent_months + 1)
    return {
        "valid_start_idx": valid_start_idx,
        "valid_end_idx": valid_end_idx,
        "valid_months": int(recent_months),
        "window_name": "RECENT4"
    }

def get_medium_prev4_window(time_windows, prev_months=4, recent_months=4):
    recent_window = get_medium_recent4_window(time_windows, recent_months=recent_months)
    valid_end_idx = int(recent_window["valid_start_idx"] - 1)
    valid_start_idx = int(valid_end_idx - prev_months + 1)

    if valid_start_idx > valid_end_idx:
        raise ValueError("Invalid MEDIUM PREV4 window.")

    return {
        "valid_start_idx": valid_start_idx,
        "valid_end_idx": valid_end_idx,
        "valid_months": int(prev_months),
        "window_name": "PREV4"
    }

def get_medium_tune_windows(df, time_windows, n_folds=2, fold_size_months=4, recent_months=4):
    temp = add_period_index(df)
    periods = (
        temp[["Year", "Month_Number", "Period_Index"]]
        .drop_duplicates()
        .sort_values("Period_Index")
        .reset_index(drop=True)
    )

    recent_window = get_medium_recent4_window(time_windows, recent_months=recent_months)
    usable = periods[periods["Period_Index"] < recent_window["valid_start_idx"]].copy()

    if len(usable) < fold_size_months * (n_folds + 1):
        raise ValueError("Not enough history to build MEDIUM tune windows.")

    windows = []
    end_idx = int(usable["Period_Index"].max())

    for _ in range(n_folds):
        valid_end_idx = end_idx
        valid_start_idx = valid_end_idx - fold_size_months + 1
        windows.append({
            "valid_start_idx": int(valid_start_idx),
            "valid_end_idx": int(valid_end_idx),
            "valid_months": int(fold_size_months)
        })
        end_idx = valid_start_idx - 1

    return list(reversed(windows))


In [32]:
MEDIUM_SUBGROUP_FEATURE_COLS = list(dict.fromkeys(
    MEDIUM_FEATURES + [
        "Medium_SKU_Type_Encoded",
        "Medium_Bonus_Frequency",
        "Medium_Bonus_Demand_Share",
        "Medium_CV",
        "Medium_ZeroRate",
        "Medium_Supply_Rate",
        "Medium_Trend_Slope",
    ]
))
# Remove if not existing
MEDIUM_SUBGROUP_FEATURE_COLS = [
    c for c in MEDIUM_SUBGROUP_FEATURE_COLS
    if c in Model_Data_All.columns or c.startswith("Medium_")
]

MEDIUM_TUNE_WINDOWS = get_medium_tune_windows(Data, TIME_WINDOWS, n_folds=2, fold_size_months=4, recent_months=4)
MEDIUM_PREV4_WINDOW = get_medium_prev4_window(TIME_WINDOWS, prev_months=4, recent_months=4)
MEDIUM_RECENT4_WINDOW = get_medium_recent4_window(TIME_WINDOWS, recent_months=4)

print("MEDIUM_TUNE_WINDOWS:", MEDIUM_TUNE_WINDOWS)
print("MEDIUM_PREV4_WINDOW:", MEDIUM_PREV4_WINDOW)
print("MEDIUM_RECENT4_WINDOW:", MEDIUM_RECENT4_WINDOW)

# Optional alias if you still want the word VALID in code
MEDIUM_VALID_WINDOW = MEDIUM_PREV4_WINDOW

MEDIUM_TUNE_WINDOWS: [{'valid_start_idx': 24300, 'valid_end_idx': 24303, 'valid_months': 4}, {'valid_start_idx': 24304, 'valid_end_idx': 24307, 'valid_months': 4}]
MEDIUM_PREV4_WINDOW: {'valid_start_idx': 24304, 'valid_end_idx': 24307, 'valid_months': 4, 'window_name': 'PREV4'}
MEDIUM_RECENT4_WINDOW: {'valid_start_idx': 24308, 'valid_end_idx': 24311, 'valid_months': 4, 'window_name': 'RECENT4'}


In [33]:
# ============================================================
# 2) MEDIUM PROFILE BUILDING
# ============================================================
def build_medium_sku_profile(train_df):
    train_df = force_itemcode_str(train_df)
    train_df = train_df.copy().sort_values(["ItemCode", "Year", "Month_Number"])
    out = []

    for item, g in train_df.groupby("ItemCode"):
        g = g.copy()

        mean_demand = float(g["Clean_Demand"].mean()) if len(g) > 0 else 0.0
        std_demand = float(g["Clean_Demand"].std()) if len(g) > 1 else 0.0
        cv = 0.0 if mean_demand <= 0 else std_demand / (mean_demand + 1)

        zero_rate = float((g["Clean_Demand"] == 0).mean()) if len(g) > 0 else 0.0
        bonus_freq = float(g["Bonus_Flag"].mean()) if "Bonus_Flag" in g.columns else 0.0
        supply_rate = float(g["Supply_Constraint_Flag"].mean()) if "Supply_Constraint_Flag" in g.columns else 0.0

        total_demand = float(g["Clean_Demand"].sum())
        bonus_demand = float(g.loc[g["Bonus_Flag"] == 1, "Clean_Demand"].sum()) if total_demand > 0 else 0.0
        bonus_share = 0.0 if total_demand <= 0 else bonus_demand / total_demand

        if len(g) >= 2:
            x = np.arange(len(g))
            y = g["Clean_Demand"].values
            trend_slope = float(np.polyfit(x, y, 1)[0])
        else:
            trend_slope = 0.0

        if zero_rate > 0.5:
            sku_type = "INTERMITTENT"
        elif bonus_freq > 0.3 or bonus_share > 0.4:
            sku_type = "PROMO_HEAVY"
        elif supply_rate > 0.3:
            sku_type = "SUPPLY_AFFECTED"
        elif abs(trend_slope) > mean_demand * 0.2:
            sku_type = "TRENDING"
        else:
            sku_type = "STABLE"

        if bonus_freq > 0.25 or bonus_share > 0.35:
            subgroup = "PROMO_HEAVY"
        else:
            subgroup = "STABLE"

        out.append({
            "ItemCode": item,
            "Medium_SKU_Type": sku_type,
            "Medium_Subgroup": subgroup,
            "Medium_Bonus_Frequency": bonus_freq,
            "Medium_Bonus_Demand_Share": bonus_share,
            "Medium_CV": cv,
            "Medium_ZeroRate": zero_rate,
            "Medium_Supply_Rate": supply_rate,
            "Medium_Trend_Slope": trend_slope
        })

    out_df = pd.DataFrame(out)
    out_df = force_itemcode_str(out_df)
    return out_df

def merge_medium_sku_profile(df, profile_df):
    df = force_itemcode_str(df)
    profile_df = force_itemcode_str(profile_df)
    df = df.copy()

    keep_cols = [
        "ItemCode",
        "Medium_SKU_Type",
        "Medium_Subgroup",
        "Medium_Bonus_Frequency",
        "Medium_Bonus_Demand_Share",
        "Medium_CV",
        "Medium_ZeroRate",
        "Medium_Supply_Rate",
        "Medium_Trend_Slope"
    ]

    df = df.drop(columns=[c for c in keep_cols if c != "ItemCode"], errors="ignore")
    df = df.merge(profile_df[keep_cols], on="ItemCode", how="left")
    df = force_itemcode_str(df)

    df["Medium_SKU_Type"] = df["Medium_SKU_Type"].fillna("STABLE")
    df["Medium_Subgroup"] = df["Medium_Subgroup"].fillna("STABLE")

    for c in [
        "Medium_Bonus_Frequency",
        "Medium_Bonus_Demand_Share",
        "Medium_CV",
        "Medium_ZeroRate",
        "Medium_Supply_Rate",
        "Medium_Trend_Slope"
    ]:
        df[c] = df[c].fillna(0)

    type_map = {
        "STABLE": 0,
        "PROMO_HEAVY": 1,
        "TRENDING": 2,
        "SUPPLY_AFFECTED": 3,
        "INTERMITTENT": 4
    }
    df["Medium_SKU_Type_Encoded"] = df["Medium_SKU_Type"].map(type_map).fillna(0).astype(int)

    return df

def enforce_unique_subgroup(profile_df):
    profile_df = profile_df.copy()
    
    dup = profile_df.groupby("ItemCode")["Medium_Subgroup"].nunique()
    bad_skus = dup[dup > 1].index.tolist()
    
    if len(bad_skus) > 0:
        print("⚠️ FIXING DUPLICATE SUBGROUP SKUs:", len(bad_skus))
        profile_df = (
            profile_df
            .sort_values(["ItemCode", "Medium_Bonus_Demand_Share"], ascending=False)
            .drop_duplicates("ItemCode")
        )
    
    return profile_df

def get_current_medium_skus(df):
    df = add_history_length_from_subset(df.copy(), df.copy())
    df = df[df["History_Segment"] == "MEDIUM"]
    
    return set(
        df["ItemCode"]
        .astype(str)
        .str.replace(".0", "", regex=False)
        .unique()
    )

def split_low_history_medium_skus(df, min_months=10):
    df = force_itemcode_str(df).copy()

    hist_counts = (
        df[["ItemCode", "Year", "Month_Number"]]
        .drop_duplicates()
        .groupby("ItemCode", as_index=False)
        .size()
        .rename(columns={"size": "History_Months"})
    )

    hist_counts["ItemCode"] = hist_counts["ItemCode"].astype(str)

    good_skus = set(
        hist_counts.loc[hist_counts["History_Months"] >= min_months, "ItemCode"]
    )
    low_skus = set(
        hist_counts.loc[hist_counts["History_Months"] < min_months, "ItemCode"]
    )

    df_good = df[df["ItemCode"].astype(str).isin(good_skus)].copy()
    df_low = df[df["ItemCode"].astype(str).isin(low_skus)].copy()

    return df_good, df_low, good_skus, low_skus

# ============================================================
# 3) FOLD-SAFE MEDIUM PREP
# ============================================================
def prepare_medium_subgroup_frame_foldsafe(train_df, valid_df):
    train_df = force_itemcode_str(train_df.copy())
    valid_df = force_itemcode_str(valid_df.copy())

    train_df = train_df[train_df["History_Segment"] == "MEDIUM"].copy()
    valid_df = valid_df[valid_df["History_Segment"] == "MEDIUM"].copy()

    if train_df.empty or valid_df.empty:
        return None, None, None

    train_df, valid_df, abc_map = apply_fold_adjustments(train_df, valid_df)

    sku_profile_df = build_sku_history_profile(train_df)

    train_df = merge_sku_history_profile(train_df, sku_profile_df)
    valid_df = merge_sku_history_profile(valid_df, sku_profile_df)

    medium_profile_df = build_medium_sku_profile(train_df)
    medium_profile_df = enforce_unique_subgroup(medium_profile_df)

    train_df = merge_medium_sku_profile(train_df, medium_profile_df)
    valid_df = merge_medium_sku_profile(valid_df, medium_profile_df)

    clip_cols = [
        "Rolling3M_Std",
        "Momentum",
        "Stock_Cover_Months",
        "Primary_Stock_Cover",
        "Distributor_Stock_Cover",
        "Free_Ratio",
        "SKU_CV",
        "Medium_CV",
        "Medium_Trend_Slope"
    ]

    caps = compute_clip_caps(train_df, clip_cols, q=0.99)
    train_df = apply_clip_caps(train_df, caps)
    valid_df = apply_clip_caps(valid_df, caps)

    if train_df.empty or valid_df.empty:
        return None, None, None

    artifacts = {
        "abc_map": abc_map,
        "clip_caps": caps,
        "medium_profile_df": medium_profile_df,
        "sku_profile_df": sku_profile_df
    }

    return train_df, valid_df, artifacts

# ============================================================
# 4) SUBGROUP FILTER
# ============================================================
def filter_medium_subgroup(df, subgroup_name):
    df = force_itemcode_str(df)
    subgroup_name = str(subgroup_name).upper()

    if subgroup_name == "PROMO_HEAVY":
        return df[df["Medium_Subgroup"] == "PROMO_HEAVY"].copy()

    if subgroup_name == "STABLE":
        return df[df["Medium_Subgroup"] == "STABLE"].copy()

    raise ValueError(f"Unknown subgroup_name: {subgroup_name}")

# ============================================================
# 5) MODEL BUILDERS / PARAMS
# ============================================================
def build_medium_model(model_name, params):
    model_name = model_name.upper()

    if model_name == "XGBOOST":
        return xgb.XGBRegressor(
            objective="reg:squarederror",
            eval_metric="rmse",
            random_state=42,
            tree_method="hist",
            n_jobs=-1,
            **params
        )

    elif model_name == "RANDOM_FOREST":
        return RandomForestRegressor(
            random_state=42,
            n_jobs=-1,
            **params
        )

    elif model_name == "CATBOOST":
        return CatBoostRegressor(
            loss_function="RMSE",
            eval_metric="RMSE",
            random_seed=42,
            verbose=0,
            **params
        )

    else:
        raise ValueError(f"Unsupported model_name: {model_name}")

def get_medium_model_params(trial, model_name, subgroup_name):
    model_name = model_name.upper()
    subgroup_name = subgroup_name.upper()

    if model_name == "XGBOOST":
        if subgroup_name == "PROMO_HEAVY":
            return {
                "n_estimators": trial.suggest_int("n_estimators", 600, 1400),
                "learning_rate": trial.suggest_float("learning_rate", 0.012, 0.035),
                "max_depth": trial.suggest_int("max_depth", 5, 7),
                "max_leaves": 64,
                "grow_policy": "lossguide",
                "min_child_weight": trial.suggest_int("min_child_weight", 3, 8),
                "subsample": trial.suggest_float("subsample", 0.70, 0.88),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.70, 0.88),
                "gamma": trial.suggest_float("gamma", 0.10, 0.45),
                "reg_lambda": trial.suggest_float("reg_lambda", 12, 22),
                "reg_alpha": trial.suggest_float("reg_alpha", 1.0, 6.0),
            }
        else:
            return {
                "n_estimators": trial.suggest_int("n_estimators", 400, 1000),
                "learning_rate": trial.suggest_float("learning_rate", 0.012, 0.030),
                "max_depth": trial.suggest_int("max_depth", 3, 5),
                "max_leaves": 64,
                "grow_policy": "lossguide",
                "min_child_weight": trial.suggest_int("min_child_weight", 5, 12),
                "subsample": trial.suggest_float("subsample", 0.72, 0.90),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 0.85),
                "gamma": trial.suggest_float("gamma", 0.15, 0.55),
                "reg_lambda": trial.suggest_float("reg_lambda", 12, 24),
                "reg_alpha": trial.suggest_float("reg_alpha", 1.0, 6.0),
            }

    elif model_name == "CATBOOST":
        if subgroup_name == "PROMO_HEAVY":
            return {
                "iterations": trial.suggest_int("iterations", 500, 1300),
                "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.045),
                "depth": trial.suggest_int("depth", 5, 8),
                "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 2.0, 15.0),
                "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 5, 35),
                "subsample": trial.suggest_float("subsample", 0.70, 0.95),
                "rsm": trial.suggest_float("rsm", 0.70, 1.0),
                "random_strength": trial.suggest_float("random_strength", 0.0, 3.0),
                "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 3.0),
            }
        else:
            return {
                "iterations": trial.suggest_int("iterations", 300, 900),
                "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.040),
                "depth": trial.suggest_int("depth", 4, 6),
                "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 2.0, 12.0),
                "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 40),
                "subsample": trial.suggest_float("subsample", 0.75, 0.95),
                "rsm": trial.suggest_float("rsm", 0.70, 1.0),
                "random_strength": trial.suggest_float("random_strength", 0.0, 2.5),
                "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 2.5),
            }

    elif model_name == "RANDOM_FOREST":
        if subgroup_name != "STABLE":
            raise ValueError("RANDOM_FOREST should be used only for STABLE subgroup.")

        return {
            "n_estimators": trial.suggest_int("n_estimators", 300, 900),
            "max_depth": trial.suggest_int("max_depth", 4, 14),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 12),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 8),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.6, 0.8]),
            "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
        }

    else:
        raise ValueError(f"Unsupported model_name: {model_name}")

def maybe_scale_features(X_train, X_valid=None, model_name="XGBOOST"):
    if model_name.upper() != "ELASTICNET":
        return X_train, X_valid, None

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    if X_valid is None:
        return X_train_scaled, None, scaler

    X_valid_scaled = scaler.transform(X_valid)
    return X_train_scaled, X_valid_scaled, scaler

# ============================================================
# 7) TUNING / TRAIN-EVAL
# ============================================================
def tune_residual_medium_subgroup(
    full_data,
    feature_cols,
    subgroup_name,
    model_name,
    n_trials=30,
    study_name=None
):
    full_data = full_data.copy()
    model_name = model_name.upper()
    subgroup_name = subgroup_name.upper()

    if study_name is None:
        study_name = f"residual_{model_name.lower()}_medium_{subgroup_name.lower()}"

    def objective(trial):
        params = get_medium_model_params(trial, model_name, subgroup_name)
        scores = []

        temp = add_period_index(full_data)

        for tune_window in MEDIUM_TUNE_WINDOWS:
            train_df = temp[temp["Period_Index"] < tune_window["valid_start_idx"]].copy()
            valid_df = temp[temp["Period_Index"].between(tune_window["valid_start_idx"], tune_window["valid_end_idx"])].copy()

            if train_df.empty or valid_df.empty:
                continue

            train_df = add_history_length_from_subset(train_df, train_df)
            valid_df = add_history_length_from_subset(train_df, valid_df)

            train_df = train_df[train_df["History_Segment"] == "MEDIUM"].copy()
            valid_df = valid_df[valid_df["History_Segment"] == "MEDIUM"].copy()

            train_df, _, good_skus, low_skus = split_low_history_medium_skus(train_df, min_months=10)
            valid_df = valid_df[valid_df["ItemCode"].astype(str).isin(good_skus)].copy()

            if train_df.empty or valid_df.empty:
                continue

            train_df, valid_df, prep_artifacts = prepare_medium_subgroup_frame_foldsafe(train_df, valid_df)
            if train_df is None or valid_df is None:
                    continue 

            train_df = recompute_target(train_df)
            train_df = add_residual_target(train_df)

            train_df = train_df.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()
            valid_df = valid_df.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()

            train_df = filter_medium_subgroup(train_df, subgroup_name)
            valid_df = filter_medium_subgroup(valid_df, subgroup_name)

            if train_df.empty or valid_df.empty:
                continue

            train_df = train_df.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()
            valid_df = valid_df.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()

            if train_df.empty or valid_df.empty:
                continue

            train_df["ItemCode_Original"] = train_df["ItemCode"]
            valid_df["ItemCode_Original"] = valid_df["ItemCode"]

            train_df, valid_df, _ = encode_itemcode(train_df, valid_df)

            assert_features_exist(train_df, feature_cols, where=f"{model_name}_{subgroup_name}_TUNE_TRAIN")
            assert_features_exist(valid_df, feature_cols, where=f"{model_name}_{subgroup_name}_TUNE_VALID")

            model = build_medium_model(model_name, params)

            Xtr = sanitize(train_df[feature_cols])
            ytr = train_df[MODEL_TARGET_COL]
            Xva = sanitize(valid_df[feature_cols])

            Xtr, Xva, _ = maybe_scale_features(Xtr, Xva, model_name=model_name)

            w_train = build_train_weights(train_df, yearly_boost=0.25)

            try:
                model.fit(Xtr, ytr, sample_weight=w_train)
            except TypeError:
                model.fit(Xtr, ytr)

            pred_residual = model.predict(Xva)
            pred_final = np.clip(valid_df[BASELINE_COL].values + pred_residual, 0, None)

            scores.append(wmape(valid_df[ACTUAL_TARGET_COL].values, pred_final))

        return 999999.0 if len(scores) == 0 else np.mean(scores)

    study = optuna.create_study(direction="minimize", study_name=study_name)
    study.optimize(objective, n_trials=n_trials)

    return study.best_params, study

def train_eval_validation_residual_medium_subgroup(
    full_data,
    feature_cols,
    best_params,
    subgroup_name,
    model_name,
    valid_window,
    yearly_boost=0.25
):
    model_name = model_name.upper()
    subgroup_name = subgroup_name.upper()

    full_data = add_period_index(full_data)

    train_df = full_data[full_data["Period_Index"] < valid_window["valid_start_idx"]].copy()
    valid_df = full_data[full_data["Period_Index"].between(valid_window["valid_start_idx"], valid_window["valid_end_idx"])].copy()

    if train_df.empty or valid_df.empty:
        raise ValueError(
            f"Need both train data before {valid_window['valid_start_idx']} "
            f"and validation data from {valid_window['valid_start_idx']} to {valid_window['valid_end_idx']}."
        )

    train_df = add_history_length_from_subset(train_df, train_df)
    valid_df = add_history_length_from_subset(train_df, valid_df)

    train_df = train_df[train_df["History_Segment"] == "MEDIUM"].copy()
    valid_df = valid_df[valid_df["History_Segment"] == "MEDIUM"].copy()

    train_df, _, good_skus, low_skus = split_low_history_medium_skus(train_df, min_months=10)
    valid_df = valid_df[valid_df["ItemCode"].astype(str).isin(good_skus)].copy()

    if train_df.empty or valid_df.empty:
        raise ValueError("No MEDIUM rows available.")

    train_df, valid_df, prep_artifacts = prepare_medium_subgroup_frame_foldsafe(train_df, valid_df)

    if train_df is None or valid_df is None:
        raise ValueError(f"No usable MEDIUM rows for subgroup {subgroup_name}.")
    
    train_df = recompute_target(train_df)
    train_df = add_residual_target(train_df)
    
    train_df = train_df.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()
    valid_df = valid_df.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()

    train_df = filter_medium_subgroup(train_df, subgroup_name)
    valid_df = filter_medium_subgroup(valid_df, subgroup_name)

    if train_df.empty or valid_df.empty:
        raise ValueError(f"No usable rows for subgroup {subgroup_name}.")

    train_df = train_df.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()
    valid_df = valid_df.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()

    if train_df.empty or valid_df.empty:
        raise ValueError(f"No usable rows after target creation for subgroup {subgroup_name}.")

    train_df["ItemCode_Original"] = train_df["ItemCode"]
    valid_df["ItemCode_Original"] = valid_df["ItemCode"]

    train_df, valid_df, itemcode_categories = encode_itemcode(train_df, valid_df)

    assert_features_exist(train_df, feature_cols, where=f"{model_name}_{subgroup_name}_VALID_TRAIN")
    assert_features_exist(valid_df, feature_cols, where=f"{model_name}_{subgroup_name}_VALID_EVAL")

    model = build_medium_model(model_name, best_params)

    Xtr = sanitize(train_df[feature_cols])
    ytr = train_df[MODEL_TARGET_COL]
    Xva = sanitize(valid_df[feature_cols])

    Xtr, Xva, scaler = maybe_scale_features(Xtr, Xva, model_name=model_name)

    w_train = build_train_weights(train_df, yearly_boost=yearly_boost)

    try:
        model.fit(Xtr, ytr, sample_weight=w_train)
    except TypeError:
        model.fit(Xtr, ytr)

    valid_df["Pred_Residual"] = model.predict(Xva)
    valid_df["Pred"] = np.clip(valid_df[BASELINE_COL] + valid_df["Pred_Residual"], 0, None)

    metrics = evaluate_all_metrics(valid_df[ACTUAL_TARGET_COL].values, valid_df["Pred"].values)

    aux = {
        "abc_map": prep_artifacts["abc_map"],
        "medium_profile_df": prep_artifacts["medium_profile_df"],
        "itemcode_categories": itemcode_categories,
        "clip_caps": prep_artifacts["clip_caps"],
        "subgroup_name": subgroup_name,
        "model_name": model_name,
        "feature_scaler": scaler
    }

    return model, valid_df, metrics, aux

# ============================================================
# 8) FEATURE PRUNING
# ============================================================
def iterative_feature_prune_residual_medium_subgroup(
    full_data,
    start_features,
    best_params,
    subgroup_name,
    model_name,
    drop_k=1,
    min_features=15,
    max_rounds=10,
    tolerance=0.10,
    n_repeats=5
):
    history = []
    features = start_features.copy()

    model, eval_df, m, aux = train_eval_validation_residual_medium_subgroup(
        full_data=full_data,
        feature_cols=features,
        best_params=best_params,
        subgroup_name=subgroup_name,
        model_name=model_name,
        valid_window=MEDIUM_PREV4_WINDOW
    )

    best_wmape = m["WMAPE"]
    last_accepted_state = (features.copy(), model, eval_df.copy(), m.copy(), aux.copy())

    for r in range(1, max_rounds + 1):
        if len(features) <= min_features:
            break

        imp = permutation_rank(
            model=model,
            eval_df=eval_df,
            feature_cols=features,
            target_col=MODEL_TARGET_COL,
            n_repeats=n_repeats
        )

        protected = {
            "ItemCode",
            "ABC_Class",
            "Lag1",
            "Lag2",
            "Lag3",
            "Rolling3M_Mean",
            "Rolling3M_Std",
            "Month_Sin",
            "Month_Cos",
            "Bonus_Flag",
            "Expected_Bonus_Month",
            "Supply_Constraint_Flag",
            "Medium_SKU_Type_Encoded"
        }

        drop_candidates = [
            f for f in imp.sort_values("perm_importance").feature.tolist()
            if f not in protected
        ]

        to_drop = drop_candidates[:drop_k]

        if not to_drop:
            break

        new_features = [f for f in features if f not in to_drop]

        new_model, new_eval_df, new_m, new_aux = train_eval_validation_residual_medium_subgroup(
            full_data=full_data,
            feature_cols=new_features,
            best_params=best_params,
            subgroup_name=subgroup_name,
            model_name=model_name,
            valid_window=MEDIUM_PREV4_WINDOW
        )

        history.append({
            "round": r,
            "subgroup": subgroup_name,
            "model_name": model_name,
            "valid_start_idx": MEDIUM_PREV4_WINDOW["valid_start_idx"],
            "valid_end_idx": MEDIUM_PREV4_WINDOW["valid_end_idx"],
            "dropped": to_drop,
            "n_features": len(new_features),
            **new_m
        })

        if new_m["WMAPE"] <= best_wmape + tolerance:
            features = new_features
            model, eval_df = new_model, new_eval_df
            best_wmape = min(best_wmape, new_m["WMAPE"])
            last_accepted_state = (features.copy(), model, eval_df.copy(), new_m.copy(), new_aux.copy())
        else:
            break

    results_df = pd.DataFrame(history)

    return (
        last_accepted_state[0],
        last_accepted_state[1],
        last_accepted_state[2],
        last_accepted_state[3],
        last_accepted_state[4],
        results_df
    )

# ============================================================
# 9) DEPLOYMENT TRAINING
# ============================================================
def prepare_medium_deploy_frame(full_data, subgroup_name):
    deploy_df = force_itemcode_str(full_data.copy())
    deploy_df = deploy_df[deploy_df["History_Segment"] == "MEDIUM"].copy()

    deploy_df, low_df, good_skus, low_skus = split_low_history_medium_skus(
        deploy_df,
        min_months=10
    )

    print(f"[MEDIUM DEPLOY] Skipping low-history SKUs: {len(low_skus)}")

    if deploy_df.empty:
        raise ValueError("No MEDIUM rows available for deployment.")

    # Fold-style adjustments, but using all deploy data
    train_df, deploy_df, abc_map = apply_fold_adjustments(
        deploy_df.copy(),
        deploy_df.copy()
    )

    sku_profile_df = build_sku_history_profile(train_df)
    deploy_df = merge_sku_history_profile(deploy_df, sku_profile_df)    

    medium_profile_df = build_medium_sku_profile(train_df)
    medium_profile_df = enforce_unique_subgroup(medium_profile_df)

    deploy_df = merge_medium_sku_profile(deploy_df, medium_profile_df)

    deploy_df = filter_medium_subgroup(deploy_df, subgroup_name)

    if deploy_df.empty:
        raise ValueError(f"No MEDIUM rows available for subgroup {subgroup_name}.")

    clip_cols = [
        "Rolling3M_Std",
        "Momentum",
        "Stock_Cover_Months",
        "Primary_Stock_Cover",
        "Distributor_Stock_Cover",
        "Free_Ratio",
        "SKU_CV",
        "Medium_CV",
        "Medium_Trend_Slope"
    ]

    caps = compute_clip_caps(deploy_df, clip_cols, q=0.99)
    deploy_df = apply_clip_caps(deploy_df, caps)

    deploy_df = deploy_df.dropna(
        subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]
    ).copy()

    if deploy_df.empty:
        raise ValueError(f"No usable MEDIUM deployment rows for {subgroup_name}.")

    deploy_df["ItemCode_Original"] = deploy_df["ItemCode"]
    deploy_df, _, itemcode_categories = encode_itemcode(
        deploy_df.copy(),
        deploy_df.copy()
    )

    artifacts = {
        "abc_map": abc_map,
        "clip_caps": caps,
        "medium_profile_df": medium_profile_df,
        "sku_profile_df": sku_profile_df,
        "itemcode_categories": itemcode_categories,
        "low_history_skus": sorted(list(low_skus)),
        "good_skus": sorted(list(good_skus))
    }

    return deploy_df, artifacts

def train_single_deployment_model_medium_subgroup_residual(
    full_data,
    feature_cols,
    best_params,
    subgroup_name,
    model_name
):
    model_name = model_name.upper()
    subgroup_name = subgroup_name.upper()

    print(
        f"\n========== {model_name} MEDIUM DEPLOYMENT MODEL "
        f"→ TRAIN ON ALL COMPLETE DATA ({subgroup_name}) =========="
    )

    deploy_df, prep_artifacts = prepare_medium_deploy_frame(
        full_data=full_data,
        subgroup_name=subgroup_name
    )

    assert_features_exist(
        deploy_df,
        feature_cols,
        where=f"{model_name}_{subgroup_name}_DEPLOY_TRAIN"
    )

    model = build_medium_model(model_name, best_params)

    Xtr = sanitize(deploy_df[feature_cols])
    ytr = deploy_df[MODEL_TARGET_COL]

    Xtr, _, scaler = maybe_scale_features(
        Xtr,
        None,
        model_name=model_name
    )

    w_train = build_train_weights(deploy_df, yearly_boost=0.25)

    try:
        model.fit(Xtr, ytr, sample_weight=w_train)
    except TypeError:
        model.fit(Xtr, ytr)

    artifacts = {
        "model": model,
        "feature_cols": feature_cols,
        "best_params": best_params,
        "itemcode_categories": prep_artifacts["itemcode_categories"],
        "abc_map": prep_artifacts["abc_map"],
        "clip_caps": prep_artifacts["clip_caps"],
        "medium_profile_df": prep_artifacts["medium_profile_df"],
        "promo_profile_df": None,
        "sku_profile_df": prep_artifacts["sku_profile_df"],
        "target_mode": "residual",
        "baseline_col": BASELINE_COL,
        "actual_target_col": ACTUAL_TARGET_COL,
        "model_target_col": MODEL_TARGET_COL,
        "segment": "MEDIUM",
        "subgroup_name": subgroup_name,
        "model_name": model_name,
        "feature_scaler": scaler,
        "low_history_skus": prep_artifacts["low_history_skus"],
        "good_skus": prep_artifacts["good_skus"]
    }

    return artifacts, deploy_df

# ============================================================
# 10) RUN MEDIUM MODEL PIPELINE
# ============================================================
def train_single_window_validation_medium_subgroup_residual(
    full_data,
    feature_cols,
    best_params,
    subgroup_name,
    model_name,
    valid_window
):
    print(
        f"\n========== {model_name} WINDOW VALIDATION "
        f"→ VALID ON PERIODS {valid_window['valid_start_idx']} to {valid_window['valid_end_idx']} "
        f"({subgroup_name}) =========="
    )

    model, valid_df, metrics, aux = train_eval_validation_residual_medium_subgroup(
        full_data=full_data,
        feature_cols=feature_cols,
        best_params=best_params,
        subgroup_name=subgroup_name,
        model_name=model_name,
        valid_window=valid_window
    )

    return valid_df, metrics

def run_medium_model_pipeline(
    full_data,
    feature_cols,
    subgroup_name,
    model_name,
    n_trials=30,
    drop_k=1,
    min_features=15,
    max_rounds=10,
    tolerance=0.10,
    n_repeats=5
):
    best_params, study = tune_residual_medium_subgroup(
        full_data=full_data,
        feature_cols=feature_cols,
        subgroup_name=subgroup_name,
        model_name=model_name,
        n_trials=n_trials,
        study_name=f"residual_{model_name.lower()}_medium_{subgroup_name.lower()}"
    )
    print(f"{model_name} {subgroup_name} best params:", best_params)

    best_feats, _, _, best_metrics, _, prune_log = iterative_feature_prune_residual_medium_subgroup(
        full_data=full_data,
        start_features=feature_cols,
        best_params=best_params,
        subgroup_name=subgroup_name,
        model_name=model_name,
        drop_k=drop_k,
        min_features=min_features,
        max_rounds=max_rounds,
        tolerance=tolerance,
        n_repeats=n_repeats
    )

    print(f"\n===== {model_name} {subgroup_name} BEST METRICS AFTER FEATURE PRUNING =====")
    print(best_metrics)

    print(f"\n===== {model_name} {subgroup_name} BEST FEATURES =====")
    print(best_feats)

    print(f"\n===== {model_name} {subgroup_name} FEATURE PRUNE LOG =====")
    print(prune_log)

    prev4_valid_df, prev4_valid_metrics = train_single_window_validation_medium_subgroup_residual(
        full_data=full_data,
        feature_cols=best_feats,
        best_params=best_params,
        subgroup_name=subgroup_name,
        model_name=model_name,
        valid_window=MEDIUM_PREV4_WINDOW
    )

    print(f"\n===== {model_name} {subgroup_name} PREV4 VALIDATION METRICS =====")
    print(prev4_valid_metrics)

    recent4_valid_df, recent4_valid_metrics = train_single_window_validation_medium_subgroup_residual(
        full_data=full_data,
        feature_cols=best_feats,
        best_params=best_params,
        subgroup_name=subgroup_name,
        model_name=model_name,
        valid_window=MEDIUM_RECENT4_WINDOW
    )

    print(f"\n===== {model_name} {subgroup_name} RECENT4 VALIDATION METRICS =====")
    print(recent4_valid_metrics)

    deploy_artifacts, deploy_train_df = train_single_deployment_model_medium_subgroup_residual(
        full_data=full_data,
        feature_cols=best_feats,
        best_params=best_params,
        subgroup_name=subgroup_name,
        model_name=model_name
    )

    return {
        "best_params": best_params,
        "best_feats": best_feats,
        "study": study,
        "prune_log": prune_log,
        "best_metrics": best_metrics,
        "deploy_artifacts": deploy_artifacts,
        "deploy_train_df": deploy_train_df,
        "prev4_valid": prev4_valid_df,
        "prev4_valid_metrics": prev4_valid_metrics,
        "recent4_valid": recent4_valid_df,
        "recent4_valid_metrics": recent4_valid_metrics,
    }


In [34]:
# ============================================================
# DEBUG HELPERS
# ============================================================
def log_medium_step(df, label):
    print(f"[MEDIUM DEBUG] {label} | rows={len(df)} | skus={df['ItemCode'].astype(str).nunique()}")

def summarize_missing_skus(full_data, compared_df, label="MEDIUM"):
    all_skus = set(
        add_history_length_from_subset(full_data.copy(), full_data.copy())
        .query("History_Segment == 'MEDIUM'")["ItemCode"]
        .astype(str).str.replace(".0", "", regex=False).unique()
    )
    covered_skus = set(
        compared_df["ItemCode"]
        .astype(str).str.replace(".0", "", regex=False).unique()
    )
    missing = sorted(list(all_skus - covered_skus))

    print(f"\n[{label} COVERAGE]")
    print("Actual MEDIUM SKUs:", len(all_skus))
    print("Covered MEDIUM SKUs:", len(covered_skus))
    print("Missing MEDIUM SKUs:", len(missing))
    print("Sample missing:", missing[:20])

    return missing

def debug_medium_survival(full_data, subgroup_name, valid_window):
    full_data = add_period_index(full_data)

    train_df = full_data[full_data["Period_Index"] < valid_window["valid_start_idx"]].copy()
    valid_df = full_data[full_data["Period_Index"].between(valid_window["valid_start_idx"], valid_window["valid_end_idx"])].copy()

    train_df = add_history_length_from_subset(train_df, train_df)
    valid_df = add_history_length_from_subset(train_df, valid_df)

    train_df = train_df[train_df["History_Segment"] == "MEDIUM"].copy()
    valid_df = valid_df[valid_df["History_Segment"] == "MEDIUM"].copy()

    print("raw valid medium skus:", valid_df["ItemCode"].astype(str).nunique())

    train_df, valid_df, *_ = prepare_medium_subgroup_frame_foldsafe(train_df, valid_df)

    if train_df is None or valid_df is None:
        print("after foldsafe: no usable rows")
        return pd.DataFrame()
    
    print("after foldsafe valid skus:", valid_df["ItemCode"].astype(str).nunique())

    valid_df = filter_medium_subgroup(valid_df, subgroup_name)
    print("after subgroup filter valid skus:", valid_df["ItemCode"].astype(str).nunique())

    valid_df = valid_df.dropna(subset=[ACTUAL_TARGET_COL, MODEL_TARGET_COL]).copy()
    print("after target build valid skus:", valid_df["ItemCode"].astype(str).nunique())

    return valid_df

print("\n===== DEBUG MEDIUM SURVIVAL: PROMO_HEAVY / PREV4 =====")
_ = debug_medium_survival(Data, "PROMO_HEAVY", MEDIUM_PREV4_WINDOW)

print("\n===== DEBUG MEDIUM SURVIVAL: STABLE / PREV4 =====")
_ = debug_medium_survival(Data, "STABLE", MEDIUM_PREV4_WINDOW)

print("\n===== DEBUG MEDIUM SURVIVAL: PROMO_HEAVY / RECENT4 =====")
_ = debug_medium_survival(Data, "PROMO_HEAVY", MEDIUM_RECENT4_WINDOW)

print("\n===== DEBUG MEDIUM SURVIVAL: STABLE / RECENT4 =====")
_ = debug_medium_survival(Data, "STABLE", MEDIUM_RECENT4_WINDOW)



===== DEBUG MEDIUM SURVIVAL: PROMO_HEAVY / PREV4 =====
raw valid medium skus: 13
after foldsafe valid skus: 13
after subgroup filter valid skus: 7
after target build valid skus: 7

===== DEBUG MEDIUM SURVIVAL: STABLE / PREV4 =====
raw valid medium skus: 13
after foldsafe valid skus: 13
after subgroup filter valid skus: 6
after target build valid skus: 6

===== DEBUG MEDIUM SURVIVAL: PROMO_HEAVY / RECENT4 =====
raw valid medium skus: 15
after foldsafe valid skus: 15
after subgroup filter valid skus: 7
after target build valid skus: 7

===== DEBUG MEDIUM SURVIVAL: STABLE / RECENT4 =====
raw valid medium skus: 15
after foldsafe valid skus: 15
after subgroup filter valid skus: 8
after target build valid skus: 8


In [35]:
# ============================================================
# 11) RUN MEDIUM MODELS
# ============================================================
medium_runs = {}

# PROMO_HEAVY
medium_runs["PROMO_HEAVY_XGBOOST"] = run_medium_model_pipeline(
    full_data=Data,
    feature_cols=MEDIUM_SUBGROUP_FEATURE_COLS,
    subgroup_name="PROMO_HEAVY",
    model_name="XGBOOST",
    n_trials=30
)

medium_runs["PROMO_HEAVY_CATBOOST"] = run_medium_model_pipeline(
    full_data=Data,
    feature_cols=MEDIUM_SUBGROUP_FEATURE_COLS,
    subgroup_name="PROMO_HEAVY",
    model_name="CATBOOST",
    n_trials=30
)

# STABLE
medium_runs["STABLE_XGBOOST"] = run_medium_model_pipeline(
    full_data=Data,
    feature_cols=MEDIUM_SUBGROUP_FEATURE_COLS,
    subgroup_name="STABLE",
    model_name="XGBOOST",
    n_trials=30
)

medium_runs["STABLE_CATBOOST"] = run_medium_model_pipeline(
    full_data=Data,
    feature_cols=MEDIUM_SUBGROUP_FEATURE_COLS,
    subgroup_name="STABLE",
    model_name="CATBOOST",
    n_trials=30
)

medium_runs["STABLE_RANDOM_FOREST"] = run_medium_model_pipeline(
    full_data=Data,
    feature_cols=MEDIUM_SUBGROUP_FEATURE_COLS,
    subgroup_name="STABLE",
    model_name="RANDOM_FOREST",
    n_trials=20
)

for k in medium_runs:
    if "prev4_valid" in medium_runs[k]:
        medium_runs[k]["prev4_valid"] = force_itemcode_str(medium_runs[k]["prev4_valid"])
    if "recent4_valid" in medium_runs[k]:
        medium_runs[k]["recent4_valid"] = force_itemcode_str(medium_runs[k]["recent4_valid"])

for run_name, run_obj in medium_runs.items():
    deploy_name = f"{run_name.lower()}_deploy_artifacts_residual.pkl"
    joblib.dump(run_obj["deploy_artifacts"], deploy_name)
    print("Saved:", deploy_name)


[I 2026-07-26 11:21:00,257] A new study created in memory with name: residual_xgboost_medium_promo_heavy
[I 2026-07-26 11:21:01,101] Trial 0 finished with value: 61.463265011891224 and parameters: {'n_estimators': 629, 'learning_rate': 0.030017749389533614, 'max_depth': 5, 'min_child_weight': 4, 'subsample': 0.8085136346246775, 'colsample_bytree': 0.777046801702625, 'gamma': 0.32047533171500553, 'reg_lambda': 17.013869194034076, 'reg_alpha': 3.7989898991293365}. Best is trial 0 with value: 61.463265011891224.
[I 2026-07-26 11:21:02,105] Trial 1 finished with value: 60.74009881697029 and parameters: {'n_estimators': 1010, 'learning_rate': 0.018826130079614985, 'max_depth': 5, 'min_child_weight': 5, 'subsample': 0.7684041959623404, 'colsample_bytree': 0.8091815775382325, 'gamma': 0.43535789867234875, 'reg_lambda': 16.806813185781635, 'reg_alpha': 4.343668243622506}. Best is trial 1 with value: 60.74009881697029.
[I 2026-07-26 11:21:03,338] Trial 2 finished with value: 59.28164882209357 a

XGBOOST PROMO_HEAVY best params: {'n_estimators': 1151, 'learning_rate': 0.019856477018257303, 'max_depth': 6, 'min_child_weight': 8, 'subsample': 0.7371311108942703, 'colsample_bytree': 0.7013207206362795, 'gamma': 0.14255445098937386, 'reg_lambda': 19.093003460753167, 'reg_alpha': 3.3088844593305207}

===== XGBOOST PROMO_HEAVY BEST METRICS AFTER FEATURE PRUNING =====
{'WMAPE': 38.50561565461747, 'Bias': 22.05086296515434, 'MAE': 1799.4902707902204, 'RMSE': 3579.027168433471, 'Underforecast_Rate': 8.227376344731576}

===== XGBOOST PROMO_HEAVY BEST FEATURES =====
['ItemCode', 'Month_Number', 'Month_Sin', 'Month_Cos', 'Lag1', 'Lag2', 'Lag3', 'Lag6', 'Lag12', 'Rolling3M_Mean', 'Rolling6M_Mean', 'Rolling3M_Std', 'Momentum', 'Bonus_Flag', 'Bonus_Flag_Lag1', 'Bonus_Flag_Lag2', 'Bonus_Flag_Lag3', 'Bonus_Frequency_12M', 'Bonus_Frequency_All', 'Recurring_Bonus_SKU', 'Bonus_Cycle_Length', 'Months_Since_Last_Bonus', 'Expected_Bonus_Month', 'Avg_Bonus_Uplift', 'Bonus_Shock', 'Last_Bonus_Demand', 

[I 2026-07-26 11:21:43,581] A new study created in memory with name: residual_catboost_medium_promo_heavy
[I 2026-07-26 11:21:44,757] Trial 0 finished with value: 55.40504217563104 and parameters: {'iterations': 749, 'learning_rate': 0.040682532278551634, 'depth': 5, 'l2_leaf_reg': 14.855010651277745, 'min_data_in_leaf': 34, 'subsample': 0.9203979264989924, 'rsm': 0.7452556554288494, 'random_strength': 0.19717126729186918, 'bagging_temperature': 2.879777895599248}. Best is trial 0 with value: 55.40504217563104.
[I 2026-07-26 11:21:45,955] Trial 1 finished with value: 54.338467929099224 and parameters: {'iterations': 508, 'learning_rate': 0.03177980362694404, 'depth': 6, 'l2_leaf_reg': 10.43325073334666, 'min_data_in_leaf': 30, 'subsample': 0.8396319624040132, 'rsm': 0.7068613670407276, 'random_strength': 0.650730783227046, 'bagging_temperature': 0.2092370089613671}. Best is trial 1 with value: 54.338467929099224.
[I 2026-07-26 11:21:49,462] Trial 2 finished with value: 51.4793797794629

CATBOOST PROMO_HEAVY best params: {'iterations': 1081, 'learning_rate': 0.020221609373301936, 'depth': 6, 'l2_leaf_reg': 12.043899501307013, 'min_data_in_leaf': 8, 'subsample': 0.7234059746803694, 'rsm': 0.7505639008144279, 'random_strength': 1.878504896081763, 'bagging_temperature': 0.8051634753745758}

===== CATBOOST PROMO_HEAVY BEST METRICS AFTER FEATURE PRUNING =====
{'WMAPE': 46.21414814224535, 'Bias': 22.969456540021987, 'MAE': 2159.734587826436, 'RMSE': 3867.6362137370315, 'Underforecast_Rate': 11.62234580111168}

===== CATBOOST PROMO_HEAVY BEST FEATURES =====
['ItemCode', 'Month_Number', 'Month_Sin', 'Month_Cos', 'Lag1', 'Lag2', 'Lag3', 'Lag6', 'Lag12', 'Rolling3M_Mean', 'Rolling6M_Mean', 'Rolling3M_Std', 'Momentum', 'Bonus_Flag', 'Bonus_Flag_Lag1', 'Bonus_Flag_Lag2', 'Bonus_Flag_Lag3', 'Bonus_Frequency_12M', 'Bonus_Frequency_All', 'Recurring_Bonus_SKU', 'Bonus_Cycle_Length', 'Months_Since_Last_Bonus', 'Expected_Bonus_Month', 'Avg_Bonus_Uplift', 'Bonus_Shock', 'Last_Bonus_Deman

[I 2026-07-26 11:22:55,380] A new study created in memory with name: residual_xgboost_medium_stable
[I 2026-07-26 11:22:55,998] Trial 0 finished with value: 41.426055902164435 and parameters: {'n_estimators': 509, 'learning_rate': 0.01509174937276311, 'max_depth': 3, 'min_child_weight': 9, 'subsample': 0.7313145591398585, 'colsample_bytree': 0.8167978047554058, 'gamma': 0.4342059962876613, 'reg_lambda': 21.875992705660966, 'reg_alpha': 3.095708780122791}. Best is trial 0 with value: 41.426055902164435.
[I 2026-07-26 11:22:56,488] Trial 1 finished with value: 40.96668299102616 and parameters: {'n_estimators': 418, 'learning_rate': 0.027289139003893098, 'max_depth': 5, 'min_child_weight': 6, 'subsample': 0.733787528734739, 'colsample_bytree': 0.7754015460831107, 'gamma': 0.2741621288062514, 'reg_lambda': 13.48989131466854, 'reg_alpha': 5.604759792659225}. Best is trial 1 with value: 40.96668299102616.
[I 2026-07-26 11:22:57,104] Trial 2 finished with value: 41.51167892935702 and paramete

XGBOOST STABLE best params: {'n_estimators': 462, 'learning_rate': 0.02858574530049276, 'max_depth': 5, 'min_child_weight': 10, 'subsample': 0.8737165980028587, 'colsample_bytree': 0.6779355192796341, 'gamma': 0.26162995267335204, 'reg_lambda': 13.425818749471665, 'reg_alpha': 2.163823573193875}

===== XGBOOST STABLE BEST METRICS AFTER FEATURE PRUNING =====
{'WMAPE': 19.247810486242702, 'Bias': -8.01405658864222, 'MAE': 73.81535321474077, 'RMSE': 114.97953903109509, 'Underforecast_Rate': 13.630933537442461}

===== XGBOOST STABLE BEST FEATURES =====
['ItemCode', 'Month_Sin', 'Month_Cos', 'Lag1', 'Lag2', 'Lag3', 'Lag6', 'Lag12', 'Rolling3M_Mean', 'Rolling6M_Mean', 'Rolling3M_Std', 'Momentum', 'Bonus_Flag', 'Bonus_Flag_Lag1', 'Bonus_Flag_Lag2', 'Bonus_Flag_Lag3', 'Bonus_Frequency_12M', 'Bonus_Frequency_All', 'Recurring_Bonus_SKU', 'Bonus_Cycle_Length', 'Months_Since_Last_Bonus', 'Expected_Bonus_Month', 'Avg_Bonus_Uplift', 'Bonus_Shock', 'Last_Bonus_Demand', 'Demand_State_Encoded', 'Bonus_

[I 2026-07-26 11:23:15,165] A new study created in memory with name: residual_catboost_medium_stable
[I 2026-07-26 11:23:15,677] Trial 0 finished with value: 43.58454853511063 and parameters: {'iterations': 633, 'learning_rate': 0.023172395573682218, 'depth': 4, 'l2_leaf_reg': 3.3123971215225816, 'min_data_in_leaf': 12, 'subsample': 0.8000537701525061, 'rsm': 0.8468423976453859, 'random_strength': 0.28879978755491365, 'bagging_temperature': 2.197759645283977}. Best is trial 0 with value: 43.58454853511063.
[I 2026-07-26 11:23:16,146] Trial 1 finished with value: 39.99336773944777 and parameters: {'iterations': 457, 'learning_rate': 0.019534772992101913, 'depth': 5, 'l2_leaf_reg': 7.7253292009562, 'min_data_in_leaf': 40, 'subsample': 0.7623812715398667, 'rsm': 0.7761815989594919, 'random_strength': 1.434530384284497, 'bagging_temperature': 0.10503086976507287}. Best is trial 1 with value: 39.99336773944777.
[I 2026-07-26 11:23:16,934] Trial 2 finished with value: 41.62709110654973 and p

CATBOOST STABLE best params: {'iterations': 412, 'learning_rate': 0.02315798984298184, 'depth': 5, 'l2_leaf_reg': 7.042725650071104, 'min_data_in_leaf': 40, 'subsample': 0.822916522959279, 'rsm': 0.8813474679871891, 'random_strength': 1.1320971992006712, 'bagging_temperature': 1.5141201989835125}

===== CATBOOST STABLE BEST METRICS AFTER FEATURE PRUNING =====
{'WMAPE': 18.273737551391005, 'Bias': -8.185895851954076, 'MAE': 70.0797835095845, 'RMSE': 107.1261832391393, 'Underforecast_Rate': 13.22981670167254}

===== CATBOOST STABLE BEST FEATURES =====
['ItemCode', 'Month_Number', 'Month_Sin', 'Month_Cos', 'Lag1', 'Lag2', 'Lag3', 'Lag6', 'Lag12', 'Rolling3M_Mean', 'Rolling6M_Mean', 'Rolling3M_Std', 'Momentum', 'Bonus_Flag', 'Bonus_Flag_Lag1', 'Bonus_Flag_Lag2', 'Bonus_Flag_Lag3', 'Bonus_Frequency_12M', 'Bonus_Frequency_All', 'Recurring_Bonus_SKU', 'Bonus_Cycle_Length', 'Months_Since_Last_Bonus', 'Expected_Bonus_Month', 'Avg_Bonus_Uplift', 'Bonus_Shock', 'Last_Bonus_Demand', 'Demand_State_

[I 2026-07-26 11:23:32,936] A new study created in memory with name: residual_random_forest_medium_stable
[I 2026-07-26 11:23:34,189] Trial 0 finished with value: 39.2655837844717 and parameters: {'n_estimators': 889, 'max_depth': 4, 'min_samples_split': 11, 'min_samples_leaf': 8, 'max_features': 0.6, 'bootstrap': False}. Best is trial 0 with value: 39.2655837844717.
[I 2026-07-26 11:23:35,362] Trial 1 finished with value: 40.78274333976249 and parameters: {'n_estimators': 709, 'max_depth': 14, 'min_samples_split': 2, 'min_samples_leaf': 7, 'max_features': 0.8, 'bootstrap': True}. Best is trial 0 with value: 39.2655837844717.
[I 2026-07-26 11:23:36,734] Trial 2 finished with value: 41.67259045534215 and parameters: {'n_estimators': 860, 'max_depth': 10, 'min_samples_split': 7, 'min_samples_leaf': 7, 'max_features': 'log2', 'bootstrap': True}. Best is trial 0 with value: 39.2655837844717.
[I 2026-07-26 11:23:37,809] Trial 3 finished with value: 42.67624194020854 and parameters: {'n_esti

RANDOM_FOREST STABLE best params: {'n_estimators': 368, 'max_depth': 6, 'min_samples_split': 12, 'min_samples_leaf': 1, 'max_features': 'log2', 'bootstrap': False}

===== RANDOM_FOREST STABLE BEST METRICS AFTER FEATURE PRUNING =====
{'WMAPE': 20.53130291419579, 'Bias': -16.738247694494458, 'MAE': 78.73754667594086, 'RMSE': 127.93024654362952, 'Underforecast_Rate': 18.634775304345126}

===== RANDOM_FOREST STABLE BEST FEATURES =====
['ItemCode', 'Month_Number', 'Month_Sin', 'Month_Cos', 'Lag1', 'Lag2', 'Lag3', 'Lag6', 'Lag12', 'Rolling3M_Mean', 'Rolling6M_Mean', 'Rolling3M_Std', 'Momentum', 'Bonus_Flag', 'Bonus_Flag_Lag1', 'Bonus_Flag_Lag2', 'Bonus_Flag_Lag3', 'Bonus_Frequency_12M', 'Bonus_Frequency_All', 'Recurring_Bonus_SKU', 'Bonus_Cycle_Length', 'Months_Since_Last_Bonus', 'Expected_Bonus_Month', 'Avg_Bonus_Uplift', 'Bonus_Shock', 'Last_Bonus_Demand', 'Demand_State_Encoded', 'Bonus_Last_Month', 'Bonus_2M_Ago', 'Post_Bonus_Month', 'Promo_Decay_Ratio', 'Demand_Regime_Encoded', 'Current_

#### COMPARISON

In [36]:
# ============================================================
# 12) STANDARDIZE OUTPUT
# ============================================================
def standardize_medium_model_output(
    df,
    actual_col,
    pred_col,
    item_col="ItemCode",
    year_col="Year",
    month_col="Month_Number",
    model_name="UNKNOWN",
    segment="MEDIUM",
    subgroup_name=""
):
    out = df.copy()
    out = force_itemcode_str(out)

    if item_col not in out.columns:
        raise KeyError(f"Missing item column: {item_col}")
    if actual_col not in out.columns:
        raise KeyError(f"Missing actual column: {actual_col}")
    if pred_col not in out.columns:
        raise KeyError(f"Missing pred column: {pred_col}")

    out["ItemCode_Original"] = out[item_col].astype(str)
    out["ItemCode"] = out["ItemCode_Original"]
    out["Actual"] = pd.to_numeric(out[actual_col], errors="coerce")
    out["Pred"] = pd.to_numeric(out[pred_col], errors="coerce").clip(lower=0)

    if year_col in out.columns:
        out["Year"] = pd.to_numeric(out[year_col], errors="coerce")
    else:
        out["Year"] = np.nan

    if month_col in out.columns:
        out["Month_Number"] = pd.to_numeric(out[month_col], errors="coerce")
    else:
        out["Month_Number"] = np.nan

    out["Error"] = out["Actual"] - out["Pred"]
    out["Abs_Error"] = np.abs(out["Error"])
    out["Segment"] = segment
    out["Model_Name"] = str(model_name).upper()
    out["Medium_Subgroup"] = str(subgroup_name).upper()

    keep_cols = [
        "ItemCode",
        "ItemCode_Original",
        "Year",
        "Month_Number",
        "Actual",
        "Pred",
        "Error",
        "Abs_Error",
        "Segment",
        "Model_Name",
        "Medium_Subgroup"
    ]

    extra_cols = [c for c in out.columns if c not in keep_cols]
    return out[keep_cols + extra_cols].copy()


# ============================================================
# 13) BUILD PREV4 / RECENT4 ROW-LEVEL EVAL DATA
# ============================================================
medium_prev4_eval_df = pd.concat(
    [
        standardize_medium_model_output(
            medium_runs["PROMO_HEAVY_XGBOOST"]["prev4_valid"],
            actual_col=ACTUAL_TARGET_COL,
            pred_col="Pred",
            item_col="ItemCode_Original",
            year_col="Year",
            month_col="Month_Number",
            model_name="XGBOOST",
            segment="MEDIUM",
            subgroup_name="PROMO_HEAVY"
        ),
        standardize_medium_model_output(
            medium_runs["PROMO_HEAVY_CATBOOST"]["prev4_valid"],
            actual_col=ACTUAL_TARGET_COL,
            pred_col="Pred",
            item_col="ItemCode_Original",
            year_col="Year",
            month_col="Month_Number",
            model_name="CATBOOST",
            segment="MEDIUM",
            subgroup_name="PROMO_HEAVY"
        ),
        standardize_medium_model_output(
            medium_runs["STABLE_XGBOOST"]["prev4_valid"],
            actual_col=ACTUAL_TARGET_COL,
            pred_col="Pred",
            item_col="ItemCode_Original",
            year_col="Year",
            month_col="Month_Number",
            model_name="XGBOOST",
            segment="MEDIUM",
            subgroup_name="STABLE"
        ),
        standardize_medium_model_output(
            medium_runs["STABLE_CATBOOST"]["prev4_valid"],
            actual_col=ACTUAL_TARGET_COL,
            pred_col="Pred",
            item_col="ItemCode_Original",
            year_col="Year",
            month_col="Month_Number",
            model_name="CATBOOST",
            segment="MEDIUM",
            subgroup_name="STABLE"
        ),
        standardize_medium_model_output(
            medium_runs["STABLE_RANDOM_FOREST"]["prev4_valid"],
            actual_col=ACTUAL_TARGET_COL,
            pred_col="Pred",
            item_col="ItemCode_Original",
            year_col="Year",
            month_col="Month_Number",
            model_name="RANDOM_FOREST",
            segment="MEDIUM",
            subgroup_name="STABLE"
        ),
    ],
    ignore_index=True
)

medium_recent4_eval_df = pd.concat(
    [
        standardize_medium_model_output(
            medium_runs["PROMO_HEAVY_XGBOOST"]["recent4_valid"],
            actual_col=ACTUAL_TARGET_COL,
            pred_col="Pred",
            item_col="ItemCode_Original",
            year_col="Year",
            month_col="Month_Number",
            model_name="XGBOOST",
            segment="MEDIUM",
            subgroup_name="PROMO_HEAVY"
        ),
        standardize_medium_model_output(
            medium_runs["PROMO_HEAVY_CATBOOST"]["recent4_valid"],
            actual_col=ACTUAL_TARGET_COL,
            pred_col="Pred",
            item_col="ItemCode_Original",
            year_col="Year",
            month_col="Month_Number",
            model_name="CATBOOST",
            segment="MEDIUM",
            subgroup_name="PROMO_HEAVY"
        ),
        standardize_medium_model_output(
            medium_runs["STABLE_XGBOOST"]["recent4_valid"],
            actual_col=ACTUAL_TARGET_COL,
            pred_col="Pred",
            item_col="ItemCode_Original",
            year_col="Year",
            month_col="Month_Number",
            model_name="XGBOOST",
            segment="MEDIUM",
            subgroup_name="STABLE"
        ),
        standardize_medium_model_output(
            medium_runs["STABLE_CATBOOST"]["recent4_valid"],
            actual_col=ACTUAL_TARGET_COL,
            pred_col="Pred",
            item_col="ItemCode_Original",
            year_col="Year",
            month_col="Month_Number",
            model_name="CATBOOST",
            segment="MEDIUM",
            subgroup_name="STABLE"
        ),
        standardize_medium_model_output(
            medium_runs["STABLE_RANDOM_FOREST"]["recent4_valid"],
            actual_col=ACTUAL_TARGET_COL,
            pred_col="Pred",
            item_col="ItemCode_Original",
            year_col="Year",
            month_col="Month_Number",
            model_name="RANDOM_FOREST",
            segment="MEDIUM",
            subgroup_name="STABLE"
        ),
    ],
    ignore_index=True
)


# ============================================================
# 14) CLEAN TYPES
# ============================================================
for df_ in [medium_prev4_eval_df, medium_recent4_eval_df]:
    for c in ["ItemCode", "ItemCode_Original", "Model_Name", "Medium_Subgroup"]:
        df_[c] = df_[c].astype(str)

    for c in ["Actual", "Pred", "Abs_Error", "Year", "Month_Number"]:
        df_[c] = pd.to_numeric(df_[c], errors="coerce")

    df_.dropna(
        subset=["ItemCode", "Medium_Subgroup", "Model_Name", "Actual", "Pred", "Abs_Error", "Year", "Month_Number"],
        inplace=True
    )

# ============================================================
# 15) REPORTS
# ============================================================
for model_name in ["XGBOOST", "CATBOOST", "RANDOM_FOREST"]:
    df_prev4 = medium_prev4_eval_df[medium_prev4_eval_df["Model_Name"] == model_name].copy()
    if not df_prev4.empty:
        print_model_eval_report(
            df_prev4,
            title=f"{model_name} MEDIUM → PREV4",
            group_cols=["Medium_Subgroup"]
        )

for model_name in ["XGBOOST", "CATBOOST", "RANDOM_FOREST"]:
    df_recent4 = medium_recent4_eval_df[medium_recent4_eval_df["Model_Name"] == model_name].copy()
    if not df_recent4.empty:
        print_model_eval_report(
            df_recent4,
            title=f"{model_name} MEDIUM → RECENT4",
            group_cols=["Medium_Subgroup"]
        )

medium_prev4_model_table = build_model_summary_table(
    medium_prev4_eval_df,
    extra_group_cols=["Medium_Subgroup"]
)

medium_recent4_model_table = build_model_summary_table(
    medium_recent4_eval_df,
    extra_group_cols=["Medium_Subgroup"]
)

print("\n===== MEDIUM PREV4 MODEL TABLE =====")
print(medium_prev4_model_table)

print("\n===== MEDIUM RECENT4 MODEL TABLE =====")
print(medium_recent4_model_table)

# ============================================================
# 16) SKU-MODEL SUMMARIES
# ============================================================
medium_sku_model_prev4_summary = (
    medium_prev4_eval_df
    .groupby(["ItemCode", "Medium_Subgroup", "Model_Name"], as_index=False)
    .agg(
        Prev4_Months=("Month_Number", "count"),
        Prev4_Actual_Sum=("Actual", "sum"),
        Prev4_Pred_Sum=("Pred", "sum"),
        Prev4_MAE=("Abs_Error", "mean"),
        Prev4_Total_Abs_Error=("Abs_Error", "sum")
    )
)

medium_sku_model_prev4_summary["Prev4_WMAPE"] = np.where(
    medium_sku_model_prev4_summary["Prev4_Actual_Sum"] > 0,
    medium_sku_model_prev4_summary["Prev4_Total_Abs_Error"] /
    medium_sku_model_prev4_summary["Prev4_Actual_Sum"] * 100,
    np.nan
)

medium_sku_model_prev4_summary["Prev4_Bias"] = np.where(
    medium_sku_model_prev4_summary["Prev4_Actual_Sum"] > 0,
    (medium_sku_model_prev4_summary["Prev4_Pred_Sum"] -
     medium_sku_model_prev4_summary["Prev4_Actual_Sum"]) /
    medium_sku_model_prev4_summary["Prev4_Actual_Sum"] * 100,
    np.nan
)

medium_sku_model_recent4_summary = (
    medium_recent4_eval_df
    .groupby(["ItemCode", "Medium_Subgroup", "Model_Name"], as_index=False)
    .agg(
        Recent4_Months=("Month_Number", "count"),
        Recent4_Actual_Sum=("Actual", "sum"),
        Recent4_Pred_Sum=("Pred", "sum"),
        Recent4_MAE=("Abs_Error", "mean"),
        Recent4_Total_Abs_Error=("Abs_Error", "sum")
    )
)

medium_sku_model_recent4_summary["Recent4_WMAPE"] = np.where(
    medium_sku_model_recent4_summary["Recent4_Actual_Sum"] > 0,
    medium_sku_model_recent4_summary["Recent4_Total_Abs_Error"] /
    medium_sku_model_recent4_summary["Recent4_Actual_Sum"] * 100,
    np.nan
)

medium_sku_model_recent4_summary["Recent4_Bias"] = np.where(
    medium_sku_model_recent4_summary["Recent4_Actual_Sum"] > 0,
    (medium_sku_model_recent4_summary["Recent4_Pred_Sum"] -
     medium_sku_model_recent4_summary["Recent4_Actual_Sum"]) /
    medium_sku_model_recent4_summary["Recent4_Actual_Sum"] * 100,
    np.nan
)

for df_ in [medium_sku_model_prev4_summary, medium_sku_model_recent4_summary]:
    df_["ItemCode"] = df_["ItemCode"].astype(str)
    df_["Model_Name"] = df_["Model_Name"].astype(str).str.upper()

# ============================================================
# 17) MERGE PREV4 + RECENT4
# ============================================================
current_medium_skus = get_current_medium_skus(Data)

medium_union_base = pd.concat([
    medium_sku_model_prev4_summary[["ItemCode", "Medium_Subgroup", "Model_Name"]],
    medium_sku_model_recent4_summary[["ItemCode", "Medium_Subgroup", "Model_Name"]],
], ignore_index=True).drop_duplicates()

medium_union_base = medium_union_base[
    medium_union_base["ItemCode"].isin(current_medium_skus)
].copy()

medium_sku_model_summary = (
    medium_union_base
    .merge(
        medium_sku_model_prev4_summary,
        on=["ItemCode", "Medium_Subgroup", "Model_Name"],
        how="left"
    )
    .merge(
        medium_sku_model_recent4_summary,
        on=["ItemCode", "Medium_Subgroup", "Model_Name"],
        how="left"
    )
)

fill_zero_cols = [
    "Prev4_Months", "Prev4_Actual_Sum", "Prev4_Pred_Sum", "Prev4_Total_Abs_Error",
    "Recent4_Months", "Recent4_Actual_Sum", "Recent4_Pred_Sum", "Recent4_Total_Abs_Error"
]
for c in fill_zero_cols:
    if c in medium_sku_model_summary.columns:
        medium_sku_model_summary[c] = medium_sku_model_summary[c].fillna(0)

# ============================================================
# 18) CHAMPION SCORE
# ============================================================
def compute_medium_champion_score(row):
    prev_months = row.get("Prev4_Months", 0)
    recent_months = row.get("Recent4_Months", 0)

    if prev_months == 0 and recent_months == 0:
        return np.nan

    recent_wmape = row.get("Recent4_WMAPE", np.nan)
    prev_wmape = row.get("Prev4_WMAPE", np.nan)
    recent_bias = row.get("Recent4_Bias", np.nan)

    parts = []
    weights = []

    # If Recent4 has enough evidence, use recent-heavy scoring
    if recent_months >= 3:
        if pd.notna(recent_wmape):
            parts.append(recent_wmape)
            weights.append(0.60)

        if prev_months > 0 and pd.notna(prev_wmape):
            parts.append(prev_wmape)
            weights.append(0.20)

        if pd.notna(recent_bias):
            parts.append(abs(recent_bias))
            weights.append(0.20)

    # If Recent4 is weak, balance Prev4 and Recent4
    else:
        if recent_months > 0 and pd.notna(recent_wmape):
            parts.append(recent_wmape)
            weights.append(0.40)

        if prev_months > 0 and pd.notna(prev_wmape):
            parts.append(prev_wmape)
            weights.append(0.40)

        if recent_months > 0 and pd.notna(recent_bias):
            parts.append(abs(recent_bias))
            weights.append(0.20)

    if len(parts) == 0:
        return np.nan

    weights = np.array(weights, dtype=float)
    weights = weights / weights.sum()

    return float(np.sum(np.array(parts) * weights))

medium_sku_model_summary["Champion_Score"] = medium_sku_model_summary.apply(
    compute_medium_champion_score,
    axis=1
)

print("\nMEDIUM SKU MODEL SUMMARY")
print(medium_sku_model_summary.head())

# ============================================================
# 19) CHAMPION MODEL PER SKU + SUBGROUP
# ============================================================
model_priority_map = {
    "XGBOOST": 1,
    "CATBOOST": 2,
    "RANDOM_FOREST": 3
}

medium_sku_model_summary["Model_Priority"] = (
    medium_sku_model_summary["Model_Name"]
    .map(model_priority_map)
    .fillna(999)
)

champion_medium_by_subgroup_df = (
    medium_sku_model_summary
    .sort_values(
        by=[
            "ItemCode",
            "Medium_Subgroup",
            "Champion_Score",
            "Recent4_WMAPE",
            "Prev4_WMAPE",
            "Prev4_MAE",
            "Model_Priority"
        ],
        ascending=[True, True, True, True, True, True, True]
    )
    .drop_duplicates(subset=["ItemCode", "Medium_Subgroup"], keep="first")
    .reset_index(drop=True)
)

champion_medium_by_subgroup_df = champion_medium_by_subgroup_df.rename(columns={
    "Model_Name": "Best_Model",
    "Champion_Score": "Best_Model_Score",
    "Prev4_WMAPE": "Best_Model_Prev4_WMAPE",
    "Recent4_WMAPE": "Best_Model_Recent4_WMAPE",
    "Prev4_MAE": "Best_Model_Prev4_MAE",
    "Recent4_MAE": "Best_Model_Recent4_MAE",
    "Prev4_Bias": "Best_Model_Prev4_Bias",
    "Recent4_Bias": "Best_Model_Recent4_Bias",
    "Prev4_Months": "Evaluation_Months_Prev4",
    "Recent4_Months": "Evaluation_Months_Recent4"
})

champion_medium_by_subgroup_df["Segment"] = "MEDIUM"

print("\nMEDIUM CHAMPION MAP BEFORE ROUTING")
print(champion_medium_by_subgroup_df.head())

# ============================================================
# 20) ROUTING RULES
# ============================================================
def assign_medium_final_routing(df):
    df = force_itemcode_str(df)
    df = df.copy()

    for c in [
        "Best_Model_Score",
        "Best_Model_Prev4_WMAPE",
        "Best_Model_Recent4_WMAPE"
    ]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df["Use_Fallback"] = np.where(
        (
            (df["Best_Model_Prev4_WMAPE"] > 85) &
            (df["Best_Model_Recent4_WMAPE"] > 85)
        ) |
        (
            df["Best_Model_Score"] > 75
        ),
        1,
        0
    )

    df["Final_Model"] = np.where(
        df["Use_Fallback"] == 1,
        "FALLBACK",
        df["Best_Model"]
    )

    df["Fallback_Type"] = np.where(
        df["Use_Fallback"] == 1,
        "ROLLING3",
        "NONE"
    )

    return df

champion_medium_by_subgroup_df["Unreliable_Flag"] = np.where(
    (
        (champion_medium_by_subgroup_df["Best_Model_Prev4_WMAPE"] > 80) &
        (champion_medium_by_subgroup_df["Best_Model_Recent4_WMAPE"] > 85)
    ) |
    (
        champion_medium_by_subgroup_df["Best_Model_Score"] > 75
    ),
    1, 0
)

champion_medium_by_subgroup_df = assign_medium_final_routing(champion_medium_by_subgroup_df)

print("\n===== MEDIUM ROUTING DIAGNOSTICS =====")
print(champion_medium_by_subgroup_df["Use_Fallback"].value_counts(dropna=False))
print(pd.crosstab(champion_medium_by_subgroup_df["Best_Model"], champion_medium_by_subgroup_df["Final_Model"]))
print(champion_medium_by_subgroup_df[[
    "ItemCode", "Medium_Subgroup", "Best_Model", "Final_Model",
    "Best_Model_Score", "Best_Model_Prev4_WMAPE",
    "Best_Model_Recent4_WMAPE", "Unreliable_Flag"
]].head(30))

# ============================================================
# 21) COVERAGE CHECK
# ============================================================
print("\n===== MEDIUM COVERAGE BY SKU =====")
print("Unique SKUs in PREV4 summary:", medium_sku_model_prev4_summary["ItemCode"].nunique())
print("Unique SKUs in RECENT4 summary:", medium_sku_model_recent4_summary["ItemCode"].nunique())
print("Unique SKUs in merged medium_sku_model_summary:", medium_sku_model_summary["ItemCode"].nunique())

all_medium_skus = set(
    add_history_length_from_subset(Model_Data_All.copy(), Model_Data_All.copy())
    .query("History_Segment == 'MEDIUM'")["ItemCode"]
    .astype(str)
    .str.strip()
    .str.replace(".0", "", regex=False)
    .unique()
)

covered_medium_skus = set(
    medium_sku_model_summary["ItemCode"]
    .astype(str)
    .str.strip()
    .str.replace(".0", "", regex=False)
    .unique()
)

missing_medium_skus = sorted(all_medium_skus - covered_medium_skus)
print("\n[MISSING MEDIUM SKUs → SUBGROUP DEFAULT MODEL]")
print("Count:", len(missing_medium_skus))

extra_medium_skus = sorted(covered_medium_skus - all_medium_skus)

print("Actual MEDIUM SKUs in data:", len(all_medium_skus))
print("Covered MEDIUM SKUs in comparison:", len(covered_medium_skus))
print("Missing MEDIUM SKUs from comparison:", len(missing_medium_skus))
print("Sample missing MEDIUM SKUs:", missing_medium_skus[:20])
print("Extra count:", len(extra_medium_skus))
print("Extra sample:", extra_medium_skus[:20])


# ============================================================
# 23) OVERALL MODEL SUMMARY
# ============================================================
overall_medium_prev4_summary = (
    medium_prev4_eval_df
    .groupby(["Medium_Subgroup", "Model_Name"], as_index=False)
    .agg(
        Prev4_Actual_Sum=("Actual", "sum"),
        Prev4_Pred_Sum=("Pred", "sum"),
        Prev4_Total_Abs_Error=("Abs_Error", "sum"),
        Prev4_MAE=("Abs_Error", "mean")
    )
)
overall_medium_prev4_summary["Prev4_WMAPE"] = np.where(
    overall_medium_prev4_summary["Prev4_Actual_Sum"] > 0,
    overall_medium_prev4_summary["Prev4_Total_Abs_Error"] /
    overall_medium_prev4_summary["Prev4_Actual_Sum"] * 100,
    np.nan
)
overall_medium_prev4_summary["Prev4_Bias"] = np.where(
    overall_medium_prev4_summary["Prev4_Actual_Sum"] > 0,
    (overall_medium_prev4_summary["Prev4_Pred_Sum"] -
     overall_medium_prev4_summary["Prev4_Actual_Sum"]) /
    overall_medium_prev4_summary["Prev4_Actual_Sum"] * 100,
    np.nan
)
overall_medium_recent4_summary = (
    medium_recent4_eval_df
    .groupby(["Medium_Subgroup", "Model_Name"], as_index=False)
    .agg(
        Recent4_Actual_Sum=("Actual", "sum"),
        Recent4_Pred_Sum=("Pred", "sum"),
        Recent4_Total_Abs_Error=("Abs_Error", "sum"),
        Recent4_MAE=("Abs_Error", "mean")
    )
)
overall_medium_recent4_summary["Recent4_WMAPE"] = np.where(
    overall_medium_recent4_summary["Recent4_Actual_Sum"] > 0,
    overall_medium_recent4_summary["Recent4_Total_Abs_Error"] /
    overall_medium_recent4_summary["Recent4_Actual_Sum"] * 100,
    np.nan
)
overall_medium_recent4_summary["Recent4_Bias"] = np.where(
    overall_medium_recent4_summary["Recent4_Actual_Sum"] > 0,
    (overall_medium_recent4_summary["Recent4_Pred_Sum"] -
     overall_medium_recent4_summary["Recent4_Actual_Sum"]) /
    overall_medium_recent4_summary["Recent4_Actual_Sum"] * 100,
    np.nan
)
overall_medium_model_summary = (
    overall_medium_prev4_summary
    .merge(overall_medium_recent4_summary, on=["Medium_Subgroup", "Model_Name"], how="outer")
)
overall_medium_model_summary["Champion_Score"] = (
    0.40 * overall_medium_model_summary["Prev4_WMAPE"] +
    0.60 * overall_medium_model_summary["Recent4_WMAPE"]
)

print("\nOVERALL MEDIUM MODEL SUMMARY")
print(overall_medium_model_summary)

print("\n===== MEDIUM COVERAGE CHECK =====")
print(
    medium_sku_model_summary.groupby("Model_Name").agg(
        SKU_Count=("ItemCode", "nunique"),
        Avg_Prev4_Months=("Prev4_Months", "mean"),
        Avg_Recent4_Months=("Recent4_Months", "mean")
    )
)


# ============================================================
# 24) SUBGROUP DEFAULT MODEL FOR MISSING MEDIUM SKUs
# ============================================================
overall_medium_model_summary["Champion_Score"] = (
    0.40 * overall_medium_model_summary["Prev4_WMAPE"].fillna(999) +
    0.60 * overall_medium_model_summary["Recent4_WMAPE"].fillna(999)
)

subgroup_default_model_df = (
    overall_medium_model_summary
    .sort_values(["Medium_Subgroup", "Champion_Score"])
    .drop_duplicates("Medium_Subgroup", keep="first")
    [["Medium_Subgroup", "Model_Name", "Champion_Score"]]
    .rename(columns={
        "Model_Name": "Default_Model",
        "Champion_Score": "Default_Model_Score"
    })
)

print("\n===== MEDIUM SUBGROUP DEFAULT MODELS =====")
print(subgroup_default_model_df)

Data_medium = add_history_length_from_subset(Model_Data_All.copy(), Model_Data_All.copy())
Data_medium = Data_medium[Data_medium["History_Segment"] == "MEDIUM"].copy()
if "Behavior_Type" not in Data_medium.columns:
    Data_medium["Behavior_Type"] = "UNKNOWN"
missing_medium_profile_df = (
    Data_medium.copy()
    .sort_values(["ItemCode", "Year", "Month_Number"])
    .groupby("ItemCode")
    .tail(1)[["ItemCode", "Behavior_Type"]]
)
missing_medium_profile_df["ItemCode"] = (
    missing_medium_profile_df["ItemCode"]
    .astype(str)
    .str.replace(".0", "", regex=False)
)

fallback_rows = pd.DataFrame({
    "ItemCode": missing_medium_skus,
    "Medium_Subgroup": "STABLE",
    "Segment": "MEDIUM"
})
fallback_rows["ItemCode"] = fallback_rows["ItemCode"].astype(str)
fallback_rows = fallback_rows.merge(
    subgroup_default_model_df,
    on="Medium_Subgroup",
    how="left"
)
fallback_rows = fallback_rows.merge(
    missing_medium_profile_df,
    on="ItemCode",
    how="left"
)
fallback_rows["Behavior_Type"] = fallback_rows["Behavior_Type"].fillna("UNKNOWN")
fallback_rows["Best_Model"] = fallback_rows["Default_Model"].fillna("CATBOOST")
fallback_rows["Final_Model"] = fallback_rows["Best_Model"]
fallback_rows["Fallback_Type"] = "SUBGROUP_DEFAULT_MODEL"
fallback_rows["Use_Fallback"] = 0
fallback_rows["Unreliable_Flag"] = 0
fallback_rows["Best_Model_Score"] = fallback_rows["Default_Model_Score"]
fallback_rows["Best_Model_Prev4_WMAPE"] = np.nan
fallback_rows["Best_Model_Recent4_WMAPE"] = np.nan
fallback_rows["Best_Model_Prev4_MAE"] = np.nan
fallback_rows["Best_Model_Recent4_MAE"] = np.nan
fallback_rows["Best_Model_Prev4_Bias"] = np.nan
fallback_rows["Best_Model_Recent4_Bias"] = np.nan
fallback_rows["Evaluation_Months_Prev4"] = 0
fallback_rows["Evaluation_Months_Recent4"] = 0
fallback_rows["Prev4_Actual_Sum"] = 0.0
fallback_rows["Prev4_Pred_Sum"] = 0.0
fallback_rows["Recent4_Actual_Sum"] = 0.0
fallback_rows["Recent4_Pred_Sum"] = 0.0

fallback_rows = fallback_rows.drop(
    columns=["Default_Model", "Default_Model_Score"],
    errors="ignore"
)


# ============================================================
# 25) FINAL CHAMPION MAP (ONE ROW PER SKU)
# ============================================================
champion_medium_map_df = champion_medium_by_subgroup_df.copy()
champion_medium_map_df["ItemCode"] = champion_medium_map_df["ItemCode"].astype(str)
champion_medium_map_df["Medium_Subgroup"] = champion_medium_map_df["Medium_Subgroup"].astype(str).str.upper()

print("\n===== MEDIUM SUBGROUP CONSISTENCY CHECK =====")
sku_subgroup_counts = champion_medium_by_subgroup_df.groupby("ItemCode")["Medium_Subgroup"].nunique()
print("SKUs appearing in multiple subgroups:", (sku_subgroup_counts > 1).sum())
print(sku_subgroup_counts[sku_subgroup_counts > 1].head(20))

champion_medium_map_df = champion_medium_map_df[
    [
        "ItemCode",
        "Medium_Subgroup",
        "Segment",
        "Best_Model",
        "Final_Model",
        "Fallback_Type",
        "Use_Fallback",
        "Unreliable_Flag",
        "Best_Model_Score",
        "Best_Model_Prev4_WMAPE",
        "Best_Model_Recent4_WMAPE",
        "Best_Model_Prev4_MAE",
        "Best_Model_Recent4_MAE",
        "Best_Model_Prev4_Bias",
        "Best_Model_Recent4_Bias",
        "Evaluation_Months_Prev4",
        "Evaluation_Months_Recent4",
        "Prev4_Actual_Sum",
        "Prev4_Pred_Sum",
        "Recent4_Actual_Sum",
        "Recent4_Pred_Sum"
    ]
].copy()

champion_medium_map_df = (
    champion_medium_map_df
    .sort_values(["ItemCode", "Best_Model_Score"])
    .drop_duplicates(subset=["ItemCode"], keep="first")
    .reset_index(drop=True)
)


# ============================================================
# 26) ADD BEHAVIOR_TYPE TO MEDIUM CHAMPION MAP
# ============================================================
medium_behavior_lookup_df = (
    Data_medium.copy()
    .sort_values(["ItemCode", "Year", "Month_Number"])
    .groupby("ItemCode")
    .tail(1)[["ItemCode", "Behavior_Type"]]
)

medium_behavior_lookup_df["ItemCode"] = (
    medium_behavior_lookup_df["ItemCode"]
    .astype(str)
    .str.replace(".0", "", regex=False)
)

champion_medium_map_df["ItemCode"] = (
    champion_medium_map_df["ItemCode"]
    .astype(str)
    .str.replace(".0", "", regex=False)
)

champion_medium_map_df = champion_medium_map_df.drop(
    columns=["Behavior_Type"],
    errors="ignore"
)

champion_medium_map_df = champion_medium_map_df.merge(
    medium_behavior_lookup_df,
    on="ItemCode",
    how="left"
)

champion_medium_map_df["Behavior_Type"] = (
    champion_medium_map_df["Behavior_Type"]
    .fillna("UNKNOWN")
)


champion_medium_map_df = pd.concat(
    [champion_medium_map_df, fallback_rows],
    ignore_index=True
)

print("\nFINAL MEDIUM CHAMPION MAP (ONE ROW PER SKU)")
print(champion_medium_map_df.head())
print("Unique medium SKUs in final champion map:", champion_medium_map_df["ItemCode"].nunique())
print("Duplicate ItemCodes in champion_medium_map_df:", champion_medium_map_df["ItemCode"].duplicated().sum())
print("Null Medium_Subgroup rows:", champion_medium_map_df["Medium_Subgroup"].isna().sum())
print("Null Best_Model rows:", champion_medium_map_df["Best_Model"].isna().sum())


# ============================================================
# 27) WIN COUNTS
# ============================================================
medium_model_win_counts = (
    champion_medium_map_df
    .groupby(["Medium_Subgroup", "Final_Model"], as_index=False)
    .size()
    .rename(columns={"size": "SKU_Count"})
)

print("\nMEDIUM MODEL WIN COUNTS")
print(medium_model_win_counts)

medium_best_model_win_counts = (
    champion_medium_map_df
    .groupby(["Medium_Subgroup", "Best_Model"], as_index=False)
    .size()
    .rename(columns={"size": "SKU_Count"})
)

print("\nMEDIUM RAW BEST MODEL WIN COUNTS")
print(medium_best_model_win_counts)


# ============================================================
# 28) MERGE CHAMPION BACK TO PREV4 ROW LEVEL
# ============================================================
medium_prev4_eval_df = force_itemcode_str(medium_prev4_eval_df)
champion_medium_map_df = force_itemcode_str(champion_medium_map_df)

medium_model_compare_with_champion = medium_prev4_eval_df.merge(
    champion_medium_map_df[
        [
            "ItemCode",
            "Medium_Subgroup",
            "Best_Model",
            "Final_Model",
            "Fallback_Type",
            "Use_Fallback",
            "Unreliable_Flag",
            "Best_Model_Score"
        ]
    ],
    on=["ItemCode", "Medium_Subgroup"],
    how="left"
)

medium_model_compare_with_champion["Is_Champion_Model"] = (
    medium_model_compare_with_champion["Model_Name"] == medium_model_compare_with_champion["Best_Model"]
).astype(int)


# ============================================================
# 29) SAVE CHAMPION MAP
# ============================================================
joblib.dump(champion_medium_map_df, "champion_medium_map_df.pkl")
print("Saved: champion_medium_map_df.pkl")

print(champion_medium_map_df.shape)
print(champion_medium_map_df["Medium_Subgroup"].value_counts(dropna=False))
print(champion_medium_map_df["Best_Model"].value_counts(dropna=False))
print(champion_medium_map_df["Final_Model"].value_counts(dropna=False))

print(
    medium_sku_model_summary.groupby("Model_Name").agg(
        SKU_Count=("ItemCode", "nunique"),
        Avg_Prev4_Months=("Prev4_Months", "mean"),
        Avg_Recent4_Months=("Recent4_Months", "mean")
    )
)
print("\n===== MEDIUM BEHAVIOR DIAGNOSTICS =====")

print("\nBehavior Type Counts:")
print(champion_medium_map_df["Behavior_Type"].value_counts(dropna=False))

print("\nBehavior Type by Subgroup:")
print(pd.crosstab(
    champion_medium_map_df["Behavior_Type"],
    champion_medium_map_df["Medium_Subgroup"]
))

print("\nFinal Model by Behavior Type:")
print(pd.crosstab(
    champion_medium_map_df["Behavior_Type"],
    champion_medium_map_df["Final_Model"]
))

print("\nFallback by Behavior Type:")
print(pd.crosstab(
    champion_medium_map_df["Behavior_Type"],
    champion_medium_map_df["Use_Fallback"]
))

seg_check = add_history_length_from_subset(Data.copy(), Data.copy())
seg_check = seg_check[["ItemCode", "History_Length", "History_Segment"]].drop_duplicates()

bad_medium = seg_check[
    (seg_check["History_Segment"] == "MEDIUM") &
    (seg_check["History_Length"] < 10)
]

print("Bad MEDIUM SKUs below 10 months:", len(bad_medium))
assert bad_medium.empty, "BUG: MEDIUM contains SKUs with <10 months history"


========== XGBOOST MEDIUM → PREV4 ==========
  Medium_Subgroup     Actual_Sum       Pred_Sum  Total_Abs_Error          MAE  \
1          STABLE    9204.000000    8466.386232      1771.568477    73.815353   
0     PROMO_HEAVY  130852.933333  159707.134349     50385.727582  1799.490271   

       WMAPE       Bias  
1  19.247810  -8.014057  
0  38.505616  22.050863  
Overall WMAPE: 37.24006717692897
{'WMAPE': 37.24006717692897, 'Bias': 20.075112725681645, 'MAE': 1003.0249242169223, 'RMSE': 2627.449334548757, 'Underforecast_Rate': 8.582477225623661}

========== CATBOOST MEDIUM → PREV4 ==========
  Medium_Subgroup     Actual_Sum       Pred_Sum  Total_Abs_Error          MAE  \
1          STABLE    9204.000000    8450.570146      1681.914804    70.079784   
0     PROMO_HEAVY  130852.933333  160909.140987     60472.568459  2159.734588   

       WMAPE       Bias  
1  18.273738  -8.185896  
0  46.214148  22.969457  
Overall WMAPE: 44.37801241545359
{'WMAPE': 44.37801241545359, 'Bias': 20.92204

#### Deployment

In [37]:
# ============================================================
# 27) DEPLOYMENT REGISTRY
# ============================================================
medium_deployment_artifact_registry = pd.DataFrame([
    {"Medium_Subgroup": "PROMO_HEAVY", "Model_Name": "XGBOOST",       "Run_Key": "PROMO_HEAVY_XGBOOST",  "Deploy_Artifact_Key": "deploy_artifacts"},
    {"Medium_Subgroup": "PROMO_HEAVY", "Model_Name": "CATBOOST",      "Run_Key": "PROMO_HEAVY_CATBOOST", "Deploy_Artifact_Key": "deploy_artifacts"},
    {"Medium_Subgroup": "STABLE",      "Model_Name": "XGBOOST",       "Run_Key": "STABLE_XGBOOST",       "Deploy_Artifact_Key": "deploy_artifacts"},
    {"Medium_Subgroup": "STABLE",      "Model_Name": "CATBOOST",      "Run_Key": "STABLE_CATBOOST",      "Deploy_Artifact_Key": "deploy_artifacts"},
    {"Medium_Subgroup": "STABLE",      "Model_Name": "RANDOM_FOREST", "Run_Key": "STABLE_RANDOM_FOREST", "Deploy_Artifact_Key": "deploy_artifacts"},
])

medium_deployment_artifact_registry["Medium_Subgroup"] = (
    medium_deployment_artifact_registry["Medium_Subgroup"].astype(str).str.upper()
)
medium_deployment_artifact_registry["Model_Name"] = (
    medium_deployment_artifact_registry["Model_Name"].astype(str).str.upper()
)

print("\nMEDIUM DEPLOYMENT ARTIFACT REGISTRY")
print(medium_deployment_artifact_registry)

medium_deploy_model_registry = {
    ("PROMO_HEAVY", "XGBOOST"): medium_runs["PROMO_HEAVY_XGBOOST"]["deploy_artifacts"],
    ("PROMO_HEAVY", "CATBOOST"): medium_runs["PROMO_HEAVY_CATBOOST"]["deploy_artifacts"],
    ("STABLE", "XGBOOST"): medium_runs["STABLE_XGBOOST"]["deploy_artifacts"],
    ("STABLE", "CATBOOST"): medium_runs["STABLE_CATBOOST"]["deploy_artifacts"],
    ("STABLE", "RANDOM_FOREST"): medium_runs["STABLE_RANDOM_FOREST"]["deploy_artifacts"],
}

# ============================================================
# 28) WINDOW SANITY CHECK
# ============================================================
print("\n===== MEDIUM WINDOW SANITY CHECK =====")
print("MEDIUM_TUNE_WINDOWS:", MEDIUM_TUNE_WINDOWS)
print("MEDIUM_PREV4_WINDOW:", MEDIUM_PREV4_WINDOW)
print("MEDIUM_RECENT4_WINDOW:", MEDIUM_RECENT4_WINDOW)

assert MEDIUM_PREV4_WINDOW["valid_end_idx"] < MEDIUM_RECENT4_WINDOW["valid_start_idx"], "MEDIUM PREV4 overlaps RECENT4"
print("MEDIUM window logic OK")

# ============================================================
# 29) SAVE EXCEL
# ============================================================
with pd.ExcelWriter("medium_model_comparison_and_champion_map.xlsx", engine="openpyxl") as writer:
    medium_prev4_eval_df.to_excel(writer, sheet_name="Prev4_RowLevel", index=False)
    medium_recent4_eval_df.to_excel(writer, sheet_name="Recent4_RowLevel", index=False)
    medium_sku_model_summary.to_excel(writer, sheet_name="SKU_Model_Summary", index=False)
    champion_medium_by_subgroup_df.to_excel(writer, sheet_name="Champion_By_Subgroup", index=False)
    champion_medium_map_df.to_excel(writer, sheet_name="Champion_Map_Final", index=False)
    medium_model_win_counts.to_excel(writer, sheet_name="Final_Model_Win_Counts", index=False)
    medium_best_model_win_counts.to_excel(writer, sheet_name="Best_Model_Win_Counts", index=False)
    overall_medium_model_summary.to_excel(writer, sheet_name="Overall_Model_Summary", index=False)
    medium_model_compare_with_champion.to_excel(writer, sheet_name="Eval_With_Routing", index=False)
    medium_deployment_artifact_registry.to_excel(writer, sheet_name="Deployment_Artifact_Registry", index=False)
    medium_prev4_model_table.to_excel(writer, sheet_name="Medium_Prev4_ModelPerf", index=False)
    medium_recent4_model_table.to_excel(writer, sheet_name="Medium_Recent4_ModelPerf", index=False)

print("\nSaved: medium_model_comparison_and_champion_map.xlsx")
print("Unique ItemCodes:", champion_medium_map_df["ItemCode"].nunique())
print("Total rows:", len(champion_medium_map_df))
print("Duplicate ItemCodes:", champion_medium_map_df["ItemCode"].duplicated().sum())


MEDIUM DEPLOYMENT ARTIFACT REGISTRY
  Medium_Subgroup     Model_Name               Run_Key Deploy_Artifact_Key
0     PROMO_HEAVY        XGBOOST   PROMO_HEAVY_XGBOOST    deploy_artifacts
1     PROMO_HEAVY       CATBOOST  PROMO_HEAVY_CATBOOST    deploy_artifacts
2          STABLE        XGBOOST        STABLE_XGBOOST    deploy_artifacts
3          STABLE       CATBOOST       STABLE_CATBOOST    deploy_artifacts
4          STABLE  RANDOM_FOREST  STABLE_RANDOM_FOREST    deploy_artifacts

===== MEDIUM WINDOW SANITY CHECK =====
MEDIUM_TUNE_WINDOWS: [{'valid_start_idx': 24300, 'valid_end_idx': 24303, 'valid_months': 4}, {'valid_start_idx': 24304, 'valid_end_idx': 24307, 'valid_months': 4}]
MEDIUM_PREV4_WINDOW: {'valid_start_idx': 24304, 'valid_end_idx': 24307, 'valid_months': 4, 'window_name': 'PREV4'}
MEDIUM_RECENT4_WINDOW: {'valid_start_idx': 24308, 'valid_end_idx': 24311, 'valid_months': 4, 'window_name': 'RECENT4'}
MEDIUM window logic OK

Saved: medium_model_comparison_and_champion_map.xls

### SHORT SEGMENT

In [38]:
# ============================================================
# 0) SHORT FEATURE LIST
# ============================================================
SHORT_FEATURE_COLS = [
    c for c in [
        "Lag1", "Lag2", "Rolling3M_Mean",
        "SKU_Mean_Demand", "Avg_Bonus_Uplift",
        "Bonus_Flag", "Expected_Bonus_Month", 
        "Supply_Constraint_Flag",
        "Available_Primary_Inventory_Qty",
        "Distributor_Inventory_Qty",
        "Short_History_Length",
        "Short_Mean_Demand",
        "Short_ZeroRate",
        "Short_Bonus_Frequency",
        "Short_Bonus_Demand_Share",
        "Last_Bonus_Demand",
        "Demand_State_Encoded",
        "Short_Supply_Rate",
    ]
    if c in Model_Data_All.columns or c.startswith("Short_")
]

# ============================================================
# 1) SHORT PROFILE BUILDING
# ============================================================
def build_short_sku_profile(train_df):
    train_df = force_itemcode_str(train_df)
    train_df = train_df.copy().sort_values(["ItemCode", "Year", "Month_Number"])
    out = []

    for item, g in train_df.groupby("ItemCode"):
        g = g.copy()

        hist_len = g[["Year", "Month_Number"]].drop_duplicates().shape[0]
        mean_demand = float(g["Clean_Demand"].mean()) if hist_len > 0 else 0.0
        zero_rate = float((g["Clean_Demand"] == 0).mean()) if hist_len > 0 else 0.0
        bonus_freq = float(g["Bonus_Flag"].mean()) if "Bonus_Flag" in g.columns else 0.0
        supply_rate = float(g["Supply_Constraint_Flag"].mean()) if "Supply_Constraint_Flag" in g.columns else 0.0

        total_demand = float(g["Clean_Demand"].sum())
        bonus_demand = float(g.loc[g["Bonus_Flag"] == 1, "Clean_Demand"].sum()) if total_demand > 0 else 0.0
        bonus_share = 0.0 if total_demand <= 0 else bonus_demand / total_demand

        if bonus_freq >= 0.25 or bonus_share >= 0.40:
            short_type = "SHORT_PROMO"
        else:
            short_type = "SHORT_NORMAL"

        out.append({
            "ItemCode": str(item),
            "Short_SKU_Type": short_type,
            "Short_History_Length": hist_len,
            "Short_Mean_Demand": mean_demand,
            "Short_ZeroRate": zero_rate,
            "Short_Bonus_Frequency": bonus_freq,
            "Short_Bonus_Demand_Share": bonus_share,
            "Short_Supply_Rate": supply_rate
        })

    out_df = pd.DataFrame(out)
    out_df = force_itemcode_str(out_df)
    return out_df

def merge_short_sku_profile(df, profile_df):
    df = force_itemcode_str(df)
    profile_df = force_itemcode_str(profile_df)
    df = df.copy()

    keep_cols = [
        "ItemCode",
        "Short_SKU_Type",
        "Short_History_Length",
        "Short_Mean_Demand",
        "Short_ZeroRate",
        "Short_Bonus_Frequency",
        "Short_Bonus_Demand_Share",
        "Short_Supply_Rate"
    ]

    df = df.drop(columns=[c for c in keep_cols if c != "ItemCode"], errors="ignore")
    df = df.merge(profile_df[keep_cols], on="ItemCode", how="left")
    df = force_itemcode_str(df)

    df["Short_SKU_Type"] = df["Short_SKU_Type"].fillna("SHORT_NORMAL")

    for c in [
        "Short_History_Length",
        "Short_Mean_Demand",
        "Short_ZeroRate",
        "Short_Bonus_Frequency",
        "Short_Bonus_Demand_Share",
        "Short_Supply_Rate"
    ]:
        df[c] = df[c].fillna(0)

    return df

def filter_short_subgroup(df, subgroup_name):
    df = force_itemcode_str(df)
    df = df.copy()

    subgroup_name = str(subgroup_name).upper()

    if subgroup_name == "SHORT_PROMO":
        return df[df["Short_SKU_Type"] == "SHORT_PROMO"].copy()

    if subgroup_name == "SHORT_NORMAL":
        return df[df["Short_SKU_Type"] == "SHORT_NORMAL"].copy()

    raise ValueError(f"Unknown short subgroup: {subgroup_name}")

# ============================================================
# 2) FOLD-SAFE SHORT PREP
# ============================================================
def prepare_short_frame_foldsafe(train_df, valid_df):
    train_df = force_itemcode_str(train_df.copy())
    valid_df = force_itemcode_str(valid_df.copy())

    train_df = train_df[train_df["History_Segment"] == "SHORT"].copy()
    valid_df = valid_df[valid_df["History_Segment"] == "SHORT"].copy()

    if train_df.empty or valid_df.empty:
        return None, None, None

    train_df, valid_df, abc_map = apply_fold_adjustments(train_df, valid_df)

    short_profile_df = build_short_sku_profile(train_df)
    train_df = merge_short_sku_profile(train_df, short_profile_df)
    valid_df = merge_short_sku_profile(valid_df, short_profile_df)

    clip_cols = [
        "Inventory_Pressure",
        "Stock_Cover_Months",
        "Primary_Stock_Cover",
        "Distributor_Stock_Cover",
        "Free_Ratio",
        "SKU_CV"
    ]

    clip_cols = [c for c in clip_cols if c in train_df.columns]

    caps = compute_clip_caps(train_df, clip_cols, q=0.99)
    train_df = apply_clip_caps(train_df, caps)
    valid_df = apply_clip_caps(valid_df, caps)

    artifacts = {
        "abc_map": abc_map,
        "clip_caps": caps,
        "short_profile_df": short_profile_df
    }

    return train_df, valid_df, artifacts

# ============================================================
# 3) SHORT RULE MODEL
# ============================================================
def short_rule_predict(row):
    lag1 = float(row.get("Lag1", 0) or 0)
    lag2 = float(row.get("Lag2", 0) or 0)
    rolling3 = float(row.get("Rolling3M_Mean", 0) or 0)
    sku_mean = float(row.get("SKU_Mean_Demand", 0) or 0)

    last_bonus_demand = float(row.get("Last_Bonus_Demand", 0) or 0)
    avg_bonus_uplift = float(row.get("Avg_Bonus_Uplift", 1.0) or 1.0)

    expected_bonus = int(row.get("Expected_Bonus_Month", 0) or 0)
    supply_flag = int(row.get("Supply_Constraint_Flag", 0) or 0)

    primary_stock = float(row.get("Available_Primary_Inventory_Qty", 0) or 0)
    distributor_stock = float(row.get("Distributor_Inventory_Qty", 0) or 0)

    short_type = row.get("Short_SKU_Type", "SHORT_NORMAL")
    hist_len = int(row.get("Short_History_Length", 0) or 0)

    anchors = [x for x in [lag1, lag2, rolling3, sku_mean] if x > 0]

    if len(anchors) == 0:
        pred = 0.0
    elif hist_len <= 2:
        pred = float(np.mean(anchors))
    else:
        pred = max(
            0.50 * float(np.mean(anchors)) + 0.50 * float(np.max(anchors)),
            0.75 * sku_mean if sku_mean > 0 else 0.0
        )

    if (short_type == "SHORT_PROMO" and expected_bonus == 1 and avg_bonus_uplift >= 1.15):
        promo_anchor = max(pred, last_bonus_demand, rolling3 * avg_bonus_uplift)
        pred = 0.50 * pred + 0.50 * promo_anchor
        pred = pred * min(max(avg_bonus_uplift, 1.0), 1.5)

    if supply_flag == 1:
        safe_cap = max(lag1, rolling3, sku_mean, 0)
        pred = min(pred, 1.10 * safe_cap)

    if (primary_stock + distributor_stock) <= 0:
        pred *= 0.90

    return max(pred, 0.0)

# ============================================================
# 4) SHORT HOLDOUT EVALUATION
# ============================================================
def evaluate_short_rule_model(full_data, subgroup_name=None):
    full_data = add_period_index(full_data)

    train_df = full_data[full_data["Period_Index"] < TIME_WINDOWS["test_start_idx"]].copy()
    test_df = full_data[
        full_data["Period_Index"].between(TIME_WINDOWS["test_start_idx"], TIME_WINDOWS["latest_idx"])
    ].copy()

    if train_df.empty or test_df.empty:
        raise ValueError("Need both pre-holdout train data and holdout test data.")

    train_df = add_history_length_from_subset(train_df, train_df)
    test_df = add_history_length_from_subset(train_df, test_df)

    train_df = train_df[train_df["History_Segment"] == "SHORT"].copy()
    test_df = test_df[test_df["History_Segment"] == "SHORT"].copy()

    if train_df.empty or test_df.empty:
        raise ValueError("No SHORT rows available.")

    train_df, test_df, prep_artifacts = prepare_short_frame_foldsafe(train_df, test_df)
    if train_df is None or test_df is None:
        raise ValueError("No usable SHORT rows after fold-safe prep.")

    if subgroup_name is not None:
        subgroup_name = str(subgroup_name).upper()
        train_df = filter_short_subgroup(train_df, subgroup_name)
        test_df = filter_short_subgroup(test_df, subgroup_name)

    if train_df.empty or test_df.empty:
        raise ValueError(f"No usable SHORT rows for subgroup: {subgroup_name}")

    caps = compute_clip_caps(train_df, cols=["Inventory_Pressure", "Stock_Cover_Months"], q=0.99)
    train_df = apply_clip_caps(train_df, caps)
    test_df = apply_clip_caps(test_df, caps)

    test_df = test_df.dropna(subset=[ACTUAL_TARGET_COL]).copy()
    test_df = force_itemcode_str(test_df)

    if test_df.empty:
        raise ValueError("No usable SHORT rows after target creation.")

    test_df["ItemCode_Original"] = test_df["ItemCode"].astype(str)
    
    test_df = test_df.drop(columns=["Pred"], errors="ignore")
    test_df["Pred"] = test_df.apply(short_rule_predict, axis=1)

    metrics = evaluate_all_metrics(test_df[ACTUAL_TARGET_COL].values, test_df["Pred"].values)

    artifacts = {
        "model": None,
        "feature_cols": SHORT_FEATURE_COLS,
        "best_params": None,
        "itemcode_categories": None,
        "abc_map": prep_artifacts["abc_map"],
        "clip_caps": caps,
        "promo_profile_df": None,
        "short_profile_df": prep_artifacts["short_profile_df"],
        "sku_profile_df": None,
        "target_mode": "rule_based",
        "baseline_col": None,
        "actual_target_col": ACTUAL_TARGET_COL,
        "model_target_col": None,
        "segment": "SHORT",
        "subgroup_name": subgroup_name if subgroup_name is not None else "ALL_SHORT",
        "rule_name": "short_rule_v2"
    }

    return artifacts, test_df, metrics

# ============================================================
# 5) SHORT DEPLOYMENT PREP
# ============================================================
def prepare_short_rule_deployment(full_data, subgroup_name=None):
    deploy_df = force_itemcode_str(full_data.copy())
    deploy_df = deploy_df[deploy_df["History_Segment"] == "SHORT"].copy()

    if deploy_df.empty:
        return {
            "model": None,
            "feature_cols": SHORT_FEATURE_COLS,
            "abc_map": {},
            "clip_caps": {},
            "short_profile_df": pd.DataFrame(),
            "target_mode": "rule_based",
            "segment": "SHORT",
            "subgroup_name": subgroup_name if subgroup_name else "ALL_SHORT",
            "rule_name": "short_rule_v2"
        }, pd.DataFrame()

    train_df, deploy_df, abc_map = apply_fold_adjustments(
        deploy_df.copy(),
        deploy_df.copy()
    )

    short_profile_df = build_short_sku_profile(deploy_df)
    deploy_df = merge_short_sku_profile(deploy_df, short_profile_df)

    if subgroup_name is not None:
        deploy_df = filter_short_subgroup(deploy_df, subgroup_name)

    clip_cols = [
        "Inventory_Pressure",
        "Stock_Cover_Months",
        "Primary_Stock_Cover",
        "Distributor_Stock_Cover",
        "Free_Ratio",
        "SKU_CV"
    ]
    clip_cols = [c for c in clip_cols if c in deploy_df.columns]

    caps = compute_clip_caps(deploy_df, clip_cols, q=0.99)
    deploy_df = apply_clip_caps(deploy_df, caps)

    artifacts = {
        "model": None,
        "feature_cols": SHORT_FEATURE_COLS,
        "best_params": None,
        "itemcode_categories": None,
        "abc_map": abc_map,
        "clip_caps": caps,
        "short_profile_df": short_profile_df,
        "target_mode": "rule_based",
        "baseline_col": None,
        "actual_target_col": ACTUAL_TARGET_COL,
        "model_target_col": None,
        "segment": "SHORT",
        "subgroup_name": subgroup_name if subgroup_name else "ALL_SHORT",
        "rule_name": "short_rule_v2"
    }

    # Prevent old Pred column from interfering
    deploy_df = deploy_df.drop(columns=["Pred"], errors="ignore")

    if deploy_df.empty:
        deploy_df["Pred"] = pd.Series(dtype=float)
    else:
        pred_values = deploy_df.apply(short_rule_predict, axis=1)
        # Safety check: short_rule_predict must return one numeric value per row
        if isinstance(pred_values, pd.DataFrame):
            print("WARNING: short_rule_predict returned DataFrame:")
            print(pred_values.head())
            pred_values = pred_values.iloc[:, 0]
        deploy_df["Pred"] = pd.to_numeric(pred_values, errors="coerce").fillna(0).clip(lower=0)

    return artifacts, deploy_df

# ============================================================
# 6) RUN SHORT HOLDOUT EVALUATION
# ============================================================
short_eval_artifacts, short_holdout_test_df, short_eval_metrics = evaluate_short_rule_model(
    full_data=Data,
    subgroup_name=None
)
print("\n===== SHORT ALL RULE EVALUATION METRICS =====")
print(short_eval_metrics)

try:
    short_promo_eval_artifacts, short_promo_holdout_test_df, short_promo_eval_metrics = evaluate_short_rule_model(
        full_data=Data,
        subgroup_name="SHORT_PROMO"
    )
except ValueError as e:
    print("SHORT_PROMO skipped:", e)
    short_promo_eval_artifacts = None
    short_promo_holdout_test_df = pd.DataFrame()
    short_promo_eval_metrics = None
print("\n===== SHORT_PROMO RULE EVALUATION METRICS =====")
print(short_promo_eval_metrics)

try:
    short_normal_eval_artifacts, short_normal_holdout_test_df, short_normal_eval_metrics = evaluate_short_rule_model(
        full_data=Data,
        subgroup_name="SHORT_NORMAL"
    )
except ValueError as e:
    print("SHORT_NORMAL skipped:", e)
    short_normal_eval_artifacts = None
    short_normal_holdout_test_df = pd.DataFrame()
    short_normal_eval_metrics = None
print("\n===== SHORT_NORMAL RULE EVALUATION METRICS =====")
print(short_normal_eval_metrics)

def standardize_short_rule_output(df, subgroup_label="ALL_SHORT"):
    out = df.copy()
    out = force_itemcode_str(out)

    out["ItemCode_Original"] = out["ItemCode"].astype(str)
    out["ItemCode"] = out["ItemCode_Original"]

    out["Actual"] = pd.to_numeric(out[ACTUAL_TARGET_COL], errors="coerce")
    out["Pred"] = pd.to_numeric(out["Pred"], errors="coerce").clip(lower=0)
    out["Error"] = out["Actual"] - out["Pred"]
    out["Abs_Error"] = np.abs(out["Error"])

    out["Segment"] = "SHORT"
    out["Model_Name"] = "RULE_BASED"
    out["Short_Subgroup"] = subgroup_label

    return out

short_holdout_test_std = standardize_short_rule_output(
    short_holdout_test_df,
    subgroup_label="ALL_SHORT"
)

short_promo_holdout_test_std = (
    standardize_short_rule_output(short_promo_holdout_test_df, "SHORT_PROMO")
    if not short_promo_holdout_test_df.empty
    else pd.DataFrame()
)

short_normal_holdout_test_std = (
    standardize_short_rule_output(short_normal_holdout_test_df, "SHORT_NORMAL")
    if not short_normal_holdout_test_df.empty
    else pd.DataFrame()
)

# ============================================================
# 7) SHORT MODEL PERFORMANCE REPORTS
# ============================================================
print_model_eval_report(
    short_holdout_test_std,
    title="RULE_BASED SHORT → HOLDOUT",
    group_cols=["Short_SKU_Type"] if "Short_SKU_Type" in short_holdout_test_std.columns else None
)

print_model_eval_report(
    short_promo_holdout_test_std,
    title="RULE_BASED SHORT_PROMO → HOLDOUT",
    group_cols=["Short_SKU_Type"] if "Short_SKU_Type" in short_promo_holdout_test_std.columns else None
)

print_model_eval_report(
    short_normal_holdout_test_std,
    title="RULE_BASED SHORT_NORMAL → HOLDOUT",
    group_cols=["Short_SKU_Type"] if "Short_SKU_Type" in short_normal_holdout_test_std.columns else None
)

short_holdout_model_table = build_model_summary_table(
    short_holdout_test_std,
    extra_group_cols=["Short_SKU_Type"] if "Short_SKU_Type" in short_holdout_test_std.columns else None
)

print("\n===== SHORT HOLDOUT MODEL TABLE =====")
print(short_holdout_model_table)


# ============================================================
# 8) RUN SHORT DEPLOYMENT PREP
# ============================================================
short_deploy_artifacts, short_deploy_train_df = prepare_short_rule_deployment(
    full_data=Data,
    subgroup_name=None
)

short_promo_deploy_artifacts, short_promo_deploy_df = prepare_short_rule_deployment(
    full_data=Data,
    subgroup_name="SHORT_PROMO"
)

short_normal_deploy_artifacts, short_normal_deploy_df = prepare_short_rule_deployment(
    full_data=Data,
    subgroup_name="SHORT_NORMAL"
)

print("\nSHORT deployment rows:", len(short_deploy_train_df))
print("SHORT_PROMO deployment rows:", len(short_promo_deploy_df))
print("SHORT_NORMAL deployment rows:", len(short_normal_deploy_df))


# ============================================================
# 9) SHORT OPTIONAL ROW-LEVEL TAGS
# ============================================================
short_holdout_test_df = force_itemcode_str(short_holdout_test_df)
short_promo_holdout_test_df = force_itemcode_str(short_promo_holdout_test_df)
short_normal_holdout_test_df = force_itemcode_str(short_normal_holdout_test_df)

if "Short_SKU_Type" in short_holdout_test_df.columns:
    short_holdout_test_df["Short_Subgroup"] = short_holdout_test_df["Short_SKU_Type"].astype(str).str.upper()
else:
    short_holdout_test_df["Short_Subgroup"] = "UNKNOWN"

short_promo_holdout_test_df["Short_Subgroup"] = "SHORT_PROMO"
short_normal_holdout_test_df["Short_Subgroup"] = "SHORT_NORMAL"

print("\nSHORT HOLDOUT TEST ROWS (ALL):", len(short_holdout_test_df))
print("SHORT HOLDOUT TEST ROWS (PROMO):", len(short_promo_holdout_test_df))
print("SHORT HOLDOUT TEST ROWS (NORMAL):", len(short_normal_holdout_test_df))


# ============================================================
# 10) SAVE SHORT ARTIFACTS
# ============================================================
joblib.dump(short_eval_artifacts, "short_rule_eval_artifacts.pkl")
joblib.dump(short_deploy_artifacts, "short_rule_deploy_artifacts.pkl")

joblib.dump(short_promo_eval_artifacts, "short_promo_rule_eval_artifacts.pkl")
joblib.dump(short_promo_deploy_artifacts, "short_promo_rule_deploy_artifacts.pkl")

joblib.dump(short_normal_eval_artifacts, "short_normal_rule_eval_artifacts.pkl")
joblib.dump(short_normal_deploy_artifacts, "short_normal_rule_deploy_artifacts.pkl")

print("\nSaved:")
print(" - short_rule_eval_artifacts.pkl")
print(" - short_rule_deploy_artifacts.pkl")
print(" - short_promo_rule_eval_artifacts.pkl")
print(" - short_promo_rule_deploy_artifacts.pkl")
print(" - short_normal_rule_eval_artifacts.pkl")
print(" - short_normal_rule_deploy_artifacts.pkl")


# ============================================================
# 11) SAVE SHORT REPORT
# ============================================================
with pd.ExcelWriter("short_model_comparison_and_rule_outputs.xlsx", engine="openpyxl") as writer:
    short_holdout_test_std.to_excel(writer, sheet_name="Short_Holdout_All", index=False)
    short_promo_holdout_test_std.to_excel(writer, sheet_name="Short_Holdout_Promo", index=False)
    short_normal_holdout_test_std.to_excel(writer, sheet_name="Short_Holdout_Normal", index=False)
    short_holdout_model_table.to_excel(writer, sheet_name="Short_Holdout_ModelPerf", index=False)

print("\nSaved: short_model_comparison_and_rule_outputs.xlsx")


===== SHORT ALL RULE EVALUATION METRICS =====
{'WMAPE': 31.49262928502297, 'Bias': -12.30897622531032, 'MAE': 940.6998167299308, 'RMSE': 3311.26217285047, 'Underforecast_Rate': 21.900802755166644}

===== SHORT_PROMO RULE EVALUATION METRICS =====
{'WMAPE': 23.55056947695576, 'Bias': -1.9804606214195528, 'MAE': 222.40619515306122, 'RMSE': 431.80727425167714, 'Underforecast_Rate': 12.765515049187659}

===== SHORT_NORMAL RULE EVALUATION METRICS =====
{'WMAPE': 32.34987464458581, 'Bias': -13.423809440695344, 'MAE': 1260.6185029041465, 'RMSE': 3970.496037542947, 'Underforecast_Rate': 22.88684204264058}

========== RULE_BASED SHORT → HOLDOUT ==========
  Short_SKU_Type    Actual_Sum      Pred_Sum  Total_Abs_Error          MAE  \
1    SHORT_PROMO  3.966384e+05  3.887831e+05     9.341060e+04   222.406195   
0   SHORT_NORMAL  3.674707e+06  3.181422e+06     1.188763e+06  1260.618503   

       WMAPE       Bias  
1  23.550569  -1.980461  
0  32.349875 -13.423809  
Overall WMAPE: 31.49262928502297

# 04_INFERENCE

In [ ]:
# ------------------------------------------------------------
# 0) SMALL SHARED HELPERS LONG+MEDIUM+SHORT
# ------------------------------------------------------------
def force_itemcode_str(df):
    df = df.copy()
    if "ItemCode" in df.columns:
        df["ItemCode"] = pd.to_numeric(df["ItemCode"], errors="coerce")
        df = df.dropna(subset=["ItemCode"])
        df["ItemCode"] = df["ItemCode"].astype(int).astype(str)
 
    if "ItemCode_Original" in df.columns:
        df["ItemCode_Original"] = df["ItemCode_Original"].astype(str)
 
    return df
 
def sanitize_model_input(df):
    x = df.copy()
    x = x.replace([np.inf, -np.inf], np.nan)
    for c in x.columns:
        x[c] = pd.to_numeric(x[c], errors="coerce")
    return x.fillna(0)
 
def ensure_inference_features(df, feature_cols):
    df = df.copy()
    for c in feature_cols:
        if c not in df.columns:
            df[c] = 0
    return df

def choose_segment_by_history(raw_data, sku_code):
    sku_code = str(sku_code)
    work = raw_data.copy()
    work["ItemCode"] = work["ItemCode"].astype(str)
 
    sku_df = work[work["ItemCode"] == sku_code].copy()
    if sku_df.empty:
        return "SHORT"
 
    if "History_Segment" in sku_df.columns:
        sku_df = sku_df.sort_values(["Year", "Month_Number"])
        return str(sku_df["History_Segment"].iloc[-1])
 
    hist_len = sku_df[["Year", "Month_Number"]].drop_duplicates().shape[0]
    if hist_len >= 18:
        return "LONG"
    elif hist_len >= 10:
        return "MEDIUM"
    return "SHORT"
 
def get_next_period_from_history(df, sku_code):
    sku_code = str(sku_code)
 
    work = df.copy()
    work["ItemCode_Original"] = work["ItemCode"].astype(str)
 
    sku_hist = work[work["ItemCode_Original"] == sku_code].copy()
    if sku_hist.empty:
        return None, None, None
 
    last_row = sku_hist.sort_values(["Year", "Month_Number"]).iloc[-1:].copy()
 
    next_month = int(last_row["Month_Number"].iloc[0]) + 1
    next_year = int(last_row["Year"].iloc[0])
 
    if next_month > 12:
        next_month = 1
        next_year += 1
 
    return last_row, next_year, next_month
 
def infer_expected_bonus_flag(sku_code, prepared_df):
    sku_code = str(sku_code)
 
    work = prepared_df.copy()
    if "ItemCode_Original" not in work.columns:
        work["ItemCode_Original"] = work["ItemCode"].astype(str)
 
    g = work[work["ItemCode_Original"] == sku_code].copy()
    g = g.sort_values(["Year", "Month_Number"])
 
    if g.empty:
        return 0
 
    recent_bonus_rate = g["Bonus_Flag"].tail(12).mean() if "Bonus_Flag" in g.columns else 0
    recurring = g["Recurring_Bonus_SKU"].iloc[-1] if "Recurring_Bonus_SKU" in g.columns else 0
    cycle_len = g["Bonus_Cycle_Length"].iloc[-1] if "Bonus_Cycle_Length" in g.columns else 0
    months_since = g["Months_Since_Last_Bonus"].iloc[-1] if "Months_Since_Last_Bonus" in g.columns else 999
 
    if recurring == 1 and cycle_len > 0 and abs((months_since + 1) - cycle_len) <= 1:
        return 1
 
    if recurring == 1 and recent_bonus_rate >= 0.25:
        return 1
 
    return 0

def apply_residual_strength(row, pred_residual, segment="LONG"):
    baseline = float(row.get(BASELINE_COL, 0) or 0)
    rolling3 = float(row.get("Rolling3M_Mean", 0) or 0)
    lag1 = float(row.get("Lag1", 0) or 0)
    sku_mean = float(row.get("SKU_Mean_Demand", 0) or 0)
 
    anchor = max(baseline, rolling3, lag1, sku_mean, 1.0)
 
    if segment == "LONG":
        pos_strength = 1.25
        neg_strength = 1.05
    elif segment == "MEDIUM":
        pos_strength = 1.15
        neg_strength = 1.00
    else:
        pos_strength = 1.00
        neg_strength = 1.00
 
    residual = float(pred_residual)
 
    if residual >= 0:
        adjusted = residual * pos_strength
    else:
        adjusted = residual * neg_strength
 
    lower = -0.45 * anchor
    upper = 1.20 * anchor
 
    return float(np.clip(adjusted, lower, upper))
 
def classify_sku_behavior(row):
    cv = float(row.get("SKU_CV", 0) or 0)
    zero_rate = float(row.get("SKU_ZeroRate", 0) or 0)
    promo_profile = str(row.get("Promo_Profile", "NORMAL"))
    bonus_freq = float(row.get("Bonus_Frequency_All", 0) or 0)
 
    lag1 = float(row.get("Lag1", 0) or 0)
    lag2 = float(row.get("Lag2", 0) or 0)
    rolling3 = float(row.get("Rolling3M_Mean", 0) or 0)
    rolling6 = float(row.get("Rolling6M_Mean", rolling3) or rolling3)
    sku_mean = float(row.get("SKU_Mean_Demand", 0) or 0)
 
    anchor = max(rolling3, rolling6, sku_mean, 1)
 
    # Priority order matters
    if zero_rate >= 0.40:
        return "INTERMITTENT"
 
    if lag1 > 2.5 * anchor or lag2 > 2.5 * anchor:
        return "RECENT_SPIKE"
 
    if lag1 < 0.35 * anchor and rolling3 > 100:
        return "RECENT_DROP"
 
    if sku_mean < 100 and cv >= 1.0:
        return "LOW_VOLUME_VOLATILE"
 
    if promo_profile in ["PROMO_INFLUENCED", "PURE_PROMO"] or bonus_freq >= 0.25:
        return "PROMO_DRIVEN"
 
    if cv >= 1.0:
        return "VOLATILE"
 
    return "STABLE"

def get_latest_behavior_type(raw_data, sku_code):
    sku_code = str(sku_code)
 
    work = raw_data.copy()
    work["ItemCode"] = work["ItemCode"].astype(str)
 
    g = work[work["ItemCode"] == sku_code].copy()
    if g.empty:
        return "UNKNOWN"
 
    g = g.sort_values(["Year", "Month_Number"])
 
    if "Behavior_Type" in g.columns and g["Behavior_Type"].notna().any():
        return str(g["Behavior_Type"].iloc[-1])
 
    # Fallback: compute behavior type directly from Clean_Demand / Bonus_Flag
    # if the upstream frame never had build_model_features applied to it.
    if "Clean_Demand" not in g.columns:
        return "UNKNOWN"
 
    demand = pd.to_numeric(g["Clean_Demand"], errors="coerce").fillna(0)
    mean_demand = float(demand.mean()) if len(demand) > 0 else 0.0
    std_demand = float(demand.std()) if len(demand) > 1 else 0.0
    zero_rate = float((demand == 0).mean()) if len(demand) > 0 else 0.0
    cv = 0.0 if mean_demand <= 0 else std_demand / (mean_demand + 1)
 
    bonus_freq = (
        float(pd.to_numeric(g["Bonus_Flag"], errors="coerce").fillna(0).mean())
        if "Bonus_Flag" in g.columns
        else 0.0
    )
 
    if zero_rate >= 0.40:
        return "INTERMITTENT"
    if bonus_freq >= 0.25:
        return "PROMO_DRIVEN"
    if cv >= 1.0:
        return "VOLATILE"
    return "STABLE"

def build_next_month_inference_frame(raw_data, sku_code):
    sku_code = str(sku_code)
 
    df = force_itemcode_str(raw_data.copy())
    df = df.sort_values(["ItemCode", "Year", "Month_Number"])
 
    sku_hist = df[df["ItemCode"] == sku_code].copy()
    if sku_hist.empty:
        return None, None, None
 
    last_row = sku_hist.tail(1).copy()
 
    next_year = int(last_row["Year"].iloc[0])
    next_month = int(last_row["Month_Number"].iloc[0]) + 1
 
    if next_month > 12:
        next_month = 1
        next_year += 1
 
    future_row = last_row.copy()
    future_row["Year"] = next_year
    future_row["Month_Number"] = next_month
 
    for c in [
        "Secondary_Sales_Qty",
        "Primary_Sales_Qty",
        "Free_Qty",
        "Observed_Demand",
        "Effective_Demand",
        "Clean_Demand"
    ]:
        if c in future_row.columns:
            future_row[c] = 0
 
    full_future_df = pd.concat([df, future_row], ignore_index=True)
    full_future_df = build_model_features(full_future_df)
 
    next_row = full_future_df[
        (full_future_df["ItemCode"].astype(str) == sku_code) &
        (full_future_df["Year"] == next_year) &
        (full_future_df["Month_Number"] == next_month)
    ].copy()
 
    if next_row.empty:
        return None, None, None
 
    return next_row, next_year, next_month
 
def get_next_forecast_period(raw_data):
    periods = raw_data[["Year", "Month_Number"]].drop_duplicates()
    periods = periods.sort_values(["Year", "Month_Number"])
 
    last_year = int(periods.iloc[-1]["Year"])
    last_month = int(periods.iloc[-1]["Month_Number"])
 
    next_year = last_year
    next_month = last_month + 1
 
    if next_month > 12:
        next_month = 1
        next_year += 1
 
    return next_year, next_month
 
def inject_abc_class_into_df(df, artifacts):
    df = df.copy()
    abc_map = artifacts.get("abc_map", {})
 
    if "ItemCode" not in df.columns:
        return df
 
    key_col = "ItemCode_Original" if "ItemCode_Original" in df.columns else "ItemCode"
    df["ABC_Class"] = df[key_col].astype(str).map(abc_map).fillna(2).astype(int)
    return df


In [ ]:
# 1) Artifact lookup helpers
def get_long_deploy_artifact(model_name):
    model_name = str(model_name).upper()
 
    if model_name == "XGBOOST":
        return globals().get("xgb_long_deploy_artifacts")
    if model_name == "CATBOOST":
        return globals().get("catboost_long_deploy_artifacts")
    if model_name == "LIGHTGBM":
        return globals().get("lgbm_long_deploy_artifacts")
    return None
 
def get_medium_routing_row(sku_code):
    df = champion_medium_map_df.copy()
    df["ItemCode"] = df["ItemCode"].astype(str)
 
    hit = df[df["ItemCode"] == str(sku_code)]
    return None if hit.empty else hit.iloc[0].to_dict()
 
def get_long_routing_row(sku_code):
    df = champion_long_map_df.copy()
    df["ItemCode"] = df["ItemCode"].astype(str)
 
    hit = df[df["ItemCode"] == str(sku_code)]
    return None if hit.empty else hit.iloc[0].to_dict()
 
def get_medium_deploy_artifact(subgroup_name, model_name):
    key = (str(subgroup_name).upper(), str(model_name).upper())
    return medium_deploy_model_registry.get(key)


In [ ]:
# 2) Fallback
def fallback_forecast_from_row(row):
    regime = str(row.get("Demand_Regime", "NORMAL"))
    behavior = str(row.get("Behavior_Type", "STABLE"))
    demand_state = str(row.get("Demand_State", "MATURE"))
 
    lag1 = float(row.get("Lag1", 0) or 0)
    lag2 = float(row.get("Lag2", 0) or 0)
    lag3 = float(row.get("Lag3", 0) or 0)
    roll3 = float(row.get("Rolling3M_Mean", 0) or 0)
    roll6 = float(row.get("Rolling6M_Mean", roll3) or roll3)
    sku_mean = float(row.get("SKU_Mean_Demand", 0) or 0)
 
    expected_bonus = int(row.get("Expected_Bonus_Month", 0) or 0)
    last_bonus_demand = float(row.get("Last_Bonus_Demand", 0) or 0)
    avg_bonus_uplift = float(row.get("Avg_Bonus_Uplift", 1.0) or 1.0)
 
    anchors = [x for x in [lag1, lag2, lag3, roll3, roll6, sku_mean] if x > 0]
 
    if len(anchors) == 0:
        return 0.0
 
    base = (
        0.35 * lag1 +
        0.30 * roll3 +
        0.20 * roll6 +
        0.15 * sku_mean
    )
 
    if regime == "PROMO" or expected_bonus == 1:
        promo_anchor = max(base, last_bonus_demand, roll3 * avg_bonus_uplift)
        return max(promo_anchor, 0)
 
    if regime == "POST_PROMO_DROP":
        return max(0.50 * roll3 + 0.50 * roll6, 0)
 
    if demand_state == "GROWING":
        return max(base * 1.10, lag1)
 
    if demand_state == "DECLINING":
        return max(base * 0.85, 0)
 
    if demand_state == "DYING_OR_INTERMITTENT":
        return float(np.median(anchors))
 
    if behavior in ["INTERMITTENT", "LOW_VOLUME_VOLATILE"]:
        return float(np.median(anchors))
 
    if behavior == "RECENT_SPIKE":
        return 0.45 * roll3 + 0.35 * roll6 + 0.20 * sku_mean
 
    if behavior == "RECENT_DROP":
        return 0.60 * lag1 + 0.40 * roll3
 
    return max(base, 0)
 
def forecast_fallback_sku(sku_code, raw_data, segment, subgroup=""):
    next_row, next_year, next_month = build_next_month_inference_frame(raw_data, sku_code)
 
    if next_row is None:
        return None
 
    row = next_row.iloc[0].to_dict()
    pred = fallback_forecast_from_row(row)
 
    return {
        "ItemCode": int(float(sku_code)),
        "Forecast_Year": int(next_year),
        "Forecast_Month": int(next_month),
        "Forecast_Prediction": float(pred),
        "Segment": segment,
        "Subgroup": subgroup,
        "Champion_Model": "FALLBACK",
        "Used_Model": "FALLBACK",
        "Fallback_Used": 1,
        "Expected_Bonus": int(row.get("Expected_Bonus_Month", 0) or 0),
        "Residual_Baseline": np.nan,
        "Predicted_Residual": np.nan,
        "Status": "Success"
    }

# For Residual (tree models: XGBoost / CatBoost / RandomForest)
def predict_residual_model_sku(sku_code, raw_data, artifacts, used_model_name, subgroup=""):
    next_row, next_year, next_month = build_next_month_inference_frame(raw_data, sku_code)
 
    if next_row is None:
        return None
 
    feature_cols = artifacts["feature_cols"]
    model = artifacts["model"]
 
    next_row = ensure_inference_features(next_row, feature_cols)
 
    # FIX #1: inject ABC_Class BEFORE encode_itemcode overwrites ItemCode
    # with an integer category code below. Must happen here, while
    # next_row["ItemCode"] still holds the original SKU string — once
    # ItemCode becomes an int code, abc_map (keyed by original SKU strings)
    # can no longer be mapped against it.
    if "ABC_Class" in feature_cols:
        next_row = inject_abc_class_into_df(next_row, artifacts)
 
    itemcode_categories = artifacts.get("itemcode_categories", pd.Index([]))
    cat_to_code = {str(k): i for i, k in enumerate(itemcode_categories)}
    unk_code = len(cat_to_code)
 
    next_row["ItemCode_Original"] = next_row["ItemCode"].astype(str)
    next_row["ItemCode"] = (
        next_row["ItemCode"].astype(str)
        .map(cat_to_code)
        .fillna(unk_code)
        .astype(int)
    )
 
    if "clip_caps" in artifacts:
        next_row = apply_clip_caps(next_row, artifacts["clip_caps"])
 
    X = sanitize_model_input(next_row[feature_cols])
 
    scaler = artifacts.get("feature_scaler", None)
    if scaler is not None:
        X = scaler.transform(X)
 
    pred_residual = float(np.array(model.predict(X)).reshape(-1)[0])
    baseline = float(next_row[BASELINE_COL].iloc[0])
 
    row = next_row.iloc[0].to_dict()
 
    # FIX #2: apply the asymmetric residual-strength guardrail before
    # adding the residual to the baseline.
    pred_residual = apply_residual_strength(
        row, pred_residual, segment=artifacts.get("segment", "LONG")
    )
 
    forecast = max(baseline + pred_residual, 0)
 
    hist_len = float(next_row["History_Length"].iloc[0]) if "History_Length" in next_row.columns else 999
 
    if artifacts.get("segment") == "MEDIUM" and hist_len < 18:
        rule_pred = fallback_forecast_from_row(row)
 
        if hist_len < 12:
            forecast = 0.4 * forecast + 0.6 * rule_pred
        else:
            forecast = 0.7 * forecast + 0.3 * rule_pred
 
    return {
        "ItemCode": int(float(sku_code)),
        "Forecast_Year": int(next_year),
        "Forecast_Month": int(next_month),
        "Forecast_Prediction": float(forecast),
        "Segment": artifacts.get("segment", "UNKNOWN"),
        "Subgroup": subgroup,
        "Champion_Model": used_model_name,
        "Used_Model": used_model_name,
        "Fallback_Used": 0,
        "Residual_Baseline": baseline,
        "Predicted_Residual": pred_residual,
        "Status": "Success"
    }
 

In [ ]:
 # For GRU
def build_next_month_full_feature_frame(raw_data, sku_code):
    sku_code = str(sku_code)
 
    df = force_itemcode_str(raw_data.copy())
    df = df.sort_values(["ItemCode", "Year", "Month_Number"])
 
    sku_hist = df[df["ItemCode"] == sku_code].copy()
    if sku_hist.empty:
        return None, None, None, None
 
    last_row = sku_hist.tail(1).copy()
 
    next_year = int(last_row["Year"].iloc[0])
    next_month = int(last_row["Month_Number"].iloc[0]) + 1
 
    if next_month > 12:
        next_month = 1
        next_year += 1
 
    future_row = last_row.copy()
    future_row["Year"] = next_year
    future_row["Month_Number"] = next_month
 
    for c in [
        "Secondary_Sales_Qty", "Primary_Sales_Qty", "Free_Qty",
        "Observed_Demand", "Effective_Demand", "Clean_Demand"
    ]:
        if c in future_row.columns:
            future_row[c] = 0
 
    full_future_df = pd.concat([df, future_row], ignore_index=True)
    full_future_df = build_model_features(full_future_df)
    full_future_df = force_itemcode_str(full_future_df)
 
    next_row = full_future_df[
        (full_future_df["ItemCode"] == sku_code) &
        (full_future_df["Year"] == next_year) &
        (full_future_df["Month_Number"] == next_month)
    ].copy()
 
    if next_row.empty:
        return None, None, None, None
 
    return full_future_df, next_row, next_year, next_month
 
def forecast_long_gru_sku(sku_code, raw_data):
    sku_code = str(sku_code)
 
    artifacts = globals().get("gru_long_deploy_artifacts")
    scalers = globals().get("gru_long_deploy_scalers")
 
    if artifacts is None or scalers is None:
        return forecast_fallback_sku(sku_code, raw_data, "LONG")
 
    model = artifacts["model"]
    item_to_idx = artifacts["item_to_idx"]
    seq_features = artifacts["seq_features"]
    static_features = artifacts["static_features"]
    seq_len = artifacts["seq_len"]
 
    if sku_code not in item_to_idx:
        return forecast_fallback_sku(sku_code, raw_data, "LONG")
 
    full_df, next_row, next_year, next_month = build_next_month_full_feature_frame(
        raw_data,
        sku_code
    )
 
    if next_row is None:
        return forecast_fallback_sku(sku_code, raw_data, "LONG")
 
    if "clip_caps" in artifacts:
        full_df = apply_clip_caps(full_df, artifacts["clip_caps"])
        next_row = apply_clip_caps(next_row, artifacts["clip_caps"])
 
    full_df = ensure_inference_features(full_df, seq_features + static_features)
 
    # FIX #1: inject ABC_Class into full_df BEFORE slicing the sequence window.
    # This is the critical placement — injecting into next_row alone never
    # reaches static_vals, since static_vals is built from seq_slice.iloc[-1],
    # which comes from g, which comes from full_df.
    if "ABC_Class" in static_features:
        full_df = inject_abc_class_into_df(full_df, artifacts)
 
    g = full_df[full_df["ItemCode"].astype(str) == sku_code].copy()
    g = g.sort_values(["Year", "Month_Number"]).reset_index(drop=True)
 
    seq_slice = g.tail(seq_len).copy()
 
    if len(seq_slice) < seq_len:
        return forecast_fallback_sku(sku_code, raw_data, "LONG")
 
    seq_vals = seq_slice[seq_features].fillna(0).values
    seq_vals = scalers.seq_scaler.transform(seq_vals)
 
    static_vals = seq_slice.iloc[-1][static_features].fillna(0).values.reshape(1, -1)
    static_vals = scalers.static_scaler.transform(static_vals)
 
    device = next(model.parameters()).device
 
    x_seq = torch.tensor(seq_vals.reshape(1, seq_len, len(seq_features)), dtype=torch.float32).to(device)
    x_static = torch.tensor(static_vals, dtype=torch.float32).to(device)
    x_item = torch.tensor([item_to_idx[sku_code]], dtype=torch.long).to(device)
 
    model.eval()
    with torch.no_grad():
        pred_res_log = float(model(x_seq, x_static, x_item).cpu().numpy().reshape(-1)[0])
 
    log_clip = float(artifacts.get("gru_log_clip_value", 7.0))
    pred_res_log = float(np.clip(pred_res_log, -log_clip, log_clip))
    pred_residual = float(gru_signed_log_inverse([pred_res_log])[0])
 
    row = next_row.iloc[0].to_dict()
    baseline = float(row.get(BASELINE_COL, 0) or 0)
 
    # FIX #2: apply the asymmetric residual-strength guardrail before
    # adding the residual to the baseline. This is the fix that would have
    # caught SKU 604925's -10,954 residual (anchor ~14,360 -> floor ~-6,462),
    # since clipping wasn't triggered (-9.30 vs +/-10.19 log bound) but the
    # un-dampened inverse-transformed residual was still far too large.
    pred_residual = apply_residual_strength(row, pred_residual, segment="LONG")
 
    forecast = max(baseline + pred_residual, 0)
 
    return {
        "ItemCode": int(float(sku_code)),
        "Forecast_Year": int(next_year),
        "Forecast_Month": int(next_month),
        "Forecast_Prediction": float(forecast),
        "Segment": "LONG",
        "Subgroup": "",
        "Champion_Model": "GRU",
        "Used_Model": "GRU",
        "Fallback_Used": 0,
        "Expected_Bonus": int(row.get("Expected_Bonus_Month", 0) or 0),
        "Residual_Baseline": baseline,
        "Predicted_Residual": pred_residual,
        "Predicted_Residual_Log": pred_res_log,
        "Status": "Success"
    }
 

In [ ]:
# 3) LONG router
def forecast_long_sku(sku_code, raw_data):
    routing = get_long_routing_row(sku_code)
 
    if routing is None:
        return forecast_fallback_sku(sku_code, raw_data, "LONG")
 
    best_model = str(routing.get("Best_Model", "XGBOOST")).upper()
    final_model = str(routing.get("Final_Model", best_model)).upper()
 
    if final_model == "FALLBACK":
        return forecast_fallback_sku(sku_code, raw_data, "LONG")
 
    selected_model = final_model
 
    if selected_model == "GRU":
        return forecast_long_gru_sku(sku_code, raw_data)
 
    artifacts = get_long_deploy_artifact(selected_model)
 
    if artifacts is None:
        return forecast_fallback_sku(sku_code, raw_data, "LONG")
 
    result = predict_residual_model_sku(
        sku_code=sku_code,
        raw_data=raw_data,
        artifacts=artifacts,
        used_model_name=selected_model,
        subgroup=""
    )
 
    return result if result is not None else forecast_fallback_sku(sku_code, raw_data, "LONG")
 
# 4) MEDIUM router
def forecast_medium_sku(sku_code, raw_data):
    routing = get_medium_routing_row(sku_code)
 
    if routing is None:
        return forecast_fallback_sku(sku_code, raw_data, "MEDIUM", subgroup="UNKNOWN")
 
    subgroup = str(routing.get("Medium_Subgroup", "STABLE")).upper()
    best_model = str(routing.get("Best_Model", "XGBOOST")).upper()
    final_model = str(routing.get("Final_Model", best_model)).upper()
 
    if final_model == "FALLBACK":
        return forecast_fallback_sku(sku_code, raw_data, "MEDIUM", subgroup=subgroup)
 
    selected_model = final_model
    artifacts = get_medium_deploy_artifact(subgroup, selected_model)
 
    if artifacts is None:
        return forecast_fallback_sku(sku_code, raw_data, "MEDIUM", subgroup=subgroup)
 
    result = predict_residual_model_sku(
        sku_code=sku_code,
        raw_data=raw_data,
        artifacts=artifacts,
        used_model_name=selected_model,
        subgroup=subgroup
    )
 
    return result if result is not None else forecast_fallback_sku(
        sku_code, raw_data, "MEDIUM", subgroup=subgroup
    )
 
# 5) SHORT router
def forecast_short_sku(sku_code, raw_data):
    next_row, next_year, next_month = build_next_month_inference_frame(raw_data, sku_code)
 
    if next_row is None:
        return None
 
    # FIX #8: merge the Short_SKU_Type / Short_History_Length / etc. profile
    # onto next_row before calling short_rule_predict. Without this,
    # short_rule_predict's row.get("Short_SKU_Type", "SHORT_NORMAL") and
    # row.get("Short_History_Length", 0) silently default for every SHORT
    # SKU at inference, forcing the hist_len <= 2 branch every time and
    # bypassing the promo-blend / sku_mean-floor logic entirely.
    short_profile_df = build_short_sku_profile(force_itemcode_str(raw_data))
    next_row = merge_short_sku_profile(next_row, short_profile_df)
 
    row = next_row.iloc[0].to_dict()
    pred = short_rule_predict(row)
 
    subgroup = row.get("Short_SKU_Type", "SHORT_NORMAL")
 
    return {
        "ItemCode": int(float(sku_code)),
        "Forecast_Year": int(next_year),
        "Forecast_Month": int(next_month),
        "Forecast_Prediction": float(pred),
        "Segment": "SHORT",
        "Subgroup": subgroup,
        "Champion_Model": "RULE_BASED",
        "Used_Model": "RULE_BASED",
        "Fallback_Used": 0,
        "Expected_Bonus": int(row.get("Expected_Bonus_Month", 0) or 0),
        "Residual_Baseline": np.nan,
        "Predicted_Residual": np.nan,
        "Status": "Success"
    }

# 6) Unified router
def forecast_one_sku(sku_code, raw_data):
    segment = choose_segment_by_history(raw_data, sku_code)
 
    if segment == "LONG":
        result = forecast_long_sku(sku_code, raw_data)
    elif segment == "MEDIUM":
        result = forecast_medium_sku(sku_code, raw_data)
    else:
        result = forecast_short_sku(sku_code, raw_data)
 
    if result is not None:
        result["Behavior_Type"] = get_latest_behavior_type(raw_data, sku_code)
 
    return result


In [ ]:
def bulk_segmented_forecast(raw_data, sku_list):
    results = []
    failed = []
 
    for sku in sku_list:
        try:
            result = forecast_one_sku(sku, raw_data)
 
            if result is None or str(result.get("Status", "")).startswith("FAILED"):
                failed.append({
                    "ItemCode": int(float(sku)),
                    "Segment": choose_segment_by_history(raw_data, sku),
                    "Status": "FAILED"
                })
            else:
                results.append(result)
 
        except Exception as e:
            failed.append({
                "ItemCode": int(float(sku)),
                "Segment": choose_segment_by_history(raw_data, sku),
                "Status": str(e)
            })
 
    return pd.DataFrame(results), pd.DataFrame(failed)
 
''''''

def get_next_forecast_period(raw_data):
    periods = raw_data[["Year", "Month_Number"]].drop_duplicates()
    periods = periods.sort_values(["Year", "Month_Number"])

    last_year = int(periods.iloc[-1]["Year"])
    last_month = int(periods.iloc[-1]["Month_Number"])

    next_year = last_year
    next_month = last_month + 1

    if next_month > 12:
        next_month = 1
        next_year += 1

    return next_year, next_month

def get_latest_3_actual_months(raw_data):
    periods = (
        raw_data[["Year", "Month_Number"]]
        .drop_duplicates()
        .sort_values(["Year", "Month_Number"])
        .tail(3)
    )

    return list(periods[["Year", "Month_Number"]].itertuples(index=False, name=None))

def build_cutoff_dataset(raw_df, cutoff_year, cutoff_month):
    df = raw_df.copy()
    df["ItemCode"] = df["ItemCode"].astype(str)

    return df[
        (df["Year"] < cutoff_year) |
        ((df["Year"] == cutoff_year) & (df["Month_Number"] <= cutoff_month))
    ].copy()

def rebuild_cutoff_preprocessed_data(base_raw_df, cutoff_year, cutoff_month):
    df = base_raw_df.copy()
    df = force_itemcode_str(df)

    df = df[
        (df["Year"] < cutoff_year) |
        ((df["Year"] == cutoff_year) & (df["Month_Number"] <= cutoff_month))
    ].copy()

    df = df.sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)

    df = df.drop(
        columns=[
            "Lag1", "Lag2", "Lag3", "Lag6", "Lag12",
            "Rolling3M_Mean", "Rolling6M_Mean", "Rolling3M_Std",
            "Momentum",
            "SKU_Mean_Demand", "SKU_ZeroRate", "SKU_CV", "SKU_Std_Demand",
            "Demand_Regime", "Demand_Regime_Encoded",
            "Behavior_Type",
            "Target", "Residual_Target", "Residual_Baseline"
        ],
        errors="ignore"
    )

    df = clean_negative_values(df)
    df = add_observed_demand_features(df)
    df = add_bonus_profile_features(df)
    df = add_stock_flow_features(df)
    df = clean_demand_signal(df)
    df = add_last_bonus_demand_feature(df)
    df = add_structural_demand_state(df)
    df = add_history_length(df)

    return df

def run_latest_3_month_forecast(raw_data, sku_list):
    target_periods = get_latest_3_actual_months(raw_data)

    all_preds = []
    all_failed = []

    for forecast_year, forecast_month in target_periods:
        cutoff_year = forecast_year
        cutoff_month = forecast_month - 1

        if cutoff_month == 0:
            cutoff_month = 12
            cutoff_year -= 1

        hist_df = rebuild_cutoff_preprocessed_data(
            base_raw_df=raw_data,
            cutoff_year=cutoff_year,
            cutoff_month=cutoff_month
        )

        preds_df, failed_df = bulk_segmented_forecast(
            raw_data=hist_df,
            sku_list=sku_list
        )

        if not preds_df.empty:
            preds_df = preds_df[
                (preds_df["Forecast_Year"] == forecast_year) &
                (preds_df["Forecast_Month"] == forecast_month)
            ].copy()

            preds_df["Forecast_Period"] = (
                preds_df["Forecast_Year"].astype(int).astype(str)
                + "-"
                + preds_df["Forecast_Month"].astype(int).astype(str).str.zfill(2)
            )

            all_preds.append(preds_df)

        if not failed_df.empty:
            failed_df["Forecast_Period"] = f"{forecast_year}-{forecast_month:02d}"
            all_failed.append(failed_df)

    final_preds = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame()
    final_failed = pd.concat(all_failed, ignore_index=True) if all_failed else pd.DataFrame()

    return final_preds, final_failed

In [ ]:

# ------------------------------------------------------------
# 7) INFERENCE VALIDATION CHECKS
# ------------------------------------------------------------
required_objects = [
    "Cleaned_Base_Data",
    "PHARMA_SKUS",
    "champion_long_map_df",
    "champion_medium_map_df",
    "medium_deploy_model_registry",
    "short_rule_predict",
    "build_model_features",
    "build_short_sku_profile",
    "merge_short_sku_profile",
]

missing_objects = [x for x in required_objects if x not in globals()]
if missing_objects:
    raise ValueError(f"Missing required inference objects: {missing_objects}")

required_cols = ["ItemCode", "Year", "Month_Number", "Secondary_Sales_Qty"]
missing_cols = [c for c in required_cols if c not in Cleaned_Base_Data.columns]
if missing_cols:
    raise ValueError(f"Missing columns in Cleaned_Base_Data: {missing_cols}")

if len(PHARMA_SKUS) == 0:
    raise ValueError("PHARMA_SKUS is empty.")

print("Inference validation passed.")
print("SKU count:", len(PHARMA_SKUS))
print("Data rows:", len(Cleaned_Base_Data))

long_artifact_check = [
    "xgb_long_deploy_artifacts",
    "catboost_long_deploy_artifacts",
    "lgbm_long_deploy_artifacts"
]

available_long_artifacts = [x for x in long_artifact_check if x in globals()]
if len(available_long_artifacts) == 0:
    print("Warning: No LONG deploy artifacts found. LONG SKUs will use fallback.")

if "GRU" in champion_long_map_df["Final_Model"].astype(str).str.upper().values:
    if "forecast_long_gru_sku" not in globals():
        raise ValueError("GRU is selected in Final_Model but forecast_long_gru_sku is missing.")

all_inference_skus = sorted(set(int(x) for x in PHARMA_SKUS))

# ============================================================
# 8) NEXT-MONTH FORECAST
# ============================================================

next_month_predictions, next_month_failed = bulk_segmented_forecast(
    raw_data=Cleaned_Base_Data.copy(),
    sku_list=all_inference_skus
)

if next_month_predictions.empty:
    print("No next-month predictions generated.")
    print(next_month_failed.head(20))
    raise ValueError("next_month_predictions is empty.")

# keep only the next forecast month
latest_forecast_year, latest_forecast_month = get_next_forecast_period(Cleaned_Base_Data)

next_month_final = next_month_predictions[
    (next_month_predictions["Forecast_Year"] == latest_forecast_year) &
    (next_month_predictions["Forecast_Month"] == latest_forecast_month)
].copy()

next_month_final["Forecast_Period"] = (
    next_month_final["Forecast_Year"].astype(int).astype(str)
    + "-"
    + next_month_final["Forecast_Month"].astype(int).astype(str).str.zfill(2)
)

preferred_cols = [
    "ItemCode",
    "Forecast_Period",
    "Forecast_Year",
    "Forecast_Month",
    "Forecast_Prediction",
    "Segment",
    "Behavior_Type",
    "Subgroup",
    "Champion_Model",
    "Used_Model",
    "Fallback_Used",
    "Residual_Baseline",
    "Predicted_Residual",
    "Predicted_Residual_Log",
    "Status"
]

next_month_final = next_month_final[
    [c for c in preferred_cols if c in next_month_final.columns]
].copy()

output_file_next = "next_month_forecast.xlsx"

with pd.ExcelWriter(output_file_next, engine="openpyxl") as writer:
    next_month_final.to_excel(writer, sheet_name="Next_Month_Forecast", index=False)
    next_month_predictions.to_excel(writer, sheet_name="All_Next_Predictions", index=False)

    if not next_month_failed.empty:
        next_month_failed.to_excel(writer, sheet_name="Failed_SKUs", index=False)

print("Saved:", output_file_next)
print("Next month:", f"{int(latest_forecast_year)}-{int(latest_forecast_month):02d}")
print("Forecast rows:", len(next_month_final))
print("Unique SKUs:", next_month_final["ItemCode"].nunique())

Inference validation passed.
SKU count: 728
Data rows: 35708


In [46]:
all_inference_skus = sorted(set(int(x) for x in PHARMA_SKUS))

'''
latest3_predictions, latest3_failed = run_latest_3_month_forecast(
    raw_data=Cleaned_Base_Data.copy(),
    sku_list=all_inference_skus
)

actuals_df = Cleaned_Base_Data.copy()
actuals_df["ItemCode"] = actuals_df["ItemCode"].astype(str)

actuals_df["Forecast_Period"] = (
    actuals_df["Year"].astype(int).astype(str)
    + "-"
    + actuals_df["Month_Number"].astype(int).astype(str).str.zfill(2)
)

actuals_lookup = actuals_df[
    ["ItemCode", "Forecast_Period", "Secondary_Sales_Qty"]
].rename(columns={"Secondary_Sales_Qty": "Actual_Sale"})

actuals_lookup = actuals_lookup.drop_duplicates(
    subset=["ItemCode", "Forecast_Period"],
    keep="last"
)

if latest3_predictions.empty:
    print("No successful predictions.")
    print("Failed preview:")
    print(latest3_failed.head(20))
    raise ValueError("latest3_predictions is empty. Check Failed_SKUs.")

latest3_predictions["ItemCode"] = latest3_predictions["ItemCode"].astype(str)

latest3_predictions = latest3_predictions.merge(
    actuals_lookup,
    on=["ItemCode", "Forecast_Period"],
    how="left"
)

latest3_predictions["Absolute_Error"] = (
    latest3_predictions["Actual_Sale"] -
    latest3_predictions["Forecast_Prediction"]
).abs()

latest3_predictions["Accuracy_%"] = np.where(
    latest3_predictions["Actual_Sale"] > 0,
    (1 - latest3_predictions["Absolute_Error"] / latest3_predictions["Actual_Sale"]).clip(lower=0) * 100,
    np.nan
)
'''

'\nlatest3_predictions, latest3_failed = run_latest_3_month_forecast(\n    raw_data=Cleaned_Base_Data.copy(),\n    sku_list=all_inference_skus\n)\n\nactuals_df = Cleaned_Base_Data.copy()\nactuals_df["ItemCode"] = actuals_df["ItemCode"].astype(str)\n\nactuals_df["Forecast_Period"] = (\n    actuals_df["Year"].astype(int).astype(str)\n    + "-"\n    + actuals_df["Month_Number"].astype(int).astype(str).str.zfill(2)\n)\n\nactuals_lookup = actuals_df[\n    ["ItemCode", "Forecast_Period", "Secondary_Sales_Qty"]\n].rename(columns={"Secondary_Sales_Qty": "Actual_Sale"})\n\nactuals_lookup = actuals_lookup.drop_duplicates(\n    subset=["ItemCode", "Forecast_Period"],\n    keep="last"\n)\n\nif latest3_predictions.empty:\n    print("No successful predictions.")\n    print("Failed preview:")\n    print(latest3_failed.head(20))\n    raise ValueError("latest3_predictions is empty. Check Failed_SKUs.")\n\nlatest3_predictions["ItemCode"] = latest3_predictions["ItemCode"].astype(str)\n\nlatest3_predictio

In [47]:
'''
output_file = "latest_3_month_inference_accuracy.xlsx"

monthly_summary = (
    latest3_predictions
    .groupby("Forecast_Period")
    .agg(
        Forecast_Rows=("ItemCode", "count"),
        Unique_SKUs=("ItemCode", "nunique"),
        Actuals_Available=("Actual_Sale", lambda x: x.notna().sum()),
        Missing_Actuals=("Actual_Sale", lambda x: x.isna().sum()),
        Mean_Accuracy=("Accuracy_%", "mean"),
        Median_Accuracy=("Accuracy_%", "median"),
        Total_Actual=("Actual_Sale", "sum"),
        Total_Forecast=("Forecast_Prediction", "sum"),
        Total_Abs_Error=("Absolute_Error", "sum")
    )
    .reset_index()
)

monthly_summary["WMAPE_%"] = np.where(
    monthly_summary["Total_Actual"] > 0,
    monthly_summary["Total_Abs_Error"] / monthly_summary["Total_Actual"] * 100,
    np.nan
)

monthly_summary["Weighted_Accuracy_%"] = np.where(
    monthly_summary["WMAPE_%"].notna(),
    (100 - monthly_summary["WMAPE_%"]).clip(lower=0),
    np.nan
)

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    latest3_predictions.to_excel(writer, sheet_name="Latest3_Forecasts", index=False)
    monthly_summary.to_excel(writer, sheet_name="Monthly_Summary", index=False)

    if not latest3_failed.empty:
        latest3_failed.to_excel(writer, sheet_name="Failed_SKUs", index=False)

print("Saved:", output_file)

'''

'\noutput_file = "latest_3_month_inference_accuracy.xlsx"\n\nmonthly_summary = (\n    latest3_predictions\n    .groupby("Forecast_Period")\n    .agg(\n        Forecast_Rows=("ItemCode", "count"),\n        Unique_SKUs=("ItemCode", "nunique"),\n        Actuals_Available=("Actual_Sale", lambda x: x.notna().sum()),\n        Missing_Actuals=("Actual_Sale", lambda x: x.isna().sum()),\n        Mean_Accuracy=("Accuracy_%", "mean"),\n        Median_Accuracy=("Accuracy_%", "median"),\n        Total_Actual=("Actual_Sale", "sum"),\n        Total_Forecast=("Forecast_Prediction", "sum"),\n        Total_Abs_Error=("Absolute_Error", "sum")\n    )\n    .reset_index()\n)\n\nmonthly_summary["WMAPE_%"] = np.where(\n    monthly_summary["Total_Actual"] > 0,\n    monthly_summary["Total_Abs_Error"] / monthly_summary["Total_Actual"] * 100,\n    np.nan\n)\n\nmonthly_summary["Weighted_Accuracy_%"] = np.where(\n    monthly_summary["WMAPE_%"].notna(),\n    (100 - monthly_summary["WMAPE_%"]).clip(lower=0),\n    np.

In [ ]:
# ============================================================
# 8) SEPARATE NEXT-MONTH FORECAST FILE
# ============================================================

next_month_predictions, next_month_failed = bulk_segmented_forecast(
    raw_data=Cleaned_Base_Data.copy(),
    sku_list=all_inference_skus
)

if next_month_predictions.empty:
    print("No next-month predictions generated.")
    print(next_month_failed.head(20))
    raise ValueError("next_month_predictions is empty.")

# keep only the next forecast month
latest_forecast_year, latest_forecast_month = get_next_forecast_period(Cleaned_Base_Data)

next_month_final = next_month_predictions[
    (next_month_predictions["Forecast_Year"] == latest_forecast_year) &
    (next_month_predictions["Forecast_Month"] == latest_forecast_month)
].copy()

next_month_final["Forecast_Period"] = (
    next_month_final["Forecast_Year"].astype(int).astype(str)
    + "-"
    + next_month_final["Forecast_Month"].astype(int).astype(str).str.zfill(2)
)

preferred_cols = [
    "ItemCode",
    "Forecast_Period",
    "Forecast_Year",
    "Forecast_Month",
    "Forecast_Prediction",
    "Segment",
    "Behavior_Type",
    "Subgroup",
    "Champion_Model",
    "Used_Model",
    "Fallback_Used",
    "Residual_Baseline",
    "Predicted_Residual",
    "Predicted_Residual_Raw",
    "Predicted_Residual_Log",
    "Status"
]

next_month_final = next_month_final[
    [c for c in preferred_cols if c in next_month_final.columns]
].copy()

'''
sku_accuracy = (
    latest3_predictions
    .groupby("ItemCode")
    .agg(
        Accuracy_Months=("Forecast_Period", "nunique"),
        Total_Actual=("Actual_Sale", "sum"),
        Total_Abs_Error=("Absolute_Error", "sum")
    )
    .reset_index()
)

sku_accuracy["Product_WMAPE"] = np.where(
    sku_accuracy["Total_Actual"] > 0,
    sku_accuracy["Total_Abs_Error"] / sku_accuracy["Total_Actual"] * 100,
    np.nan
)

sku_accuracy["Product_Accuracy"] = np.where(
    sku_accuracy["Product_WMAPE"].notna(),
    (100 - sku_accuracy["Product_WMAPE"]).clip(lower=0),
    np.nan
)

sku_accuracy["ItemCode"] = sku_accuracy["ItemCode"].astype(str)
next_month_final["ItemCode"] = next_month_final["ItemCode"].astype(str)

next_month_final = next_month_final.merge(
    sku_accuracy[
        ["ItemCode", "Accuracy_Months", "Product_WMAPE", "Product_Accuracy"]
    ],
    on="ItemCode",
    how="left"
)
'''
output_file_next = "next_month_forecast.xlsx"

with pd.ExcelWriter(output_file_next, engine="openpyxl") as writer:
    next_month_final.to_excel(writer, sheet_name="Next_Month_Forecast", index=False)
    next_month_predictions.to_excel(writer, sheet_name="All_Next_Predictions", index=False)

    if not next_month_failed.empty:
        next_month_failed.to_excel(writer, sheet_name="Failed_SKUs", index=False)

print("Saved:", output_file_next)
print("Next month:", f"{int(latest_forecast_year)}-{int(latest_forecast_month):02d}")
print("Forecast rows:", len(next_month_final))
print("Unique SKUs:", next_month_final["ItemCode"].nunique())

KeyboardInterrupt: 

In [49]:
# 04_INFERENCE
# ------------------------------------------------------------
# 0) SMALL SHARED HELPERS LONG+MEDIUM+SHORT
# ------------------------------------------------------------

# FIX #3: force_itemcode_str now matches the 02_MODEL_FEATURE_ENGINEERING
# version exactly (numeric coercion -> int -> str). This prevents
# "600308.0"-style string keys from breaking lookups against item_to_idx,
# abc_map, and champion_*_map_df, which were all built with clean int-string keys.
def force_itemcode_str(df):
    df = df.copy()
    if "ItemCode" in df.columns:
        df["ItemCode"] = pd.to_numeric(df["ItemCode"], errors="coerce")
        df = df.dropna(subset=["ItemCode"])
        df["ItemCode"] = df["ItemCode"].astype(int).astype(str)

    if "ItemCode_Original" in df.columns:
        df["ItemCode_Original"] = df["ItemCode_Original"].astype(str)

    return df


def sanitize_model_input(df):
    x = df.copy()
    x = x.replace([np.inf, -np.inf], np.nan)
    for c in x.columns:
        x[c] = pd.to_numeric(x[c], errors="coerce")
    return x.fillna(0)


def ensure_inference_features(df, feature_cols):
    df = df.copy()
    for c in feature_cols:
        if c not in df.columns:
            df[c] = 0
    return df


def choose_segment_by_history(raw_data, sku_code):
    sku_code = str(sku_code)
    work = raw_data.copy()
    work["ItemCode"] = work["ItemCode"].astype(str)

    sku_df = work[work["ItemCode"] == sku_code].copy()
    if sku_df.empty:
        return "SHORT"

    if "History_Segment" in sku_df.columns:
        sku_df = sku_df.sort_values(["Year", "Month_Number"])
        return str(sku_df["History_Segment"].iloc[-1])

    hist_len = sku_df[["Year", "Month_Number"]].drop_duplicates().shape[0]
    if hist_len >= 18:
        return "LONG"
    elif hist_len >= 10:
        return "MEDIUM"
    return "SHORT"


def get_next_period_from_history(df, sku_code):
    sku_code = str(sku_code)

    work = df.copy()
    work["ItemCode_Original"] = work["ItemCode"].astype(str)

    sku_hist = work[work["ItemCode_Original"] == sku_code].copy()
    if sku_hist.empty:
        return None, None, None

    last_row = sku_hist.sort_values(["Year", "Month_Number"]).iloc[-1:].copy()

    next_month = int(last_row["Month_Number"].iloc[0]) + 1
    next_year = int(last_row["Year"].iloc[0])

    if next_month > 12:
        next_month = 1
        next_year += 1

    return last_row, next_year, next_month


def infer_expected_bonus_flag(sku_code, prepared_df):
    sku_code = str(sku_code)

    work = prepared_df.copy()
    if "ItemCode_Original" not in work.columns:
        work["ItemCode_Original"] = work["ItemCode"].astype(str)

    g = work[work["ItemCode_Original"] == sku_code].copy()
    g = g.sort_values(["Year", "Month_Number"])

    if g.empty:
        return 0

    recent_bonus_rate = g["Bonus_Flag"].tail(12).mean() if "Bonus_Flag" in g.columns else 0
    recurring = g["Recurring_Bonus_SKU"].iloc[-1] if "Recurring_Bonus_SKU" in g.columns else 0
    cycle_len = g["Bonus_Cycle_Length"].iloc[-1] if "Bonus_Cycle_Length" in g.columns else 0
    months_since = g["Months_Since_Last_Bonus"].iloc[-1] if "Months_Since_Last_Bonus" in g.columns else 999

    if recurring == 1 and cycle_len > 0 and abs((months_since + 1) - cycle_len) <= 1:
        return 1

    if recurring == 1 and recent_bonus_rate >= 0.25:
        return 1

    return 0


# FIX #2: uncommented and now actually wired into both
# predict_residual_model_sku (tree models) and forecast_long_gru_sku (GRU)
# right before the baseline is added to the residual. This is the asymmetric
# guardrail that prevents a single bad residual prediction from collapsing
# the forecast (e.g. the -10,954 residual seen on SKU 604925, which sat
# well within the log-clip bound but still produced a 60%+ forecast drop).
def apply_residual_strength(row, pred_residual, segment="LONG"):
    baseline = float(row.get(BASELINE_COL, 0) or 0)
    rolling3 = float(row.get("Rolling3M_Mean", 0) or 0)
    lag1 = float(row.get("Lag1", 0) or 0)
    sku_mean = float(row.get("SKU_Mean_Demand", 0) or 0)

    anchor = max(baseline, rolling3, lag1, sku_mean, 1.0)

    if segment == "LONG":
        pos_strength = 1.25
        neg_strength = 1.05
    elif segment == "MEDIUM":
        pos_strength = 1.15
        neg_strength = 1.00
    else:
        pos_strength = 1.00
        neg_strength = 1.00

    residual = float(pred_residual)

    if residual >= 0:
        adjusted = residual * pos_strength
    else:
        adjusted = residual * neg_strength

    lower = -0.45 * anchor
    upper = 1.20 * anchor

    return float(np.clip(adjusted, lower, upper))


def classify_sku_behavior(row):
    cv = float(row.get("SKU_CV", 0) or 0)
    zero_rate = float(row.get("SKU_ZeroRate", 0) or 0)
    promo_profile = str(row.get("Promo_Profile", "NORMAL"))
    bonus_freq = float(row.get("Bonus_Frequency_All", 0) or 0)

    lag1 = float(row.get("Lag1", 0) or 0)
    lag2 = float(row.get("Lag2", 0) or 0)
    rolling3 = float(row.get("Rolling3M_Mean", 0) or 0)
    rolling6 = float(row.get("Rolling6M_Mean", rolling3) or rolling3)
    sku_mean = float(row.get("SKU_Mean_Demand", 0) or 0)

    anchor = max(rolling3, rolling6, sku_mean, 1)

    # Priority order matters
    if zero_rate >= 0.40:
        return "INTERMITTENT"

    if lag1 > 2.5 * anchor or lag2 > 2.5 * anchor:
        return "RECENT_SPIKE"

    if lag1 < 0.35 * anchor and rolling3 > 100:
        return "RECENT_DROP"

    if sku_mean < 100 and cv >= 1.0:
        return "LOW_VOLUME_VOLATILE"

    if promo_profile in ["PROMO_INFLUENCED", "PURE_PROMO"] or bonus_freq >= 0.25:
        return "PROMO_DRIVEN"

    if cv >= 1.0:
        return "VOLATILE"

    return "STABLE"


# FIX #9: get_latest_behavior_type now computes Behavior_Type on the fly
# from raw stats if the column doesn't exist on raw_data, instead of
# returning "UNKNOWN" for every SKU. This mirrors the same logic used in
# build_model_features's Behavior_Type assignment, applied to a single SKU's
# history rather than requiring the full pipeline to have run already.
def get_latest_behavior_type(raw_data, sku_code):
    sku_code = str(sku_code)

    work = raw_data.copy()
    work["ItemCode"] = work["ItemCode"].astype(str)

    g = work[work["ItemCode"] == sku_code].copy()
    if g.empty:
        return "UNKNOWN"

    g = g.sort_values(["Year", "Month_Number"])

    if "Behavior_Type" in g.columns and g["Behavior_Type"].notna().any():
        return str(g["Behavior_Type"].iloc[-1])

    # Fallback: compute behavior type directly from Clean_Demand / Bonus_Flag
    # if the upstream frame never had build_model_features applied to it.
    if "Clean_Demand" not in g.columns:
        return "UNKNOWN"

    demand = pd.to_numeric(g["Clean_Demand"], errors="coerce").fillna(0)
    mean_demand = float(demand.mean()) if len(demand) > 0 else 0.0
    std_demand = float(demand.std()) if len(demand) > 1 else 0.0
    zero_rate = float((demand == 0).mean()) if len(demand) > 0 else 0.0
    cv = 0.0 if mean_demand <= 0 else std_demand / (mean_demand + 1)

    bonus_freq = (
        float(pd.to_numeric(g["Bonus_Flag"], errors="coerce").fillna(0).mean())
        if "Bonus_Flag" in g.columns
        else 0.0
    )

    if zero_rate >= 0.40:
        return "INTERMITTENT"
    if bonus_freq >= 0.25:
        return "PROMO_DRIVEN"
    if cv >= 1.0:
        return "VOLATILE"
    return "STABLE"


def build_next_month_inference_frame(raw_data, sku_code):
    sku_code = str(sku_code)

    df = force_itemcode_str(raw_data.copy())
    df = df.sort_values(["ItemCode", "Year", "Month_Number"])

    sku_hist = df[df["ItemCode"] == sku_code].copy()
    if sku_hist.empty:
        return None, None, None

    last_row = sku_hist.tail(1).copy()

    next_year = int(last_row["Year"].iloc[0])
    next_month = int(last_row["Month_Number"].iloc[0]) + 1

    if next_month > 12:
        next_month = 1
        next_year += 1

    future_row = last_row.copy()
    future_row["Year"] = next_year
    future_row["Month_Number"] = next_month

    for c in [
        "Secondary_Sales_Qty",
        "Primary_Sales_Qty",
        "Free_Qty",
        "Observed_Demand",
        "Effective_Demand",
        "Clean_Demand"
    ]:
        if c in future_row.columns:
            future_row[c] = 0

    full_future_df = pd.concat([df, future_row], ignore_index=True)
    full_future_df = build_model_features(full_future_df)

    next_row = full_future_df[
        (full_future_df["ItemCode"].astype(str) == sku_code) &
        (full_future_df["Year"] == next_year) &
        (full_future_df["Month_Number"] == next_month)
    ].copy()

    if next_row.empty:
        return None, None, None

    return next_row, next_year, next_month


def get_next_forecast_period(raw_data):
    periods = raw_data[["Year", "Month_Number"]].drop_duplicates()
    periods = periods.sort_values(["Year", "Month_Number"])

    last_year = int(periods.iloc[-1]["Year"])
    last_month = int(periods.iloc[-1]["Month_Number"])

    next_year = last_year
    next_month = last_month + 1

    if next_month > 12:
        next_month = 1
        next_year += 1

    return next_year, next_month


# 1) Artifact lookup helpers
def get_long_deploy_artifact(model_name):
    model_name = str(model_name).upper()

    if model_name == "XGBOOST":
        return globals().get("xgb_long_deploy_artifacts")
    if model_name == "CATBOOST":
        return globals().get("catboost_long_deploy_artifacts")
    if model_name == "LIGHTGBM":
        return globals().get("lgbm_long_deploy_artifacts")
    return None


def get_medium_routing_row(sku_code):
    df = champion_medium_map_df.copy()
    df["ItemCode"] = df["ItemCode"].astype(str)

    hit = df[df["ItemCode"] == str(sku_code)]
    return None if hit.empty else hit.iloc[0].to_dict()


def get_long_routing_row(sku_code):
    df = champion_long_map_df.copy()
    df["ItemCode"] = df["ItemCode"].astype(str)

    hit = df[df["ItemCode"] == str(sku_code)]
    return None if hit.empty else hit.iloc[0].to_dict()


def get_medium_deploy_artifact(subgroup_name, model_name):
    key = (str(subgroup_name).upper(), str(model_name).upper())
    return medium_deploy_model_registry.get(key)


# FIX #1 (helper): generic ABC_Class injector that works on ANY frame —
# next_row (tree models, before encode_itemcode overwrites ItemCode with an
# int category code) or full_df/g (GRU, before the sequence window is sliced).
# Without this, ABC_Class was either MISSING entirely (GRU) or silently
# defaulted to 0 via ensure_inference_features, which happens to collide
# with "Class A" encoding but isn't the SKU's actual abc_map lookup.
def inject_abc_class_into_df(df, artifacts):
    df = df.copy()
    abc_map = artifacts.get("abc_map", {})

    if "ItemCode" not in df.columns:
        return df

    key_col = "ItemCode_Original" if "ItemCode_Original" in df.columns else "ItemCode"
    df["ABC_Class"] = df[key_col].astype(str).map(abc_map).fillna(2).astype(int)
    return df


# 2) Fallback
def fallback_forecast_from_row(row):
    regime = str(row.get("Demand_Regime", "NORMAL"))
    behavior = str(row.get("Behavior_Type", "STABLE"))
    demand_state = str(row.get("Demand_State", "MATURE"))

    lag1 = float(row.get("Lag1", 0) or 0)
    lag2 = float(row.get("Lag2", 0) or 0)
    lag3 = float(row.get("Lag3", 0) or 0)
    roll3 = float(row.get("Rolling3M_Mean", 0) or 0)
    roll6 = float(row.get("Rolling6M_Mean", roll3) or roll3)
    sku_mean = float(row.get("SKU_Mean_Demand", 0) or 0)

    expected_bonus = int(row.get("Expected_Bonus_Month", 0) or 0)
    last_bonus_demand = float(row.get("Last_Bonus_Demand", 0) or 0)
    avg_bonus_uplift = float(row.get("Avg_Bonus_Uplift", 1.0) or 1.0)

    anchors = [x for x in [lag1, lag2, lag3, roll3, roll6, sku_mean] if x > 0]

    if len(anchors) == 0:
        return 0.0

    base = (
        0.35 * lag1 +
        0.30 * roll3 +
        0.20 * roll6 +
        0.15 * sku_mean
    )

    if regime == "PROMO" or expected_bonus == 1:
        promo_anchor = max(base, last_bonus_demand, roll3 * avg_bonus_uplift)
        return max(promo_anchor, 0)

    if regime == "POST_PROMO_DROP":
        return max(0.50 * roll3 + 0.50 * roll6, 0)

    if demand_state == "GROWING":
        return max(base * 1.10, lag1)

    if demand_state == "DECLINING":
        return max(base * 0.85, 0)

    if demand_state == "DYING_OR_INTERMITTENT":
        return float(np.median(anchors))

    if behavior in ["INTERMITTENT", "LOW_VOLUME_VOLATILE"]:
        return float(np.median(anchors))

    if behavior == "RECENT_SPIKE":
        return 0.45 * roll3 + 0.35 * roll6 + 0.20 * sku_mean

    if behavior == "RECENT_DROP":
        return 0.60 * lag1 + 0.40 * roll3

    return max(base, 0)


def forecast_fallback_sku(sku_code, raw_data, segment, subgroup=""):
    next_row, next_year, next_month = build_next_month_inference_frame(raw_data, sku_code)

    if next_row is None:
        return None

    row = next_row.iloc[0].to_dict()
    pred = fallback_forecast_from_row(row)

    return {
        "ItemCode": int(float(sku_code)),
        "Forecast_Year": int(next_year),
        "Forecast_Month": int(next_month),
        "Forecast_Prediction": float(pred),
        "Segment": segment,
        "Subgroup": subgroup,
        "Champion_Model": "FALLBACK",
        "Used_Model": "FALLBACK",
        "Fallback_Used": 1,
        "Expected_Bonus": int(row.get("Expected_Bonus_Month", 0) or 0),
        "Residual_Baseline": np.nan,
        "Predicted_Residual": np.nan,
        "Status": "Success"
    }


# For Residual (tree models: XGBoost / CatBoost / RandomForest)
def predict_residual_model_sku(sku_code, raw_data, artifacts, used_model_name, subgroup=""):
    next_row, next_year, next_month = build_next_month_inference_frame(raw_data, sku_code)

    if next_row is None:
        return None

    feature_cols = artifacts["feature_cols"]
    model = artifacts["model"]

    next_row = ensure_inference_features(next_row, feature_cols)

    # FIX #1: inject ABC_Class BEFORE encode_itemcode overwrites ItemCode
    # with an integer category code below. Must happen here, while
    # next_row["ItemCode"] still holds the original SKU string — once
    # ItemCode becomes an int code, abc_map (keyed by original SKU strings)
    # can no longer be mapped against it.
    if "ABC_Class" in feature_cols:
        next_row = inject_abc_class_into_df(next_row, artifacts)

    itemcode_categories = artifacts.get("itemcode_categories", pd.Index([]))
    cat_to_code = {str(k): i for i, k in enumerate(itemcode_categories)}
    unk_code = len(cat_to_code)

    next_row["ItemCode_Original"] = next_row["ItemCode"].astype(str)
    next_row["ItemCode"] = (
        next_row["ItemCode"].astype(str)
        .map(cat_to_code)
        .fillna(unk_code)
        .astype(int)
    )

    if "clip_caps" in artifacts:
        next_row = apply_clip_caps(next_row, artifacts["clip_caps"])

    X = sanitize_model_input(next_row[feature_cols])

    scaler = artifacts.get("feature_scaler", None)
    if scaler is not None:
        X = scaler.transform(X)

    pred_residual = float(np.array(model.predict(X)).reshape(-1)[0])
    baseline = float(next_row[BASELINE_COL].iloc[0])

    row = next_row.iloc[0].to_dict()

    # FIX #2: apply the asymmetric residual-strength guardrail before
    # adding the residual to the baseline.
    pred_residual = apply_residual_strength(
        row, pred_residual, segment=artifacts.get("segment", "LONG")
    )

    forecast = max(baseline + pred_residual, 0)

    hist_len = float(next_row["History_Length"].iloc[0]) if "History_Length" in next_row.columns else 999

    if artifacts.get("segment") == "MEDIUM" and hist_len < 18:
        rule_pred = fallback_forecast_from_row(row)

        if hist_len < 12:
            forecast = 0.4 * forecast + 0.6 * rule_pred
        else:
            forecast = 0.7 * forecast + 0.3 * rule_pred

    return {
        "ItemCode": int(float(sku_code)),
        "Forecast_Year": int(next_year),
        "Forecast_Month": int(next_month),
        "Forecast_Prediction": float(forecast),
        "Segment": artifacts.get("segment", "UNKNOWN"),
        "Subgroup": subgroup,
        "Champion_Model": used_model_name,
        "Used_Model": used_model_name,
        "Fallback_Used": 0,
        "Residual_Baseline": baseline,
        "Predicted_Residual": pred_residual,
        "Status": "Success"
    }


# For GRU
def build_next_month_full_feature_frame(raw_data, sku_code):
    sku_code = str(sku_code)

    df = force_itemcode_str(raw_data.copy())
    df = df.sort_values(["ItemCode", "Year", "Month_Number"])

    sku_hist = df[df["ItemCode"] == sku_code].copy()
    if sku_hist.empty:
        return None, None, None, None

    last_row = sku_hist.tail(1).copy()

    next_year = int(last_row["Year"].iloc[0])
    next_month = int(last_row["Month_Number"].iloc[0]) + 1

    if next_month > 12:
        next_month = 1
        next_year += 1

    future_row = last_row.copy()
    future_row["Year"] = next_year
    future_row["Month_Number"] = next_month

    for c in [
        "Secondary_Sales_Qty", "Primary_Sales_Qty", "Free_Qty",
        "Observed_Demand", "Effective_Demand", "Clean_Demand"
    ]:
        if c in future_row.columns:
            future_row[c] = 0

    full_future_df = pd.concat([df, future_row], ignore_index=True)
    full_future_df = build_model_features(full_future_df)
    full_future_df = force_itemcode_str(full_future_df)

    next_row = full_future_df[
        (full_future_df["ItemCode"] == sku_code) &
        (full_future_df["Year"] == next_year) &
        (full_future_df["Month_Number"] == next_month)
    ].copy()

    if next_row.empty:
        return None, None, None, None

    return full_future_df, next_row, next_year, next_month


def forecast_long_gru_sku(sku_code, raw_data):
    sku_code = str(sku_code)

    artifacts = globals().get("gru_long_deploy_artifacts")
    scalers = globals().get("gru_long_deploy_scalers")

    if artifacts is None or scalers is None:
        return forecast_fallback_sku(sku_code, raw_data, "LONG")

    model = artifacts["model"]
    item_to_idx = artifacts["item_to_idx"]
    seq_features = artifacts["seq_features"]
    static_features = artifacts["static_features"]
    seq_len = artifacts["seq_len"]

    if sku_code not in item_to_idx:
        return forecast_fallback_sku(sku_code, raw_data, "LONG")

    full_df, next_row, next_year, next_month = build_next_month_full_feature_frame(
        raw_data,
        sku_code
    )

    if next_row is None:
        return forecast_fallback_sku(sku_code, raw_data, "LONG")

    if "clip_caps" in artifacts:
        full_df = apply_clip_caps(full_df, artifacts["clip_caps"])
        next_row = apply_clip_caps(next_row, artifacts["clip_caps"])

    full_df = ensure_inference_features(full_df, seq_features + static_features)

    # FIX #1: inject ABC_Class into full_df BEFORE slicing the sequence window.
    # This is the critical placement — injecting into next_row alone never
    # reaches static_vals, since static_vals is built from seq_slice.iloc[-1],
    # which comes from g, which comes from full_df.
    if "ABC_Class" in static_features:
        full_df = inject_abc_class_into_df(full_df, artifacts)

    g = full_df[full_df["ItemCode"].astype(str) == sku_code].copy()
    g = g.sort_values(["Year", "Month_Number"]).reset_index(drop=True)

    seq_slice = g.tail(seq_len).copy()

    if len(seq_slice) < seq_len:
        return forecast_fallback_sku(sku_code, raw_data, "LONG")

    seq_vals = seq_slice[seq_features].fillna(0).values
    seq_vals = scalers.seq_scaler.transform(seq_vals)

    static_vals = seq_slice.iloc[-1][static_features].fillna(0).values.reshape(1, -1)
    static_vals = scalers.static_scaler.transform(static_vals)

    device = next(model.parameters()).device

    x_seq = torch.tensor(seq_vals.reshape(1, seq_len, len(seq_features)), dtype=torch.float32).to(device)
    x_static = torch.tensor(static_vals, dtype=torch.float32).to(device)
    x_item = torch.tensor([item_to_idx[sku_code]], dtype=torch.long).to(device)

    model.eval()
    with torch.no_grad():
        pred_res_log = float(model(x_seq, x_static, x_item).cpu().numpy().reshape(-1)[0])

    log_clip = float(artifacts.get("gru_log_clip_value", 7.0))
    pred_res_log = float(np.clip(pred_res_log, -log_clip, log_clip))
    pred_residual = float(gru_signed_log_inverse([pred_res_log])[0])

    row = next_row.iloc[0].to_dict()
    baseline = float(row.get(BASELINE_COL, 0) or 0)

    # FIX #2: apply the asymmetric residual-strength guardrail before
    # adding the residual to the baseline. This is the fix that would have
    # caught SKU 604925's -10,954 residual (anchor ~14,360 -> floor ~-6,462),
    # since clipping wasn't triggered (-9.30 vs +/-10.19 log bound) but the
    # un-dampened inverse-transformed residual was still far too large.
    pred_residual = apply_residual_strength(row, pred_residual, segment="LONG")

    forecast = max(baseline + pred_residual, 0)

    return {
        "ItemCode": int(float(sku_code)),
        "Forecast_Year": int(next_year),
        "Forecast_Month": int(next_month),
        "Forecast_Prediction": float(forecast),
        "Segment": "LONG",
        "Subgroup": "",
        "Champion_Model": "GRU",
        "Used_Model": "GRU",
        "Fallback_Used": 0,
        "Expected_Bonus": int(row.get("Expected_Bonus_Month", 0) or 0),
        "Residual_Baseline": baseline,
        "Predicted_Residual": pred_residual,
        "Predicted_Residual_Log": pred_res_log,
        "Status": "Success"
    }


# 3) LONG router
def forecast_long_sku(sku_code, raw_data):
    routing = get_long_routing_row(sku_code)

    if routing is None:
        return forecast_fallback_sku(sku_code, raw_data, "LONG")

    best_model = str(routing.get("Best_Model", "XGBOOST")).upper()
    final_model = str(routing.get("Final_Model", best_model)).upper()

    if final_model == "FALLBACK":
        return forecast_fallback_sku(sku_code, raw_data, "LONG")

    selected_model = final_model

    if selected_model == "GRU":
        return forecast_long_gru_sku(sku_code, raw_data)

    artifacts = get_long_deploy_artifact(selected_model)

    if artifacts is None:
        return forecast_fallback_sku(sku_code, raw_data, "LONG")

    result = predict_residual_model_sku(
        sku_code=sku_code,
        raw_data=raw_data,
        artifacts=artifacts,
        used_model_name=selected_model,
        subgroup=""
    )

    return result if result is not None else forecast_fallback_sku(sku_code, raw_data, "LONG")


# 4) MEDIUM router
def forecast_medium_sku(sku_code, raw_data):
    routing = get_medium_routing_row(sku_code)

    if routing is None:
        return forecast_fallback_sku(sku_code, raw_data, "MEDIUM", subgroup="UNKNOWN")

    subgroup = str(routing.get("Medium_Subgroup", "STABLE")).upper()
    best_model = str(routing.get("Best_Model", "XGBOOST")).upper()
    final_model = str(routing.get("Final_Model", best_model)).upper()

    if final_model == "FALLBACK":
        return forecast_fallback_sku(sku_code, raw_data, "MEDIUM", subgroup=subgroup)

    selected_model = final_model
    artifacts = get_medium_deploy_artifact(subgroup, selected_model)

    if artifacts is None:
        return forecast_fallback_sku(sku_code, raw_data, "MEDIUM", subgroup=subgroup)

    result = predict_residual_model_sku(
        sku_code=sku_code,
        raw_data=raw_data,
        artifacts=artifacts,
        used_model_name=selected_model,
        subgroup=subgroup
    )

    return result if result is not None else forecast_fallback_sku(
        sku_code, raw_data, "MEDIUM", subgroup=subgroup
    )


# 5) SHORT router
def forecast_short_sku(sku_code, raw_data):
    next_row, next_year, next_month = build_next_month_inference_frame(raw_data, sku_code)

    if next_row is None:
        return None

    # FIX #8: merge the Short_SKU_Type / Short_History_Length / etc. profile
    # onto next_row before calling short_rule_predict. Without this,
    # short_rule_predict's row.get("Short_SKU_Type", "SHORT_NORMAL") and
    # row.get("Short_History_Length", 0) silently default for every SHORT
    # SKU at inference, forcing the hist_len <= 2 branch every time and
    # bypassing the promo-blend / sku_mean-floor logic entirely.
    short_profile_df = build_short_sku_profile(force_itemcode_str(raw_data))
    next_row = merge_short_sku_profile(next_row, short_profile_df)

    row = next_row.iloc[0].to_dict()
    pred = short_rule_predict(row)

    subgroup = row.get("Short_SKU_Type", "SHORT_NORMAL")

    return {
        "ItemCode": int(float(sku_code)),
        "Forecast_Year": int(next_year),
        "Forecast_Month": int(next_month),
        "Forecast_Prediction": float(pred),
        "Segment": "SHORT",
        "Subgroup": subgroup,
        "Champion_Model": "RULE_BASED",
        "Used_Model": "RULE_BASED",
        "Fallback_Used": 0,
        "Expected_Bonus": int(row.get("Expected_Bonus_Month", 0) or 0),
        "Residual_Baseline": np.nan,
        "Predicted_Residual": np.nan,
        "Status": "Success"
    }


# 6) Unified router
def forecast_one_sku(sku_code, raw_data):
    segment = choose_segment_by_history(raw_data, sku_code)

    if segment == "LONG":
        result = forecast_long_sku(sku_code, raw_data)
    elif segment == "MEDIUM":
        result = forecast_medium_sku(sku_code, raw_data)
    else:
        result = forecast_short_sku(sku_code, raw_data)

    if result is not None:
        result["Behavior_Type"] = get_latest_behavior_type(raw_data, sku_code)

    return result


def bulk_segmented_forecast(raw_data, sku_list):
    results = []
    failed = []

    for sku in sku_list:
        try:
            result = forecast_one_sku(sku, raw_data)

            if result is None or str(result.get("Status", "")).startswith("FAILED"):
                failed.append({
                    "ItemCode": int(float(sku)),
                    "Segment": choose_segment_by_history(raw_data, sku),
                    "Status": "FAILED"
                })
            else:
                results.append(result)

        except Exception as e:
            failed.append({
                "ItemCode": int(float(sku)),
                "Segment": choose_segment_by_history(raw_data, sku),
                "Status": str(e)
            })

    return pd.DataFrame(results), pd.DataFrame(failed)


# ------------------------------------------------------------
# 7) INFERENCE VALIDATION CHECKS
# ------------------------------------------------------------
required_objects = [
    "Cleaned_Base_Data",
    "PHARMA_SKUS",
    "champion_long_map_df",
    "champion_medium_map_df",
    "medium_deploy_model_registry",
    "short_rule_predict",
    "build_model_features",
    "build_short_sku_profile",
    "merge_short_sku_profile",
]

missing_objects = [x for x in required_objects if x not in globals()]
if missing_objects:
    raise ValueError(f"Missing required inference objects: {missing_objects}")

required_cols = ["ItemCode", "Year", "Month_Number", "Secondary_Sales_Qty"]
missing_cols = [c for c in required_cols if c not in Cleaned_Base_Data.columns]
if missing_cols:
    raise ValueError(f"Missing columns in Cleaned_Base_Data: {missing_cols}")

if len(PHARMA_SKUS) == 0:
    raise ValueError("PHARMA_SKUS is empty.")

print("Inference validation passed.")
print("SKU count:", len(PHARMA_SKUS))
print("Data rows:", len(Cleaned_Base_Data))

long_artifact_check = [
    "xgb_long_deploy_artifacts",
    "catboost_long_deploy_artifacts",
    "lgbm_long_deploy_artifacts"
]

available_long_artifacts = [x for x in long_artifact_check if x in globals()]
if len(available_long_artifacts) == 0:
    print("Warning: No LONG deploy artifacts found. LONG SKUs will use fallback.")

if "GRU" in champion_long_map_df["Final_Model"].astype(str).str.upper().values:
    if "forecast_long_gru_sku" not in globals():
        raise ValueError("GRU is selected in Final_Model but forecast_long_gru_sku is missing.")

all_inference_skus = sorted(set(int(x) for x in PHARMA_SKUS))

# ============================================================
# 8) NEXT-MONTH FORECAST
# ============================================================

next_month_predictions, next_month_failed = bulk_segmented_forecast(
    raw_data=Cleaned_Base_Data.copy(),
    sku_list=all_inference_skus
)

if next_month_predictions.empty:
    print("No next-month predictions generated.")
    print(next_month_failed.head(20))
    raise ValueError("next_month_predictions is empty.")

# keep only the next forecast month
latest_forecast_year, latest_forecast_month = get_next_forecast_period(Cleaned_Base_Data)

next_month_final = next_month_predictions[
    (next_month_predictions["Forecast_Year"] == latest_forecast_year) &
    (next_month_predictions["Forecast_Month"] == latest_forecast_month)
].copy()

next_month_final["Forecast_Period"] = (
    next_month_final["Forecast_Year"].astype(int).astype(str)
    + "-"
    + next_month_final["Forecast_Month"].astype(int).astype(str).str.zfill(2)
)

preferred_cols = [
    "ItemCode",
    "Forecast_Period",
    "Forecast_Year",
    "Forecast_Month",
    "Forecast_Prediction",
    "Segment",
    "Behavior_Type",
    "Subgroup",
    "Champion_Model",
    "Used_Model",
    "Fallback_Used",
    "Residual_Baseline",
    "Predicted_Residual",
    "Predicted_Residual_Log",
    "Status"
]

next_month_final = next_month_final[
    [c for c in preferred_cols if c in next_month_final.columns]
].copy()

output_file_next = "next_month_forecast.xlsx"

with pd.ExcelWriter(output_file_next, engine="openpyxl") as writer:
    next_month_final.to_excel(writer, sheet_name="Next_Month_Forecast", index=False)
    next_month_predictions.to_excel(writer, sheet_name="All_Next_Predictions", index=False)

    if not next_month_failed.empty:
        next_month_failed.to_excel(writer, sheet_name="Failed_SKUs", index=False)

print("Saved:", output_file_next)
print("Next month:", f"{int(latest_forecast_year)}-{int(latest_forecast_month):02d}")
print("Forecast rows:", len(next_month_final))
print("Unique SKUs:", next_month_final["ItemCode"].nunique())

Inference validation passed.
SKU count: 728
Data rows: 35708
Saved: next_month_forecast.xlsx
Next month: 2026-02
Forecast rows: 727
Unique SKUs: 727


# 05_Analyse

In [ ]:
# 05_ANALYSE
# ============================================================
# Forecast vs Actual analysis using fixed deployed model artifacts
# Training artifacts are fixed.
# Analysis uses FULL raw data to simulate:
# Jan cutoff -> predict Feb -> compare Feb actual
# Feb cutoff -> predict Mar -> compare Mar actual
# Mar cutoff -> predict Apr -> compare Apr actual
# ============================================================

import pandas as pd
import numpy as np

ANALYSIS_OUTPUT_FILE = "forecast_vs_actual_analysis.xlsx"

EVAL_TARGET_PERIODS = [
    (2026, 2),
    (2026, 3),
    (2026, 4),
]


# ------------------------------------------------------------
# Load FULL raw data for analysis only
# ------------------------------------------------------------
def load_full_raw_for_analysis():
    raw = pd.read_excel(DATA_PATH)

    raw = raw.rename(columns={"MonthNo": "Month_Number"})

    if "Month" in raw.columns:
        raw["Month"] = pd.to_datetime(raw["Month"], errors="coerce")

    raw = force_itemcode_str(raw)

    raw = raw[raw["ItemCode"].isin(PHARMA_SKUS)].copy()
    raw = raw.sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)

    return raw


Raw_Full_Data = load_full_raw_for_analysis()

print("Full raw rows:", len(Raw_Full_Data))
print("Full raw max period:")
print(
    Raw_Full_Data[["Year", "Month_Number"]]
    .drop_duplicates()
    .sort_values(["Year", "Month_Number"])
    .tail()
)


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def previous_month(year, month):
    if month == 1:
        return year - 1, 12
    return year, month - 1


def period_label(year, month):
    return f"{int(year)}-{int(month):02d}"


def month_display_label(year, month):
    return pd.Timestamp(year=int(year), month=int(month), day=1).strftime("%b-%y")


def build_actual_lookup(raw_data):
    actual = raw_data.copy()
    actual = force_itemcode_str(actual)

    actual["Forecast_Period"] = (
        actual["Year"].astype(int).astype(str)
        + "-"
        + actual["Month_Number"].astype(int).astype(str).str.zfill(2)
    )

    actual = actual.rename(columns={"Secondary_Sales_Qty": "Actual"})
    actual = actual[["ItemCode", "Forecast_Period", "Actual"]].copy()

    actual["Actual"] = (
        pd.to_numeric(actual["Actual"], errors="coerce")
        .fillna(0)
        .clip(lower=0)
    )

    actual = actual.drop_duplicates(
        subset=["ItemCode", "Forecast_Period"],
        keep="last"
    )

    return actual


def accuracy_percent(actual, forecast):
    actual = float(actual or 0)
    forecast = float(forecast or 0)

    if actual <= 0:
        return np.nan

    acc = 1 - abs(actual - forecast) / actual
    return max(acc, 0) * 100


def run_cutoff_forecast_for_period(raw_data, sku_list, target_year, target_month):
    cutoff_year, cutoff_month = previous_month(target_year, target_month)

    print("=" * 70)
    print(f"Target forecast month : {period_label(target_year, target_month)}")
    print(f"Cutoff history up to  : {period_label(cutoff_year, cutoff_month)}")
    print("=" * 70)

    hist_df = rebuild_cutoff_preprocessed_data(
        base_raw_df=raw_data,
        cutoff_year=cutoff_year,
        cutoff_month=cutoff_month
    )

    preds_df, failed_df = bulk_segmented_forecast(
        raw_data=hist_df,
        sku_list=sku_list
    )

    if preds_df.empty:
        return pd.DataFrame(), failed_df

    preds_df = preds_df[
        (preds_df["Forecast_Year"].astype(int) == int(target_year)) &
        (preds_df["Forecast_Month"].astype(int) == int(target_month))
    ].copy()

    preds_df["Cutoff_Period"] = period_label(cutoff_year, cutoff_month)
    preds_df["Forecast_Period"] = period_label(target_year, target_month)

    return preds_df, failed_df


def build_forecast_actual_analysis(raw_data, sku_list, target_periods):
    actual_lookup = build_actual_lookup(raw_data)

    all_results = []
    all_failed = []

    for target_year, target_month in target_periods:
        preds_df, failed_df = run_cutoff_forecast_for_period(
            raw_data=raw_data,
            sku_list=sku_list,
            target_year=target_year,
            target_month=target_month
        )

        if not preds_df.empty:
            preds_df["ItemCode"] = preds_df["ItemCode"].astype(str)

            compare_df = preds_df.merge(
                actual_lookup,
                on=["ItemCode", "Forecast_Period"],
                how="left"
            )

            compare_df["Forecast"] = (
                pd.to_numeric(compare_df["Forecast_Prediction"], errors="coerce")
                .fillna(0)
                .clip(lower=0)
            )

            compare_df["Actual"] = (
                pd.to_numeric(compare_df["Actual"], errors="coerce")
                .fillna(0)
                .clip(lower=0)
            )

            compare_df["Absolute_Error"] = (
                compare_df["Actual"] - compare_df["Forecast"]
            ).abs()

            compare_df["Accuracy_%"] = compare_df.apply(
                lambda r: accuracy_percent(r["Actual"], r["Forecast"]),
                axis=1
            )

            all_results.append(compare_df)

        if failed_df is not None and not failed_df.empty:
            failed_df["Forecast_Period"] = period_label(target_year, target_month)
            all_failed.append(failed_df)

    final_results = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
    final_failed = pd.concat(all_failed, ignore_index=True) if all_failed else pd.DataFrame()

    return final_results, final_failed


# ------------------------------------------------------------
# RUN ANALYSIS
# ------------------------------------------------------------
analysis_skus = sorted(set(int(x) for x in PHARMA_SKUS))

analysis_results, analysis_failed = build_forecast_actual_analysis(
    raw_data=Raw_Full_Data.copy(),
    sku_list=analysis_skus,
    target_periods=EVAL_TARGET_PERIODS
)

if analysis_results.empty:
    print("No analysis results generated.")
    if not analysis_failed.empty:
        print(analysis_failed.head(30))
else:
    monthly_summary = (
        analysis_results
        .groupby("Forecast_Period")
        .agg(
            Forecast_Rows=("ItemCode", "count"),
            Unique_SKUs=("ItemCode", "nunique"),
            Total_Actual=("Actual", "sum"),
            Total_Forecast=("Forecast", "sum"),
            Total_Abs_Error=("Absolute_Error", "sum"),
            Mean_Accuracy=("Accuracy_%", "mean"),
            Median_Accuracy=("Accuracy_%", "median")
        )
        .reset_index()
    )

    monthly_summary["WMAPE_%"] = np.where(
        monthly_summary["Total_Actual"] > 0,
        monthly_summary["Total_Abs_Error"] / monthly_summary["Total_Actual"] * 100,
        np.nan
    )

    monthly_summary["Weighted_Accuracy_%"] = (
        100 - monthly_summary["WMAPE_%"]
    ).clip(lower=0)

    sku_summary = (
        analysis_results
        .groupby("ItemCode")
        .agg(
            Months=("Forecast_Period", "nunique"),
            Total_Actual=("Actual", "sum"),
            Total_Forecast=("Forecast", "sum"),
            Total_Abs_Error=("Absolute_Error", "sum")
        )
        .reset_index()
    )

    sku_summary["SKU_WMAPE_%"] = np.where(
        sku_summary["Total_Actual"] > 0,
        sku_summary["Total_Abs_Error"] / sku_summary["Total_Actual"] * 100,
        np.nan
    )

    sku_summary["SKU_Accuracy_%"] = (
        100 - sku_summary["SKU_WMAPE_%"]
    ).clip(lower=0)

    with pd.ExcelWriter(ANALYSIS_OUTPUT_FILE, engine="openpyxl") as writer:
        analysis_results.to_excel(writer, sheet_name="Forecast_vs_Actual", index=False)
        monthly_summary.to_excel(writer, sheet_name="Monthly_Summary", index=False)
        sku_summary.to_excel(writer, sheet_name="SKU_Summary", index=False)

        if not analysis_failed.empty:
            analysis_failed.to_excel(writer, sheet_name="Failed_SKUs", index=False)

    print("Saved:", ANALYSIS_OUTPUT_FILE)
    print("Rows:", len(analysis_results))
    print("Unique SKUs:", analysis_results["ItemCode"].nunique())
    print(monthly_summary)

In [ ]:
# Save champion selection Excel
with pd.ExcelWriter("long_champion_selection.xlsx", engine="openpyxl") as writer:
    champion_long_map_df.to_excel(
        writer,
        sheet_name="Champion_Map",
        index=False
    )

    long_sku_model_summary.to_excel(
        writer,
        sheet_name="All_Model_Scores",
        index=False
    )

    long_eval_with_champion_df.to_excel(
        writer,
        sheet_name="Eval_With_Champion",
        index=False
    )

    behavior_model_summary.to_excel(
        writer,
        sheet_name="Behavior_Model_Summary",
        index=False
    )

    behavior_champion_df.to_excel(
        writer,
        sheet_name="Behavior_Champion",
        index=False
    )

print("Saved: long_champion_selection.xlsx")

In [ ]:
# ============================================================
# DEBUG: Inspect GRU inference inputs for a single SKU
# ============================================================
DEBUG_SKU = "604925"

# ------------------------------------------------------------
# STEP 1 — Pull the raw history this SKU has in Cleaned_Base_Data
# ------------------------------------------------------------
raw_hist = force_itemcode_str(Cleaned_Base_Data.copy())
raw_hist = raw_hist[raw_hist["ItemCode"] == DEBUG_SKU].copy()
raw_hist = raw_hist.sort_values(["Year", "Month_Number"])

print("===== RAW HISTORY (last 8 rows) =====")
print(raw_hist[[
    "Year", "Month_Number", "Clean_Demand", "Secondary_Sales_Qty",
    "Bonus_Flag", "Supply_Constraint_Flag",
    "Available_Primary_Inventory_Qty", "Distributor_Inventory_Qty"
]].tail(8).to_string(index=False))

# ------------------------------------------------------------
# STEP 2 — Rebuild the exact inference frame the GRU router uses
# ------------------------------------------------------------
full_df, next_row, next_year, next_month = build_next_month_full_feature_frame(
    Cleaned_Base_Data.copy(), DEBUG_SKU
)

print(f"\n===== INFERENCE TARGET: {next_year}-{next_month:02d} =====")

if next_row is None:
    raise ValueError("next_row is None — SKU history too short or missing.")

# ------------------------------------------------------------
# STEP 3 — Compare static features at inference vs last known training row
# ------------------------------------------------------------
artifacts = gru_long_deploy_artifacts
seq_features = artifacts["seq_features"]
static_features = artifacts["static_features"]
seq_len = artifacts["seq_len"]
scalers = gru_long_deploy_scalers

# the last row used for STATIC features at inference
inference_static_row = next_row.iloc[0]

# the corresponding row from the deployment training frame (if available)
train_static_row = None
if "gru_long_deploy_df" in globals():
    tdf = force_itemcode_str(gru_long_deploy_df.copy())
    tdf = tdf[tdf.get("ItemCode_Original", tdf["ItemCode"]) == DEBUG_SKU]
    if not tdf.empty:
        train_static_row = tdf.sort_values(["Year", "Month_Number"]).iloc[-1]

print("\n===== STATIC FEATURE COMPARISON (inference vs last training row) =====")
for f in static_features:
    inf_val = inference_static_row.get(f, "MISSING")
    train_val = train_static_row.get(f, "N/A") if train_static_row is not None else "N/A (gru_long_deploy_df not in scope)"
    flag = ""
    try:
        if pd.isna(inf_val) or inf_val == 0:
            flag = "  <-- ZERO/NaN"
    except Exception:
        pass
    print(f"{f:30s} | inference={inf_val!s:>15s} | train_last={train_val!s:>15s}{flag}")

# ------------------------------------------------------------
# STEP 4 — Inspect the full 18-month sequence window fed to the GRU
# ------------------------------------------------------------
full_df_str = force_itemcode_str(full_df.copy())
full_df_str = ensure_inference_features(full_df_str, seq_features + static_features)

g = full_df_str[full_df_str["ItemCode"].astype(str) == DEBUG_SKU].copy()
g = g.sort_values(["Year", "Month_Number"]).reset_index(drop=True)

seq_slice = g.tail(seq_len).copy()

print(f"\n===== SEQUENCE WINDOW ({len(seq_slice)} of {seq_len} required rows) =====")
print(seq_slice[["Year", "Month_Number"] + seq_features].to_string(index=False))

if len(seq_slice) < seq_len:
    print(f"\n*** WARNING: only {len(seq_slice)} rows available, need {seq_len}. "
          f"This SKU would silently fall back. ***")

# Check for zero/NaN-heavy rows inside the sequence (most likely culprit)
print("\n===== PER-ROW ZERO/NaN COUNT ACROSS SEQ FEATURES =====")
zero_or_nan = seq_slice[seq_features].apply(
    lambda row: sum((pd.isna(v) or v == 0) for v in row), axis=1
)
diag = seq_slice[["Year", "Month_Number"]].copy()
diag["Zero_or_NaN_Feature_Count"] = zero_or_nan.values
diag["Total_Features"] = len(seq_features)
print(diag.to_string(index=False))

# Flag any single feature that's ALL zero across the whole window (dead feature)
print("\n===== FEATURES THAT ARE 100% ZERO ACROSS THE WHOLE SEQUENCE WINDOW =====")
dead_feats = []
for f in seq_features:
    col = pd.to_numeric(seq_slice[f], errors="coerce").fillna(0)
    if (col == 0).all():
        dead_feats.append(f)
print(dead_feats if dead_feats else "None")

# ------------------------------------------------------------
# STEP 5 — Run the scaler transform and inspect for outlier scaled values
# ------------------------------------------------------------
seq_vals_raw = seq_slice[seq_features].fillna(0).values
seq_vals_scaled = scalers.seq_scaler.transform(seq_vals_raw)

static_vals_raw = seq_slice.iloc[-1][static_features].fillna(0).values.reshape(1, -1)
static_vals_scaled = scalers.static_scaler.transform(static_vals_raw)

print("\n===== SCALED SEQUENCE VALUES — last row (most recent month) =====")
last_scaled = pd.Series(seq_vals_scaled[-1], index=seq_features)
print(last_scaled.sort_values(key=abs, ascending=False).to_string())
print("\n(Any |scaled value| > 4-5 is a strong outlier relative to training distribution)")

print("\n===== SCALED STATIC VALUES =====")
static_scaled_series = pd.Series(static_vals_scaled[0], index=static_features)
print(static_scaled_series.sort_values(key=abs, ascending=False).to_string())

# ------------------------------------------------------------
# STEP 6 — Run the model and inspect the residual + clip behavior
# ------------------------------------------------------------
device = next(artifacts["model"].parameters()).device
item_to_idx = artifacts["item_to_idx"]

if DEBUG_SKU not in item_to_idx:
    print(f"\n*** {DEBUG_SKU} NOT in item_to_idx — would silently fall back to UNK embedding or fallback forecast ***")
else:
    x_seq = torch.tensor(seq_vals_scaled.reshape(1, seq_len, len(seq_features)), dtype=torch.float32).to(device)
    x_static = torch.tensor(static_vals_scaled, dtype=torch.float32).to(device)
    x_item = torch.tensor([item_to_idx[DEBUG_SKU]], dtype=torch.long).to(device)

    artifacts["model"].eval()
    with torch.no_grad():
        pred_res_log_raw = float(artifacts["model"](x_seq, x_static, x_item).cpu().numpy().reshape(-1)[0])

    log_clip = float(artifacts.get("gru_log_clip_value", 7.0))
    pred_res_log_clipped = float(np.clip(pred_res_log_raw, -log_clip, log_clip))
    pred_residual = float(gru_signed_log_inverse([pred_res_log_clipped])[0])

    baseline = float(next_row.iloc[0].get(BASELINE_COL, 0) or 0)
    forecast = max(baseline + pred_residual, 0)

    print("\n===== MODEL OUTPUT DIAGNOSIS =====")
    print(f"Raw model output (log space)     : {pred_res_log_raw:.4f}")
    print(f"Stored gru_log_clip_value         : {log_clip:.4f}")
    print(f"Was clipping triggered?           : {abs(pred_res_log_raw) > log_clip}")
    print(f"Clipped log residual              : {pred_res_log_clipped:.4f}")
    print(f"Inverse-transformed residual       : {pred_residual:.4f}")
    print(f"Baseline (Rolling3M_Mean/Lag1)     : {baseline:.4f}")
    print(f"Final forecast                     : {forecast:.4f}")